In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
import os

base_path = "/content/drive/MyDrive/Behavior Detection/Behavioral_State_Model"

print("Folder exists:", os.path.exists(base_path))
print("Files inside:")
print(os.listdir(base_path))

Folder exists: True
Files inside:
['behavioral_dataset.csv', 'Behavioral_State_Model.ipynb']


In [55]:
import pandas as pd

columns = [
    "session_id",
    "subject_id",

    # Voice / Reading features
    "reading_accuracy",
    "reading_speed",
    "hesitation_count",
    "pause_duration",
    "fluency_score",
    "severity",
    "error_count",
    "repeated_errors",

    # Interaction features
    "response_latency",
    "retry_count",
    "skip_count",
    "idle_time",
    "task_completion_rate",
    "session_duration",
    "help_requests",

    # Camera features
    "gaze_away_frequency",
    "attention_duration",
    "head_pose_variation",
    "face_visibility",

    # Reading / Eye-tracking features
    "sum_sacc_dur",
    "sum_fix_dur",
    "dwell_time",
    "mean_fix_dur",
    "n_sacc",
    "n_regress",

    # Target
    "behavior_state"
]

behavioral_dataset = pd.DataFrame(columns=columns)

output_path = base_path + "/behavioral_dataset.csv"
behavioral_dataset.to_csv(output_path, index=False)

print("Dataset created successfully!")
print("Shape:", behavioral_dataset.shape)
print("Saved to:", output_path)
print("\nColumns:")
for col in behavioral_dataset.columns:
    print("-", col)

Dataset created successfully!
Shape: (0, 28)
Saved to: /content/drive/MyDrive/Behavior Detection/Behavioral_State_Model/behavioral_dataset.csv

Columns:
- session_id
- subject_id
- reading_accuracy
- reading_speed
- hesitation_count
- pause_duration
- fluency_score
- severity
- error_count
- repeated_errors
- response_latency
- retry_count
- skip_count
- idle_time
- task_completion_rate
- session_duration
- help_requests
- gaze_away_frequency
- attention_duration
- head_pose_variation
- face_visibility
- sum_sacc_dur
- sum_fix_dur
- dwell_time
- mean_fix_dur
- n_sacc
- n_regress
- behavior_state


In [56]:
import pandas as pd
import glob
import os

# ETDD70 paths
etdd_path = "/content/drive/MyDrive/Behavior Detection/ETDD70/data/data"
behavioral_path = "/content/drive/MyDrive/Behavior Detection/Behavioral_State_Model/behavioral_dataset.csv"

# Find all metrics files
metrics_files = glob.glob(os.path.join(etdd_path, "*_metrics.csv"))

print("Metrics files found:", len(metrics_files))

rows = []

for file in metrics_files:

    # Read metrics file
    df = pd.read_csv(file)

    if df.empty:
        continue

    # Keep one row per trial/session
    trial_df = df.drop_duplicates(subset=["trialid"]).iloc[0]

    filename = os.path.basename(file)

    # Example:
    # Subject_1235_T5_Pseudo_Text_metrics.csv
    name = filename.replace("_metrics.csv", "")

    parts = name.split("_")

    subject_id = int(parts[1])

    # Create session ID
    session_id = name

    # Extract task
    task = trial_df["task"] if "task" in trial_df.index else ""

    row = {
        "session_id": session_id,
        "subject_id": subject_id,

        # ETDD70 eye-tracking features
        "sum_sacc_dur": trial_df["sum_sacc_dur_trial"],
        "sum_fix_dur": trial_df["sum_fix_dur_trial"],
        "dwell_time": trial_df["dwell_time_trial"],
        "mean_fix_dur": trial_df["mean_fix_dur_trial"],
        "n_sacc": trial_df["n_sacc_trial"],
        "n_regress": trial_df["n_regress_trial"],

        # Behavioral state is NOT assigned yet
        "behavior_state": None
    }

    rows.append(row)

# Create dataframe
etdd_behavior = pd.DataFrame(rows)

print("\nETDD70 behavioral feature dataset")
print("Shape:", etdd_behavior.shape)
print("\nFirst 5 rows:")
display(etdd_behavior.head())

print("\nSubjects:", etdd_behavior["subject_id"].nunique())
print("Sessions:", len(etdd_behavior))

Metrics files found: 210

ETDD70 behavioral feature dataset
Shape: (210, 9)

First 5 rows:


,session_id,subject_id,sum_sacc_dur,sum_fix_dur,dwell_time,mean_fix_dur,n_sacc,n_regress,behavior_state
0,Subject_1235_T5_Pseudo_Text,1235,9798.889,118469.347,127308.354,493.622279,239,16,None
1,Subject_1090_T5_Pseudo_Text,1090,8547.732,142779.041,150346.851,582.771596,244,15,None
2,Subject_1040_T1_Syllables,1040,10379.919,67457.308,77077.278,355.038463,189,21,None
3,Subject_1019_T1_Syllables,1019,6415.297,54347.358,60130.722,343.970620,157,17,None
4,Subject_1417_T5_Pseudo_Text,1417,11850.152,134584.873,145383.124,511.729555,262,23,None



Subjects: 70
Sessions: 210


In [57]:
# Load ETDD70 dyslexia labels
label_path = "/content/drive/MyDrive/Behavior Detection/ETDD70/dyslexia_class_label.csv"

dyslexia_labels = pd.read_csv(label_path)

print("Dyslexia label dataset:")
print(dyslexia_labels.shape)
display(dyslexia_labels.head())

# Merge dyslexia labels with our session-level features
etdd_behavior = etdd_behavior.merge(
    dyslexia_labels[["subject_id", "label"]],
    on="subject_id",
    how="left"
)

# Rename for clarity
etdd_behavior = etdd_behavior.rename(
    columns={"label": "dyslexia_status"}
)

print("\nAfter merging:")
print("Shape:", etdd_behavior.shape)

print("\nDyslexia status distribution:")
print(etdd_behavior["dyslexia_status"].value_counts())

print("\nMissing dyslexia labels:",
      etdd_behavior["dyslexia_status"].isna().sum())

display(etdd_behavior.head())

Dyslexia label dataset:
(70, 3)


,subject_id,class_id,label
0,1003,0,non-dyslexic
1,1016,0,non-dyslexic
2,1019,0,non-dyslexic
3,1033,0,non-dyslexic
4,1040,0,non-dyslexic



After merging:
Shape: (210, 10)

Dyslexia status distribution:
dyslexia_status
non-dyslexic    105
dyslexic        105
Name: count, dtype: int64

Missing dyslexia labels: 0


,session_id,subject_id,sum_sacc_dur,sum_fix_dur,dwell_time,mean_fix_dur,n_sacc,n_regress,behavior_state,dyslexia_status
0,Subject_1235_T5_Pseudo_Text,1235,9798.889,118469.347,127308.354,493.622279,239,16,None,non-dyslexic
1,Subject_1090_T5_Pseudo_Text,1090,8547.732,142779.041,150346.851,582.771596,244,15,None,non-dyslexic
2,Subject_1040_T1_Syllables,1040,10379.919,67457.308,77077.278,355.038463,189,21,None,non-dyslexic
3,Subject_1019_T1_Syllables,1019,6415.297,54347.358,60130.722,343.970620,157,17,None,non-dyslexic
4,Subject_1417_T5_Pseudo_Text,1417,11850.152,134584.873,145383.124,511.729555,262,23,None,dyslexic


In [58]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [59]:
import os

EMOREACT_PATH = "/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0"

print("EmoReact exists:", os.path.exists(EMOREACT_PATH))
print("Contents:")
print(os.listdir(EMOREACT_PATH))

EmoReact exists: True
Contents:
['ReadMe.txt', 'Visual_features', 'Labels', 'Data', 'Test_names.txt', 'Train_names.txt', 'Val_names.txt', 'emoreact_visual_labels_clean.csv', 'train_visual_features.csv', 'val_visual_features.csv', 'test_visual_features.csv', 'val_missing_reconstructed.csv', 'test_missing_reconstructed.csv']


In [60]:
import os

TRAIN_FEAT = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Train_feat"
)

print("Train_feat exists:", os.path.exists(TRAIN_FEAT))

files = os.listdir(TRAIN_FEAT)

params_files = [f for f in files if f.endswith(".params.txt")]
au_files = [f for f in files if f.endswith("_au.txt")]
gaze_files = [f for f in files if f.endswith("_gaze.txt")]

print("Total files:", len(files))
print("Params files:", len(params_files))
print("AU files:", len(au_files))
print("Gaze files:", len(gaze_files))

print("\nFirst 5 files:")
print(files[:5])

Train_feat exists: True
Total files: 1297
Params files: 432
AU files: 432
Gaze files: 432

First 5 files:
['WALKMANS58_2.params.txt', 'SKIPIT64_2.params.txt', 'BULLYING83_2_au.txt', 'Gay_Marriage74_2.params.txt', 'VCR98_2_gaze.txt']


In [61]:
# Check whether params, AU and gaze files can be matched by video ID

params_ids = {
    f.replace(".params.txt", "")
    for f in params_files
}

au_ids = {
    f.replace("_au.txt", "")
    for f in au_files
}

gaze_ids = {
    f.replace("_gaze.txt", "")
    for f in gaze_files
}

print("Params IDs:", len(params_ids))
print("AU IDs:", len(au_ids))
print("Gaze IDs:", len(gaze_ids))

print("\nParams == AU:", params_ids == au_ids)
print("Params == Gaze:", params_ids == gaze_ids)

print("\nMissing in AU:", len(params_ids - au_ids))
print("Missing in Gaze:", len(params_ids - gaze_ids))

print("\nExample matched IDs:")
print(list(sorted(params_ids))[:10])

Params IDs: 432
AU IDs: 432
Gaze IDs: 432

Params == AU: True
Params == Gaze: True

Missing in AU: 0
Missing in Gaze: 0

Example matched IDs:
["80'sTOYS105_2", "80'sTOYS118_2", "80'sTOYS127_2", "80'sTOYS128_2", "80'sTOYS129_2", "80'sTOYS132_2", "80'sTOYS134_2", "80'sTOYS24_2", "80'sTOYS2_2", "80'sTOYS30_2"]


In [62]:
# Test feature extraction for ONE video

import pandas as pd
import numpy as np
import os

video_id = sorted(params_ids)[0]

params_path = os.path.join(TRAIN_FEAT, video_id + ".params.txt")
au_path = os.path.join(TRAIN_FEAT, video_id + "_au.txt")
gaze_path = os.path.join(TRAIN_FEAT, video_id + "_gaze.txt")

print("Testing video:", video_id)

params_df = pd.read_csv(params_path)
au_df = pd.read_csv(au_path)
gaze_df = pd.read_csv(gaze_path)

print("\nPARAMS")
print(params_df.shape)
print(params_df.columns.tolist())

print("\nAU")
print(au_df.shape)
print(au_df.columns.tolist())

print("\nGAZE")
print(gaze_df.shape)
print(gaze_df.columns.tolist())

Testing video: 80'sTOYS105_2

PARAMS
(104, 44)
['frame', ' timestamp', ' confidence', ' success', ' scale', ' rx', ' ry', ' rz', ' tx', ' ty', ' p0', ' p1', ' p2', ' p3', ' p4', ' p5', ' p6', ' p7', ' p8', ' p9', ' p10', ' p11', ' p12', ' p13', ' p14', ' p15', ' p16', ' p17', ' p18', ' p19', ' p20', ' p21', ' p22', ' p23', ' p24', ' p25', ' p26', ' p27', ' p28', ' p29', ' p30', ' p31', ' p32', ' p33']

AU
(104, 24)
['frame', ' timestamp', ' confidence', ' success', ' AU01_r', ' AU02_r', ' AU04_r', ' AU05_r', ' AU06_r', ' AU09_r', ' AU10_r', ' AU12_r', ' AU14_r', ' AU15_r', ' AU17_r', ' AU20_r', ' AU25_r', ' AU26_r', ' AU04_c', ' AU12_c', ' AU15_c', ' AU23_c', ' AU28_c', ' AU45_c']

GAZE
(104, 16)
['frame', ' timestamp', ' confidence', ' success', ' x_0', ' y_0', ' z_0', ' x_1', ' y_1', ' z_1', ' x_h0', ' y_h0', ' z_h0', ' x_h1', ' y_h1', ' z_h1']


In [63]:
# Check Train_names.txt

TRAIN_NAMES = os.path.join(
    EMOREACT_PATH,
    "Train_names.txt"
)

with open(TRAIN_NAMES, "r", encoding="utf-8") as f:
    train_names = [line.strip() for line in f if line.strip()]

print("Number of train names:", len(train_names))
print("\nFirst 10 names:")
for name in train_names[:10]:
    print(name)

Number of train names: 432

First 10 names:
'80''sTOYS105_2.mp4'
'80''sTOYS118_2.mp4'
'80''sTOYS127_2.mp4'
'80''sTOYS128_2.mp4'
'80''sTOYS129_2.mp4'
'80''sTOYS132_2.mp4'
'80''sTOYS134_2.mp4'
'80''sTOYS24_2.mp4'
'80''sTOYS2_2.mp4'
'80''sTOYS30_2.mp4'


In [64]:
# Clean video names and match them with feature IDs

clean_train_names = [
    name.strip().strip("'").strip('"')
    for name in train_names
]

# Remove .mp4 to get the feature/video ID
train_video_ids = [
    os.path.splitext(name)[0]
    for name in clean_train_names
]

# Check matching with our feature IDs
feature_id_set = params_ids

matched_ids = set(train_video_ids) & feature_id_set
missing_features = set(train_video_ids) - feature_id_set
extra_features = feature_id_set - set(train_video_ids)

print("Train names:", len(clean_train_names))
print("Unique video IDs:", len(set(train_video_ids)))
print("Matched with features:", len(matched_ids))
print("Missing feature IDs:", len(missing_features))
print("Extra feature IDs:", len(extra_features))

print("\nFirst 10 cleaned names:")
print(clean_train_names[:10])

print("\nFirst 10 video IDs:")
print(train_video_ids[:10])

Train names: 432
Unique video IDs: 432
Matched with features: 420
Missing feature IDs: 12
Extra feature IDs: 12

First 10 cleaned names:
["80''sTOYS105_2.mp4", "80''sTOYS118_2.mp4", "80''sTOYS127_2.mp4", "80''sTOYS128_2.mp4", "80''sTOYS129_2.mp4", "80''sTOYS132_2.mp4", "80''sTOYS134_2.mp4", "80''sTOYS24_2.mp4", "80''sTOYS2_2.mp4", "80''sTOYS30_2.mp4"]

First 10 video IDs:
["80''sTOYS105_2", "80''sTOYS118_2", "80''sTOYS127_2", "80''sTOYS128_2", "80''sTOYS129_2", "80''sTOYS132_2", "80''sTOYS134_2", "80''sTOYS24_2", "80''sTOYS2_2", "80''sTOYS30_2"]


In [65]:
# Fix apostrophe formatting in Train_names.txt

clean_train_names_fixed = [
    name.strip().strip("'").strip('"').replace("''", "'")
    for name in train_names
]

train_video_ids_fixed = [
    os.path.splitext(name)[0]
    for name in clean_train_names_fixed
]

matched_ids = set(train_video_ids_fixed) & feature_id_set
missing_features = set(train_video_ids_fixed) - feature_id_set
extra_features = feature_id_set - set(train_video_ids_fixed)

print("Train names:", len(clean_train_names_fixed))
print("Unique video IDs:", len(set(train_video_ids_fixed)))
print("Matched with features:", len(matched_ids))
print("Missing feature IDs:", len(missing_features))
print("Extra feature IDs:", len(extra_features))

print("\nFirst 10 fixed IDs:")
print(train_video_ids_fixed[:10])

print("\nMissing IDs:")
print(sorted(missing_features))

Train names: 432
Unique video IDs: 432
Matched with features: 432
Missing feature IDs: 0
Extra feature IDs: 0

First 10 fixed IDs:
["80'sTOYS105_2", "80'sTOYS118_2", "80'sTOYS127_2", "80'sTOYS128_2", "80'sTOYS129_2", "80'sTOYS132_2", "80'sTOYS134_2", "80'sTOYS24_2", "80'sTOYS2_2", "80'sTOYS30_2"]

Missing IDs:
[]


In [66]:
# Inspect EmoReact Train label file

LABELS_PATH = os.path.join(
    EMOREACT_PATH,
    "Labels",
    "train_labels.text"
)

print("Labels file exists:", os.path.exists(LABELS_PATH))

with open(LABELS_PATH, "r", encoding="utf-8") as f:
    label_lines = [line.strip() for line in f if line.strip()]

print("Number of label rows:", len(label_lines))

print("\nFirst 5 raw label rows:")
for row in label_lines[:5]:
    print(row)

Labels file exists: True
Number of label rows: 432

First 5 raw label rows:
0,0,0,1,0,0,0,0,3.3333
0,0,1,1,0,0,0,0,6.3333
0,0,0,0,0,0,0,0,2.6667
0,0,0,0,0,0,0,0,2.3333
0,0,0,1,0,0,0,0,4.6667


In [67]:
# Read EmoReact ReadMe using compatible encoding

README_PATH = os.path.join(
    EMOREACT_PATH,
    "ReadMe.txt"
)

with open(README_PATH, "r", encoding="latin-1") as f:
    readme_text = f.read()

print(readme_text)

This folder contains the followings:
Data:
This folder contains all videos from EmoReact. We have divided videos into train, validation and test folders. This
division is done in a person independent manner( people from training data do not appear in validation or test data.)

Labels:
This folder contains labels for the following affects ( First column --> curiosity , second column --> Uncertainty,...)
1- Curiosity
2- Uncertainty
3- Excitement
4- Happiness
5- Surprise
6- Disgust
7- Fear
8- Frustration
9- Valence

Visual features:
We realese the visual features that we have extracted using OpenFace. If you decide to use these features, please consider
citing :OpenFace: an open source facial behavior analysis toolkit Tadas Baltruaitis, Peter Robinson, and Louis-Philippe Morency, in IEEE Winter Conference on Applications of Computer Vision, 2016"




In [68]:
# Create structured EmoReact training labels

label_columns = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]

# Convert each comma-separated row into numbers
label_data = [
    [float(x) for x in row.split(",")]
    for row in label_lines
]

labels_df = pd.DataFrame(
    label_data,
    columns=label_columns
)

# Add the correctly matched video ID
labels_df["video_id"] = train_video_ids_fixed

# Put video_id first
labels_df = labels_df[
    ["video_id"] + label_columns
]

print("Shape:", labels_df.shape)
print("\nColumns:")
print(labels_df.columns.tolist())

print("\nFirst 5 rows:")
display(labels_df.head())

print("\nFrustration distribution:")
print(labels_df["frustration"].value_counts().sort_index())

print("\nValence range:")
print(
    labels_df["valence"].min(),
    "to",
    labels_df["valence"].max()
)

Shape: (432, 10)

Columns:
['video_id', 'curiosity', 'uncertainty', 'excitement', 'happiness', 'surprise', 'disgust', 'fear', 'frustration', 'valence']

First 5 rows:


,video_id,curiosity,uncertainty,excitement,happiness,surprise,disgust,fear,frustration,valence
0,80'sTOYS105_2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3.3333
1,80'sTOYS118_2,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,6.3333
2,80'sTOYS127_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.6667
3,80'sTOYS128_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.3333
4,80'sTOYS129_2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4.6667



Frustration distribution:
frustration
0.0    379
1.0     53
Name: count, dtype: int64

Valence range:
1.3333 to 6.6667


In [69]:
# ============================================================
# Build aggregated visual features for all 432 Train videos
# ============================================================

def aggregate_visual_features(video_id):
    params_path = os.path.join(
        TRAIN_FEAT, video_id + ".params.txt"
    )
    au_path = os.path.join(
        TRAIN_FEAT, video_id + "_au.txt"
    )
    gaze_path = os.path.join(
        TRAIN_FEAT, video_id + "_gaze.txt"
    )

    # Read the three feature files
    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # --------------------------------------------------------
    # PARAMS: head pose / face parameters
    # --------------------------------------------------------
    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:
        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        # Treat -100 as missing/invalid
        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # --------------------------------------------------------
    # AU: facial Action Units
    # --------------------------------------------------------
    au_numeric = au.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in au_numeric.columns:
        values = pd.to_numeric(
            au_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # --------------------------------------------------------
    # GAZE
    # --------------------------------------------------------
    gaze_numeric = gaze.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in gaze_numeric.columns:
        values = pd.to_numeric(
            gaze_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    return features


# ------------------------------------------------------------
# Test on all 432 videos
# ------------------------------------------------------------

visual_rows = []

for i, video_id in enumerate(train_video_ids_fixed, start=1):

    try:
        row = aggregate_visual_features(video_id)
        visual_rows.append(row)

    except Exception as e:
        print(f"Error processing {video_id}: {e}")

    if i % 50 == 0:
        print(f"Processed {i}/432 videos")


visual_features_df = pd.DataFrame(visual_rows)

print("\nFinal visual feature dataset:")
print("Shape:", visual_features_df.shape)

print("\nFirst 5 rows:")
display(visual_features_df.head())

Processed 50/432 videos
Processed 100/432 videos
Processed 150/432 videos
Processed 200/432 videos
Processed 250/432 videos
Processed 300/432 videos
Processed 350/432 videos
Processed 400/432 videos

Final visual feature dataset:
Shape: (432, 151)

First 5 rows:


,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,80'sTOYS105_2,2.147979,1.258182,0.970218,0.003600,1.0,0.0,1.460534,0.024545,0.190502,...,0.398612,0.082259,-0.895826,0.034312,-0.108495,0.033026,0.299370,0.083425,-0.943173,0.032689
1,80'sTOYS118_2,4.984146,2.895652,0.954697,0.022784,1.0,0.0,1.249333,0.054770,0.092727,...,0.321813,0.092623,-0.897008,0.092924,-0.056610,0.235858,0.297562,0.080987,-0.918309,0.055178
2,80'sTOYS127_2,1.897729,1.113699,0.981012,0.008426,1.0,0.0,1.620695,0.022446,0.105488,...,0.269444,0.080947,-0.929847,0.028388,-0.363157,0.094172,0.421468,0.088916,-0.818253,0.066337
3,80'sTOYS128_2,2.314813,1.354504,0.965176,0.013088,1.0,0.0,1.436963,0.020159,0.073711,...,0.253662,0.090014,-0.887938,0.089816,0.056984,0.226363,0.237928,0.104632,-0.936400,0.040982
4,80'sTOYS129_2,1.835167,1.077577,0.984900,0.009282,1.0,0.0,1.266610,0.025625,0.145561,...,0.224451,0.039168,-0.937590,0.052074,-0.179699,0.245991,0.225923,0.046226,-0.924283,0.020695


In [70]:
# ============================================================
# Create Train Labels DataFrame
# ============================================================

LABELS_PATH = os.path.join(
    EMOREACT_PATH,
    "Labels",
    "train_labels.text"
)

with open(LABELS_PATH, "r") as f:
    label_rows = [
        line.strip().split(",")
        for line in f
        if line.strip()
    ]

train_labels_df = pd.DataFrame(
    label_rows,
    columns=[
        "curiosity",
        "uncertainty",
        "excitement",
        "happiness",
        "surprise",
        "disgust",
        "fear",
        "frustration",
        "valence"
    ]
)

# Convert labels to numeric
train_labels_df = train_labels_df.apply(
    pd.to_numeric,
    errors="coerce"
)

# Add video IDs in the exact order provided by Train_names.txt
train_labels_df.insert(
    0,
    "video_id",
    train_video_ids_fixed
)

print("Train labels shape:", train_labels_df.shape)
print("\nColumns:")
print(train_labels_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_labels_df.head())

Train labels shape: (432, 10)

Columns:
['video_id', 'curiosity', 'uncertainty', 'excitement', 'happiness', 'surprise', 'disgust', 'fear', 'frustration', 'valence']

First 5 rows:


,video_id,curiosity,uncertainty,excitement,happiness,surprise,disgust,fear,frustration,valence
0,80'sTOYS105_2,0,0,0,1,0,0,0,0,3.3333
1,80'sTOYS118_2,0,0,1,1,0,0,0,0,6.3333
2,80'sTOYS127_2,0,0,0,0,0,0,0,0,2.6667
3,80'sTOYS128_2,0,0,0,0,0,0,0,0,2.3333
4,80'sTOYS129_2,0,0,0,1,0,0,0,0,4.6667


In [71]:
# ============================================================
# Merge Visual Features + EmoReact Labels
# ============================================================

emoreact_visual_labels = visual_features_df.merge(
    train_labels_df,
    on="video_id",
    how="inner"
)

print("Merged dataset shape:")
print(emoreact_visual_labels.shape)

print("\nNumber of matched videos:")
print(len(emoreact_visual_labels))

print("\nMissing values:")
print(emoreact_visual_labels.isna().sum().sum())

print("\nFirst 5 rows:")
display(emoreact_visual_labels.head())

Merged dataset shape:
(432, 160)

Number of matched videos:
432

Missing values:
270

First 5 rows:


,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,z_h1_std,curiosity,uncertainty,excitement,happiness,surprise,disgust,fear,frustration,valence
0,80'sTOYS105_2,2.147979,1.258182,0.970218,0.003600,1.0,0.0,1.460534,0.024545,0.190502,...,0.032689,0,0,0,1,0,0,0,0,3.3333
1,80'sTOYS118_2,4.984146,2.895652,0.954697,0.022784,1.0,0.0,1.249333,0.054770,0.092727,...,0.055178,0,0,1,1,0,0,0,0,6.3333
2,80'sTOYS127_2,1.897729,1.113699,0.981012,0.008426,1.0,0.0,1.620695,0.022446,0.105488,...,0.066337,0,0,0,0,0,0,0,0,2.6667
3,80'sTOYS128_2,2.314813,1.354504,0.965176,0.013088,1.0,0.0,1.436963,0.020159,0.073711,...,0.040982,0,0,0,0,0,0,0,0,2.3333
4,80'sTOYS129_2,1.835167,1.077577,0.984900,0.009282,1.0,0.0,1.266610,0.025625,0.145561,...,0.020695,0,0,0,1,0,0,0,0,4.6667


In [72]:
# ============================================================
# Check which features contain missing values
# ============================================================

missing_counts = (
    emoreact_visual_labels
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_counts = missing_counts[missing_counts > 0]

print("Number of columns with missing values:",
      len(missing_counts))

print("\nColumns with missing values:")
display(missing_counts)

Number of columns with missing values: 12

Columns with missing values:


,0
AU04_c_std,26
AU12_c_std,26
AU15_c_std,26
AU23_c_std,26
AU45_c_std,26
AU28_c_std,26
AU12_c_mean,19
AU04_c_mean,19
AU15_c_mean,19
AU45_c_mean,19


In [73]:
# ============================================================
# Check exactly which videos have missing AU features
# ============================================================

missing_video_rows = emoreact_visual_labels[
    emoreact_visual_labels[missing_counts.index]
    .isna()
    .any(axis=1)
]

print("Number of videos affected:",
      len(missing_video_rows))

print("\nAffected video IDs:")
print(missing_video_rows["video_id"].tolist())

print("\nMissing-value count per affected video:")
display(
    missing_video_rows[
        ["video_id"] + missing_counts.index.tolist()
    ].isna().sum(axis=1).value_counts()
)

Number of videos affected: 26

Affected video IDs:
["80'sTOYS36_2", 'BULLYING99_2', 'CAN_YOUR_PET26_2', 'CAN_YOUR_PET37_2', 'CAN_YOUR_PET48_2', 'CAN_YOUR_PET62_2', 'CAN_YOUR_PET72_2', 'CAN_YOUR_PET78_2', 'FROGLEGS94_2', 'Gay_Marriage148_2', 'Gay_Marriage224_2', 'Gay_Marriage26_2', 'Gay_Marriage9_2', 'HARLEMSHAKE33_2', 'MAKEUP_APPS112_2', 'MAKEUP_APPS114_2', 'MAKEUP_APPS187_2', 'NYAN_CAT4_2', 'REBECCA50_2', 'SWING_COPTERS158_2', 'SWING_COPTERS162_2', 'SWING_COPTERS54_2', 'SWING_COPTERS6_2', 'TOFU18_2', 'VEGEMITE20_2', 'VEGEMITE66_2']

Missing-value count per affected video:


,count
12,19
6,7


In [74]:
# ============================================================
# Handle missing visual features using median imputation
# ============================================================

from sklearn.impute import SimpleImputer

# Keep video_id and labels separate
label_columns = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]

id_column = "video_id"

feature_columns = [
    col for col in emoreact_visual_labels.columns
    if col not in [id_column] + label_columns
]

# Create imputer
imputer = SimpleImputer(strategy="median")

# Apply median imputation to visual features
emoreact_visual_labels[feature_columns] = imputer.fit_transform(
    emoreact_visual_labels[feature_columns]
)

print("Missing values AFTER imputation:",
      emoreact_visual_labels.isna().sum().sum())

print("Final dataset shape:",
      emoreact_visual_labels.shape)

Missing values AFTER imputation: 0
Final dataset shape: (432, 160)


In [75]:
# ============================================
# EmoReact - Frustration Classification
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------
# 1. Select features and target
# --------------------------------------------

X = emoreact_visual_labels.drop(
    columns=["video_id", "curiosity", "uncertainty", "excitement",
             "happiness", "surprise", "disgust", "fear",
             "frustration", "valence"]
)

y = emoreact_visual_labels["frustration"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())


# --------------------------------------------
# 2. Train-test split
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


# --------------------------------------------
# 3. Random Forest
# --------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)


# --------------------------------------------
# 4. Evaluation
# --------------------------------------------

print("\n==============================")
print("RANDOM FOREST RESULTS")
print("==============================")

print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred, zero_division=0))
print("Recall   :", recall_score(y_test, rf_pred, zero_division=0))
print("Macro F1 :", f1_score(y_test, rf_pred, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_pred,
    target_names=["Not Frustrated", "Frustrated"],
    zero_division=0
))

X shape: (432, 150)
y shape: (432,)

Class distribution:
frustration
0    379
1     53
Name: count, dtype: int64

Train shape: (345, 150)
Test shape: (87, 150)

RANDOM FOREST RESULTS
Accuracy : 0.8735632183908046
Precision: 0.0
Recall   : 0.0
Macro F1 : 0.4662576687116564

Confusion Matrix:
[[76  0]
 [11  0]]

Classification Report:
                precision    recall  f1-score   support

Not Frustrated       0.87      1.00      0.93        76
    Frustrated       0.00      0.00      0.00        11

      accuracy                           0.87        87
     macro avg       0.44      0.50      0.47        87
  weighted avg       0.76      0.87      0.81        87



In [76]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

rf_balanced = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    max_features="sqrt",
    n_jobs=-1
)

rf_balanced.fit(X_train, y_train)

y_pred = rf_balanced.predict(X_test)

print("=== BALANCED RANDOM FOREST ===")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Frustrated", "Frustrated"],
        zero_division=0
    )
)

=== BALANCED RANDOM FOREST ===
Accuracy: 0.8735632183908046
Precision: 0.0
Recall: 0.0
Macro F1: 0.4662576687116564

Confusion Matrix:
[[76  0]
 [11  0]]

Classification Report:
                precision    recall  f1-score   support

Not Frustrated       0.87      1.00      0.93        76
    Frustrated       0.00      0.00      0.00        11

      accuracy                           0.87        87
     macro avg       0.44      0.50      0.47        87
  weighted avg       0.76      0.87      0.81        87



In [77]:
print("=== FRUSTRATION DISTRIBUTION ===")

print("\nOverall:")
print(train_labels_df["frustration"].value_counts())

print("\nOverall percentage:")
print(train_labels_df["frustration"].value_counts(normalize=True) * 100)

print("\nMerged dataset:")
print(emoreact_visual_labels["frustration"].value_counts())

print("\nUnique values:")
print(emoreact_visual_labels["frustration"].unique())

=== FRUSTRATION DISTRIBUTION ===

Overall:
frustration
0    379
1     53
Name: count, dtype: int64

Overall percentage:
frustration
0    87.731481
1    12.268519
Name: proportion, dtype: float64

Merged dataset:
frustration
0    379
1     53
Name: count, dtype: int64

Unique values:
[0 1]


In [78]:
print("=== TRAIN / TEST CLASS DISTRIBUTION ===")

print("\nTRAIN:")
print(y_train.value_counts())
print("\nTRAIN percentage:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTEST:")
print(y_test.value_counts())
print("\nTEST percentage:")
print(y_test.value_counts(normalize=True) * 100)

=== TRAIN / TEST CLASS DISTRIBUTION ===

TRAIN:
frustration
0    303
1     42
Name: count, dtype: int64

TRAIN percentage:
frustration
0    87.826087
1    12.173913
Name: proportion, dtype: float64

TEST:
frustration
0    76
1    11
Name: count, dtype: int64

TEST percentage:
frustration
0    87.356322
1    12.643678
Name: proportion, dtype: float64


In [79]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Before SMOTE:")
print(Counter(y_train))

smote = SMOTE(
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE:")
print(Counter(y_train_smote))

print("\nOriginal train shape:", X_train.shape)
print("SMOTE train shape:", X_train_smote.shape)

Before SMOTE:
Counter({0: 303, 1: 42})

After SMOTE:
Counter({0: 303, 1: 303})

Original train shape: (345, 150)
SMOTE train shape: (606, 150)


In [80]:
rf_smote = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    max_features="sqrt",
    n_jobs=-1
)

rf_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = rf_smote.predict(X_test)

print("=== SMOTE + RANDOM FOREST ===")

print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_smote, zero_division=0))
print("Macro F1:", f1_score(y_test, y_pred_smote, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_smote,
        target_names=["Not Frustrated", "Frustrated"],
        zero_division=0
    )
)

=== SMOTE + RANDOM FOREST ===
Accuracy: 0.8735632183908046
Precision: 0.5
Recall: 0.09090909090909091
Macro F1: 0.5427615862398472

Confusion Matrix:
[[75  1]
 [10  1]]

Classification Report:
                precision    recall  f1-score   support

Not Frustrated       0.88      0.99      0.93        76
    Frustrated       0.50      0.09      0.15        11

      accuracy                           0.87        87
     macro avg       0.69      0.54      0.54        87
  weighted avg       0.83      0.87      0.83        87



In [81]:
from xgboost import XGBClassifier

xgb_smote = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb_smote.predict(X_test)

print("=== SMOTE + XGBOOST ===")

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_xgb, zero_division=0))
print("Macro F1:", f1_score(y_test, y_pred_xgb, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=["Not Frustrated", "Frustrated"],
        zero_division=0
    )
)

=== SMOTE + XGBOOST ===
Accuracy: 0.8620689655172413
Precision: 0.3333333333333333
Recall: 0.09090909090909091
Macro F1: 0.5339285714285714

Confusion Matrix:
[[74  2]
 [10  1]]

Classification Report:
                precision    recall  f1-score   support

Not Frustrated       0.88      0.97      0.93        76
    Frustrated       0.33      0.09      0.14        11

      accuracy                           0.86        87
     macro avg       0.61      0.53      0.53        87
  weighted avg       0.81      0.86      0.83        87



In [82]:
# Save cleaned EmoReact visual + label dataset

OUTPUT_PATH = os.path.join(
    EMOREACT_PATH,
    "emoreact_visual_labels_clean.csv"
)

emoreact_visual_labels.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved successfully!")
print("Path:", OUTPUT_PATH)
print("Shape:", emoreact_visual_labels.shape)

Saved successfully!
Path: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/emoreact_visual_labels_clean.csv
Shape: (432, 160)


In [83]:
# Check Validation files

VAL_NAMES = os.path.join(
    EMOREACT_PATH,
    "Val_names.txt"
)

VAL_LABELS = os.path.join(
    EMOREACT_PATH,
    "Labels",
    "val_labels.text"
)

VAL_FEATURES = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Validation"
)

print("Val names exists:", os.path.exists(VAL_NAMES))
print("Val labels exists:", os.path.exists(VAL_LABELS))
print("Val features exists:", os.path.exists(VAL_FEATURES))

if os.path.exists(VAL_FEATURES):
    print("Number of validation feature files:",
          len(os.listdir(VAL_FEATURES)))

Val names exists: True
Val labels exists: True
Val features exists: False


In [84]:
# Find Validation feature files/folders

VISUAL_PATH = os.path.join(
    EMOREACT_PATH,
    "Visual_features"
)

print("Visual_features contents:")
print(os.listdir(VISUAL_PATH))

Visual_features contents:
['Val_feat', 'Train_feat', 'Test_feat']


In [85]:
VAL_FEATURES = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Val_feat"
)

print("Val features exists:", os.path.exists(VAL_FEATURES))

if os.path.exists(VAL_FEATURES):
    val_files = os.listdir(VAL_FEATURES)
    print("Number of validation feature files:", len(val_files))
    print("First 10 files:")
    print(val_files[:10])

Val features exists: True
Number of validation feature files: 909
First 10 files:
['OLDCAMERAS17_2_au.txt', 'HARLEMSHAKE101_2.params.txt', 'PIZZATOPPINGS73_2_au.txt', 'iPOD82_2_gaze.txt', 'KIMCHI20_2.params.txt', 'BRUSSELSSPROUTS21_2_au.txt', 'Gay_Marriage20_2_au.txt', 'HARLEMSHAKE139_2.params.txt', 'SWING_COPTERS82_2.params.txt', 'CAN_YOUR_PET97_2.params.txt']


In [86]:
# Check Validation video IDs against feature IDs

import re

# Get video IDs from Val_names.txt
with open(VAL_NAMES, "r", encoding="utf-8") as f:
    val_names = [line.strip() for line in f if line.strip()]

val_video_ids = [
    re.sub(r"\.mp4$", "", name)
    for name in val_names
]

# Get IDs from feature filenames
val_feature_files = os.listdir(VAL_FEATURES)

val_feature_ids = set()

for file in val_feature_files:
    if file.endswith(".params.txt"):
        video_id = file.replace(".params.txt", "")
        val_feature_ids.add(video_id)

print("Validation names:", len(val_video_ids))
print("Validation feature IDs:", len(val_feature_ids))

missing_ids = set(val_video_ids) - val_feature_ids
extra_ids = val_feature_ids - set(val_video_ids)

print("Missing feature IDs:", len(missing_ids))
print("Extra feature IDs:", len(extra_ids))

print("\nFirst 10 Validation IDs:")
print(val_video_ids[:10])

Validation names: 303
Validation feature IDs: 303
Missing feature IDs: 303
Extra feature IDs: 303

First 10 Validation IDs:
["'80''sTOYS115_2.mp4'", "'80''sTOYS119_2.mp4'", "'80''sTOYS120_2.mp4'", "'80''sTOYS126_2.mp4'", "'80''sTOYS133_2.mp4'", "'80''sTOYS16_2.mp4'", "'80''sTOYS27_2.mp4'", "'80''sTOYS39_2.mp4'", "'80''sTOYS40_2.mp4'", "'ANNOYING_ORANGE137_2.mp4'"]


In [87]:
# Inspect validation feature filenames
import os

print("Total validation feature files:", len(val_files))

for f in val_files[:30]:
    print(f)

Total validation feature files: 909
OLDCAMERAS17_2_au.txt
HARLEMSHAKE101_2.params.txt
PIZZATOPPINGS73_2_au.txt
iPOD82_2_gaze.txt
KIMCHI20_2.params.txt
BRUSSELSSPROUTS21_2_au.txt
Gay_Marriage20_2_au.txt
HARLEMSHAKE139_2.params.txt
SWING_COPTERS82_2.params.txt
CAN_YOUR_PET97_2.params.txt
TOILET61_2_gaze.txt
MINECRAFT34_2_au.txt
SPAM49_2.params.txt
Gay_Marriage288_2.params.txt
OLDCOMPUTERS155_2_gaze.txt
SPAM83_2_gaze.txt
WALKMANS122_2.params.txt
APPLEWATCH28_2_gaze.txt
SWING_COPTERS80_2_gaze.txt
ROTARYPHONES37_2_au.txt
OLDCAMERAS36_2_gaze.txt
SQUID40_2.params.txt
VCR102_2_au.txt
HARLEMSHAKE12_2_au.txt
CAKE12_2_gaze.txt
Gay_Marriage313_2.params.txt
VCR102_2_gaze.txt
iPOD36_2.params.txt
VCR33_2_gaze.txt
GAMEBOY36_2.params.txt


In [88]:
# ==============================
# FIX VALIDATION VIDEO IDs
# ==============================

import os
import re

def get_video_id(filename):
    """
    Convert:
    80'sTOYS120_2_params.txt
    80'sTOYS120_2_au.txt
    80'sTOYS120_2_gaze.txt

    into:
    80'sTOYS120_2
    """

    name = os.path.splitext(filename)[0]

    for suffix in ["_params", "_au", "_gaze"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break

    return name


# Extract video IDs
val_feature_ids = sorted(
    set(get_video_id(f) for f in val_files)
)

print("Total feature files:", len(val_files))
print("Unique validation video IDs:", len(val_feature_ids))

print("\nFirst 20 extracted video IDs:")
print(val_feature_ids[:20])


# Compare with validation names
clean_val_names = []

for name in val_names:
    name = name.strip()
    name = os.path.splitext(name)[0]
    clean_val_names.append(name)

print("\nValidation names:", len(clean_val_names))
print("Validation feature IDs:", len(val_feature_ids))

missing_ids = sorted(set(clean_val_names) - set(val_feature_ids))
extra_ids = sorted(set(val_feature_ids) - set(clean_val_names))

print("\nMissing IDs:", len(missing_ids))
print(missing_ids[:20])

print("\nExtra IDs:", len(extra_ids))
print(extra_ids[:20])

Total feature files: 909
Unique validation video IDs: 606

First 20 extracted video IDs:
["80'sTOYS115_2", "80'sTOYS115_2.params", "80'sTOYS119_2", "80'sTOYS119_2.params", "80'sTOYS120_2", "80'sTOYS120_2.params", "80'sTOYS126_2", "80'sTOYS126_2.params", "80'sTOYS133_2", "80'sTOYS133_2.params", "80'sTOYS16_2", "80'sTOYS16_2.params", "80'sTOYS27_2", "80'sTOYS27_2.params", "80'sTOYS39_2", "80'sTOYS39_2.params", "80'sTOYS40_2", "80'sTOYS40_2.params", 'ANNOYING_ORANGE137_2', 'ANNOYING_ORANGE137_2.params']

Validation names: 303
Validation feature IDs: 606

Missing IDs: 303
["'80''sTOYS115_2", "'80''sTOYS119_2", "'80''sTOYS120_2", "'80''sTOYS126_2", "'80''sTOYS133_2", "'80''sTOYS16_2", "'80''sTOYS27_2", "'80''sTOYS39_2", "'80''sTOYS40_2", "'ANNOYING_ORANGE137_2", "'ANNOYING_ORANGE70_2", "'APPLEWATCH28_2", "'APPLEWATCH32_2", "'APPLEWATCH40_2", "'BRUSSELSSPROUTS20_2", "'BRUSSELSSPROUTS21_2", "'BRUSSELSSPROUTS3_2", "'BRUSSELSSPROUTS54_2", "'BRUSSELSSPROUTS74_2", "'BULLYING111_2"]

Extra IDs: 60

In [89]:
def get_video_id(filename):
    name = os.path.splitext(filename)[0]

    # params file
    if name.endswith(".params"):
        name = name[:-len(".params")]

    # AU / GAZE files
    elif name.endswith("_au"):
        name = name[:-len("_au")]

    elif name.endswith("_gaze"):
        name = name[:-len("_gaze")]

    return name


# Extract correct video IDs
val_feature_ids = sorted(
    set(get_video_id(f) for f in val_files)
)

print("Total feature files:", len(val_files))
print("Unique validation video IDs:", len(val_feature_ids))

print("\nFirst 20 IDs:")
print(val_feature_ids[:20])


# Clean validation names
clean_val_names = []

for name in val_names:
    name = name.strip()
    name = os.path.splitext(name)[0]
    clean_val_names.append(name)


missing_ids = sorted(
    set(clean_val_names) - set(val_feature_ids)
)

extra_ids = sorted(
    set(val_feature_ids) - set(clean_val_names)
)

print("\nValidation names:", len(clean_val_names))
print("Validation feature IDs:", len(val_feature_ids))

print("\nMissing IDs:", len(missing_ids))
print(missing_ids[:20])

print("\nExtra IDs:", len(extra_ids))
print(extra_ids[:20])

Total feature files: 909
Unique validation video IDs: 303

First 20 IDs:
["80'sTOYS115_2", "80'sTOYS119_2", "80'sTOYS120_2", "80'sTOYS126_2", "80'sTOYS133_2", "80'sTOYS16_2", "80'sTOYS27_2", "80'sTOYS39_2", "80'sTOYS40_2", 'ANNOYING_ORANGE137_2', 'ANNOYING_ORANGE70_2', 'APPLEWATCH28_2', 'APPLEWATCH32_2', 'APPLEWATCH40_2', 'BRUSSELSSPROUTS20_2', 'BRUSSELSSPROUTS21_2', 'BRUSSELSSPROUTS3_2', 'BRUSSELSSPROUTS54_2', 'BRUSSELSSPROUTS74_2', 'BULLYING111_2']

Validation names: 303
Validation feature IDs: 303

Missing IDs: 303
["'80''sTOYS115_2", "'80''sTOYS119_2", "'80''sTOYS120_2", "'80''sTOYS126_2", "'80''sTOYS133_2", "'80''sTOYS16_2", "'80''sTOYS27_2", "'80''sTOYS39_2", "'80''sTOYS40_2", "'ANNOYING_ORANGE137_2", "'ANNOYING_ORANGE70_2", "'APPLEWATCH28_2", "'APPLEWATCH32_2", "'APPLEWATCH40_2", "'BRUSSELSSPROUTS20_2", "'BRUSSELSSPROUTS21_2", "'BRUSSELSSPROUTS3_2", "'BRUSSELSSPROUTS54_2", "'BRUSSELSSPROUTS74_2", "'BULLYING111_2"]

Extra IDs: 303
["80'sTOYS115_2", "80'sTOYS119_2", "80'sTOYS120_2

In [90]:
print("Validation name RAW:")
print(repr(clean_val_names[0]))

print("\nFeature ID RAW:")
print(repr(val_feature_ids[0]))

print("\nValidation name length:", len(clean_val_names[0]))
print("Feature ID length:", len(val_feature_ids[0]))

Validation name RAW:
"'80''sTOYS115_2"

Feature ID RAW:
"80'sTOYS115_2"

Validation name length: 15
Feature ID length: 13


In [91]:
print("FIRST ID EQUAL:", clean_val_names[0] == val_feature_ids[0])

print("\nValidation first 5:")
for x in clean_val_names[:5]:
    print(repr(x))

print("\nFeature first 5:")
for x in val_feature_ids[:5]:
    print(repr(x))

print("\nCommon IDs:", len(set(clean_val_names) & set(val_feature_ids)))

print("Validation only:", len(set(clean_val_names) - set(val_feature_ids)))
print("Feature only:", len(set(val_feature_ids) - set(clean_val_names)))

FIRST ID EQUAL: False

Validation first 5:
"'80''sTOYS115_2"
"'80''sTOYS119_2"
"'80''sTOYS120_2"
"'80''sTOYS126_2"
"'80''sTOYS133_2"

Feature first 5:
"80'sTOYS115_2"
"80'sTOYS119_2"
"80'sTOYS120_2"
"80'sTOYS126_2"
"80'sTOYS133_2"

Common IDs: 0
Validation only: 303
Feature only: 303


In [92]:
# Remove extra leading/trailing quote characters from validation names

clean_val_names = [
    name.strip().strip("'").strip('"')
    for name in clean_val_names
]

print("Validation first 5 AFTER cleaning:")
for x in clean_val_names[:5]:
    print(repr(x))

print("\nFeature first 5:")
for x in val_feature_ids[:5]:
    print(repr(x))

print("\nFIRST ID EQUAL:",
      clean_val_names[0] == val_feature_ids[0])

print("\nCommon IDs:",
      len(set(clean_val_names) & set(val_feature_ids)))

print("Validation only:",
      len(set(clean_val_names) - set(val_feature_ids)))

print("Feature only:",
      len(set(val_feature_ids) - set(clean_val_names)))

Validation first 5 AFTER cleaning:
"80''sTOYS115_2"
"80''sTOYS119_2"
"80''sTOYS120_2"
"80''sTOYS126_2"
"80''sTOYS133_2"

Feature first 5:
"80'sTOYS115_2"
"80'sTOYS119_2"
"80'sTOYS120_2"
"80'sTOYS126_2"
"80'sTOYS133_2"

FIRST ID EQUAL: False

Common IDs: 294
Validation only: 9
Feature only: 9


In [93]:
# Find the 9 validation IDs that do not match

validation_set = set(clean_val_names)
feature_set = set(val_feature_ids)

validation_only = sorted(validation_set - feature_set)
feature_only = sorted(feature_set - validation_set)

print("VALIDATION ONLY:", len(validation_only))
for x in validation_only:
    print(repr(x))

print("\nFEATURE ONLY:", len(feature_only))
for x in feature_only:
    print(repr(x))

VALIDATION ONLY: 9
"80''sTOYS115_2"
"80''sTOYS119_2"
"80''sTOYS120_2"
"80''sTOYS126_2"
"80''sTOYS133_2"
"80''sTOYS16_2"
"80''sTOYS27_2"
"80''sTOYS39_2"
"80''sTOYS40_2"

FEATURE ONLY: 9
"80'sTOYS115_2"
"80'sTOYS119_2"
"80'sTOYS120_2"
"80'sTOYS126_2"
"80'sTOYS133_2"
"80'sTOYS16_2"
"80'sTOYS27_2"
"80'sTOYS39_2"
"80'sTOYS40_2"


In [94]:
# Check exact character differences for the first mismatch

v = validation_only[0]
f = feature_only[0]

print("VALIDATION:")
print(repr(v))
print("Length:", len(v))
print("Characters:", [ord(c) for c in v])

print("\nFEATURE:")
print(repr(f))
print("Length:", len(f))
print("Characters:", [ord(c) for c in f])

print("\nCharacter-by-character comparison:")
for i, (a, b) in enumerate(zip(v, f)):
    if a != b:
        print(
            "Position:", i,
            "Validation:", repr(a), ord(a),
            "Feature:", repr(b), ord(b)
        )

VALIDATION:
"80''sTOYS115_2"
Length: 14
Characters: [56, 48, 39, 39, 115, 84, 79, 89, 83, 49, 49, 53, 95, 50]

FEATURE:
"80'sTOYS115_2"
Length: 13
Characters: [56, 48, 39, 115, 84, 79, 89, 83, 49, 49, 53, 95, 50]

Character-by-character comparison:
Position: 3 Validation: "'" 39 Feature: 's' 115
Position: 4 Validation: 's' 115 Feature: 'T' 84
Position: 5 Validation: 'T' 84 Feature: 'O' 79
Position: 6 Validation: 'O' 79 Feature: 'Y' 89
Position: 7 Validation: 'Y' 89 Feature: 'S' 83
Position: 8 Validation: 'S' 83 Feature: '1' 49
Position: 10 Validation: '1' 49 Feature: '5' 53
Position: 11 Validation: '5' 53 Feature: '_' 95
Position: 12 Validation: '_' 95 Feature: '2' 50


In [95]:
# Fix duplicated apostrophe in validation video IDs

clean_val_names = [
    name.replace("''", "'")
    for name in clean_val_names
]

# Check matching again
validation_set = set(clean_val_names)
feature_set = set(val_feature_ids)

print("Validation names:", len(clean_val_names))
print("Feature IDs:", len(val_feature_ids))

print("Common IDs:", len(validation_set & feature_set))
print("Validation only:", len(validation_set - feature_set))
print("Feature only:", len(feature_set - validation_set))

print("\nFirst 5 validation IDs after final cleaning:")
for x in clean_val_names[:5]:
    print(repr(x))

Validation names: 303
Feature IDs: 303
Common IDs: 303
Validation only: 0
Feature only: 0

First 5 validation IDs after final cleaning:
"80'sTOYS115_2"
"80'sTOYS119_2"
"80'sTOYS120_2"
"80'sTOYS126_2"
"80'sTOYS133_2"


In [96]:
# Safely inspect currently defined functions/variables
print("Possible feature-extraction functions:")

for name, obj in list(globals().items()):
    if callable(obj):
        name_lower = name.lower()
        if (
            "feature" in name_lower
            or "extract" in name_lower
            or "visual" in name_lower
        ):
            print(name)

Possible feature-extraction functions:
aggregate_visual_features
aggregate_visual_features_v2
clean_feature_id


In [97]:
# Check the function signature before using it
import inspect

print(inspect.signature(aggregate_visual_features))
print(inspect.getsource(aggregate_visual_features))

(video_id)
def aggregate_visual_features(video_id):
    params_path = os.path.join(
        TRAIN_FEAT, video_id + ".params.txt"
    )
    au_path = os.path.join(
        TRAIN_FEAT, video_id + "_au.txt"
    )
    gaze_path = os.path.join(
        TRAIN_FEAT, video_id + "_gaze.txt"
    )

    # Read the three feature files
    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # --------------------------------------------------------
    # PARAMS: head pose / face parameters
    # --------------------------------------------------------
    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:
        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        # Treat -100 as missing/invalid
        values =

In [99]:
def aggregate_visual_features(video_id, feature_dir=TRAIN_FEAT):

_IncompleteInputError: incomplete input (3647649069.py, line 1)

In [100]:
# ============================================================
# NEW VERSION - Visual Feature Extraction
# Works for BOTH Train and Validation folders
# ============================================================

def aggregate_visual_features_v2(video_id, feature_dir):

    # --------------------------------------------------------
    # File paths
    # --------------------------------------------------------
    params_path = os.path.join(
        feature_dir,
        video_id + ".params.txt"
    )

    au_path = os.path.join(
        feature_dir,
        video_id + "_au.txt"
    )

    gaze_path = os.path.join(
        feature_dir,
        video_id + "_gaze.txt"
    )

    # --------------------------------------------------------
    # Check files exist
    # --------------------------------------------------------
    if not os.path.exists(params_path):
        raise FileNotFoundError(f"Missing params file: {params_path}")

    if not os.path.exists(au_path):
        raise FileNotFoundError(f"Missing AU file: {au_path}")

    if not os.path.exists(gaze_path):
        raise FileNotFoundError(f"Missing gaze file: {gaze_path}")

    # --------------------------------------------------------
    # Read feature files
    # --------------------------------------------------------
    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # ========================================================
    # PARAMS - Head pose / face parameters
    # ========================================================

    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:

        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        # -100 = invalid / missing
        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # ========================================================
    # AU - Facial Action Units
    # ========================================================

    au_numeric = au.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in au_numeric.columns:

        values = pd.to_numeric(
            au_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # ========================================================
    # GAZE
    # ========================================================

    gaze_numeric = gaze.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in gaze_numeric.columns:

        values = pd.to_numeric(
            gaze_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    return features


print("New feature extraction function created successfully!")
print("Function:", aggregate_visual_features_v2)

New feature extraction function created successfully!
Function: <function aggregate_visual_features_v2 at 0x7afa909c3880>


In [101]:
# ============================================================
# EXTRACT TRAIN VISUAL FEATURES
# ============================================================

train_feature_rows = []

for i, video_id in enumerate(clean_train_names):

    try:
        features = aggregate_visual_features_v2(
            video_id,
            TRAIN_FEAT
        )

        train_feature_rows.append(features)

    except Exception as e:
        print(f"Error processing {video_id}: {e}")

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(clean_train_names)} train videos")


# Convert to DataFrame
train_visual_features_df = pd.DataFrame(train_feature_rows)

print("\n========================================")
print("TRAIN VISUAL FEATURE DATASET")
print("========================================")
print("Shape:", train_visual_features_df.shape)
print("Expected rows:", 432)
print("Expected columns:", 151)

print("\nFirst 5 rows:")
display(train_visual_features_df.head())

Error processing 80''sTOYS105_2.mp4: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS105_2.mp4.params.txt
Error processing 80''sTOYS118_2.mp4: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS118_2.mp4.params.txt
Error processing 80''sTOYS127_2.mp4: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS127_2.mp4.params.txt
Error processing 80''sTOYS128_2.mp4: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS128_2.mp4.params.txt
Error processing 80''sTOYS129_2.mp4: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS129_2.mp4.params.txt
Error processing 80''sTOYS132_2.mp4: Missing params file: /conten

""


In [102]:
# ============================================================
# FIX VIDEO IDs
# Remove .mp4 and quotes correctly
# ============================================================

def clean_feature_id(x):
    x = str(x).strip()

    # Remove quotes
    x = x.replace('"', '').replace("'", "")

    # Remove .mp4 extension
    if x.lower().endswith(".mp4"):
        x = x[:-4]

    return x.strip()


# Re-create clean IDs from ORIGINAL raw names
clean_train_names = [
    clean_feature_id(x)
    for x in train_names_raw
]

clean_val_names = [
    clean_feature_id(x)
    for x in val_names_raw
]


print("Train IDs:", len(clean_train_names))
print("Validation IDs:", len(clean_val_names))

print("\nFirst 10 TRAIN IDs:")
for x in clean_train_names[:10]:
    print(repr(x))

print("\nFirst 10 VALIDATION IDs:")
for x in clean_val_names[:10]:
    print(repr(x))

NameError: name 'train_names_raw' is not defined

In [103]:
# ============================================
# RECREATE TRAIN + VALIDATION NAME LISTS
# ============================================

import os

# EmoReact main path
EMOREACT_PATH = "/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0"

# ------------------------------------------------
# 1. TRAIN NAMES
# ------------------------------------------------
TRAIN_NAMES_FILE = os.path.join(EMOREACT_PATH, "Train_names.txt")

with open(TRAIN_NAMES_FILE, "r", encoding="utf-8") as f:
    train_names_raw = [line.strip() for line in f if line.strip()]

# ------------------------------------------------
# 2. VALIDATION NAMES
# ------------------------------------------------
VAL_NAMES_FILE = os.path.join(EMOREACT_PATH, "Val_names.txt")

with open(VAL_NAMES_FILE, "r", encoding="utf-8") as f:
    val_names_raw = [line.strip() for line in f if line.strip()]

# ------------------------------------------------
# 3. CLEAN VIDEO IDs
# Remove .mp4 and extra quotes
# ------------------------------------------------
def clean_video_id(x):
    x = str(x).strip()
    x = x.replace('"', '')
    x = x.replace("'", '')

    if x.lower().endswith(".mp4"):
        x = x[:-4]

    return x.strip()


clean_train_names = [
    clean_video_id(x)
    for x in train_names_raw
]

clean_val_names = [
    clean_video_id(x)
    for x in val_names_raw
]

# ------------------------------------------------
# 4. PRINT RESULTS
# ------------------------------------------------
print("TRAIN names:", len(clean_train_names))
print("VALIDATION names:", len(clean_val_names))

print("\nFIRST 10 TRAIN IDs:")
for x in clean_train_names[:10]:
    print(repr(x))

print("\nFIRST 10 VALIDATION IDs:")
for x in clean_val_names[:10]:
    print(repr(x))

# ------------------------------------------------
# 5. CHECK DUPLICATES
# ------------------------------------------------
print("\nDuplicate TRAIN IDs:",
      len(clean_train_names) - len(set(clean_train_names)))

print("Duplicate VALIDATION IDs:",
      len(clean_val_names) - len(set(clean_val_names)))

TRAIN names: 432
VALIDATION names: 303

FIRST 10 TRAIN IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'

FIRST 10 VALIDATION IDs:
'80sTOYS115_2'
'80sTOYS119_2'
'80sTOYS120_2'
'80sTOYS126_2'
'80sTOYS133_2'
'80sTOYS16_2'
'80sTOYS27_2'
'80sTOYS39_2'
'80sTOYS40_2'
'ANNOYING_ORANGE137_2'

Duplicate TRAIN IDs: 0
Duplicate VALIDATION IDs: 0


In [104]:
# ==========================================
# CHECK TRAIN FEATURE FILES
# ==========================================

print("Checking TRAIN visual feature files...")

TRAIN_FEATURE_PATH = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Train_feat"
)

print("Train feature folder exists:", os.path.exists(TRAIN_FEATURE_PATH))

if os.path.exists(TRAIN_FEATURE_PATH):

    train_feature_files = os.listdir(TRAIN_FEATURE_PATH)

    print("Total train feature files:", len(train_feature_files))

    # Get video IDs from feature filenames
    train_feature_ids = set()

    for f in train_feature_files:
        if f.endswith(".txt"):
            base = f.rsplit(".", 2)[0]

            # Remove feature suffix
            for suffix in ["_params", "_au", "_gaze"]:
                if base.endswith(suffix):
                    base = base[:-len(suffix)]
                    break

            train_feature_ids.add(base)

    print("Unique train feature IDs:", len(train_feature_ids))

    # Compare with clean train IDs
    train_name_set = set(clean_train_names)

    common_ids = train_name_set & train_feature_ids
    missing_ids = train_name_set - train_feature_ids
    extra_ids = train_feature_ids - train_name_set

    print("\n========== TRAIN FEATURE MATCHING ==========")
    print("Train names:", len(train_name_set))
    print("Feature IDs:", len(train_feature_ids))
    print("Common IDs:", len(common_ids))
    print("Missing IDs:", len(missing_ids))
    print("Extra IDs:", len(extra_ids))

    print("\nFirst 10 common IDs:")
    for x in sorted(common_ids)[:10]:
        print(repr(x))

else:
    print("ERROR: Train feature folder not found!")

Checking TRAIN visual feature files...
Train feature folder exists: True
Total train feature files: 1297
Unique train feature IDs: 433

========== TRAIN FEATURE MATCHING ==========
Train names: 432
Feature IDs: 433
Common IDs: 420
Missing IDs: 12
Extra IDs: 13

First 10 common IDs:
'ANNOYING_ORANGE105_2'
'ANNOYING_ORANGE119_2'
'ANNOYING_ORANGE11_2'
'ANNOYING_ORANGE123_2'
'ANNOYING_ORANGE126_2'
'ANNOYING_ORANGE155_2'
'ANNOYING_ORANGE157_2'
'ANNOYING_ORANGE159_2'
'ANNOYING_ORANGE162_2'
'ANNOYING_ORANGE166_2'


In [105]:
# ==========================================
# SHOW MISSING AND EXTRA TRAIN FEATURE IDs
# ==========================================

print("========== MISSING TRAIN FEATURE IDs ==========")

for x in sorted(missing_ids):
    print(repr(x))

print("\nTotal missing:", len(missing_ids))


print("\n========== EXTRA TRAIN FEATURE IDs ==========")

for x in sorted(extra_ids):
    print(repr(x))

print("\nTotal extra:", len(extra_ids))

========== MISSING TRAIN FEATURE IDs ==========
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

Total missing: 12

========== EXTRA TRAIN FEATURE IDs ==========
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"
"80'sTOYS36_2"
"80'sTOYS96_2"
'MINECRAFT25_2_au (1)'

Total extra: 13


In [106]:
# ==========================================
# FINAL CLEANING OF TRAIN FEATURE IDs
# Remove BOTH single and double quotes
# ==========================================

def clean_feature_id(feature_id):
    feature_id = str(feature_id).strip()

    # Remove leading/trailing single OR double quotes
    feature_id = feature_id.strip("'\"")

    # Remove feature suffixes if present
    for suffix in ["_params", "_au", "_gaze"]:
        if feature_id.endswith(suffix):
            feature_id = feature_id[:-len(suffix)]
            break

    return feature_id


# Clean all train feature IDs again
clean_train_feature_ids = set(
    clean_feature_id(x) for x in train_feature_files
    if x.endswith(".txt")
)


# Compare with clean train names
train_name_set = set(clean_train_names)

common_train = train_name_set & clean_train_feature_ids
missing_train = train_name_set - clean_train_feature_ids
extra_train = clean_train_feature_ids - train_name_set


print("========== FINAL TRAIN FEATURE MATCH ==========")
print("Train names:", len(train_name_set))
print("Clean feature IDs:", len(clean_train_feature_ids))
print("Common IDs:", len(common_train))
print("Missing IDs:", len(missing_train))
print("Extra IDs:", len(extra_train))


print("\nMissing IDs:")
for x in sorted(missing_train):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(extra_train):
    print(repr(x))

========== FINAL TRAIN FEATURE MATCH ==========
Train names: 432
Clean feature IDs: 1297
Common IDs: 0
Missing IDs: 432
Extra IDs: 1297

Missing IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'
'ANNOYING_ORANGE105_2'
'ANNOYING_ORANGE119_2'
'ANNOYING_ORANGE11_2'
'ANNOYING_ORANGE123_2'
'ANNOYING_ORANGE126_2'
'ANNOYING_ORANGE155_2'
'ANNOYING_ORANGE157_2'
'ANNOYING_ORANGE159_2'
'ANNOYING_ORANGE162_2'
'ANNOYING_ORANGE166_2'
'ANNOYING_ORANGE169_2'
'ANNOYING_ORANGE171_2'
'ANNOYING_ORANGE172_2'
'ANNOYING_ORANGE174_2'
'ANNOYING_ORANGE175_2'
'ANNOYING_ORANGE30_2'
'ANNOYING_ORANGE39_2'
'ANNOYING_ORANGE72_2'
'ANNOYING_ORANGE73_2'
'ANNOYING_ORANGE93_2'
'APPLEWATCH16_2'
'APPLEWATCH18_2'
'APPLEWATCH19_2'
'APPLEWATCH2_2'
'APPLEWATCH31_2'
'APPLEWATCH4_2'
'BRUSSELSSPROUTS11_2'
'BRUSSELSSPROUTS16_2'
'BRUSSELSSPROUTS26_2'
'BRUSSELSSPROUTS53_2'
'BRUSSELSSPROUTS75_2'
'BULLYING101

In [107]:
# ==========================================
# CORRECT TRAIN FEATURE ID CLEANING
# ==========================================

def get_feature_video_id(filename):

    filename = str(filename).strip()

    # Remove exact feature-file suffix
    for suffix in [
        ".params.txt",
        ".au.txt",
        ".gaze.txt"
    ]:
        if filename.endswith(suffix):
            filename = filename[:-len(suffix)]
            break

    # Remove accidental quotes
    filename = filename.strip("'\"")

    return filename


# Create IDs from actual feature filenames
train_feature_ids_correct = set(
    get_feature_video_id(f)
    for f in train_feature_files
)


# Compare with train names
train_name_set = set(clean_train_names)

common_train = train_name_set & train_feature_ids_correct
missing_train = train_name_set - train_feature_ids_correct
extra_train = train_feature_ids_correct - train_name_set


print("========== CORRECT TRAIN FEATURE MATCH ==========")
print("Train names:", len(train_name_set))
print("Feature video IDs:", len(train_feature_ids_correct))
print("Common IDs:", len(common_train))
print("Missing IDs:", len(missing_train))
print("Extra IDs:", len(extra_train))

print("\nFirst 10 feature IDs:")
for x in sorted(train_feature_ids_correct)[:10]:
    print(repr(x))

print("\nMissing IDs:")
for x in sorted(missing_train):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(extra_train):
    print(repr(x))

========== CORRECT TRAIN FEATURE MATCH ==========
Train names: 432
Feature video IDs: 1297
Common IDs: 420
Missing IDs: 12
Extra IDs: 877

First 10 feature IDs:
"80'sTOYS105_2"
"80'sTOYS105_2_au.txt"
"80'sTOYS105_2_gaze.txt"
"80'sTOYS118_2"
"80'sTOYS118_2_au.txt"
"80'sTOYS118_2_gaze.txt"
"80'sTOYS127_2"
"80'sTOYS127_2_au.txt"
"80'sTOYS127_2_gaze.txt"
"80'sTOYS128_2"

Missing IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

Extra IDs:
"80'sTOYS105_2"
"80'sTOYS105_2_au.txt"
"80'sTOYS105_2_gaze.txt"
"80'sTOYS118_2"
"80'sTOYS118_2_au.txt"
"80'sTOYS118_2_gaze.txt"
"80'sTOYS127_2"
"80'sTOYS127_2_au.txt"
"80'sTOYS127_2_gaze.txt"
"80'sTOYS128_2"
"80'sTOYS128_2_au.txt"
"80'sTOYS128_2_gaze.txt"
"80'sTOYS129_2"
"80'sTOYS129_2_au.txt"
"80'sTOYS129_2_gaze.txt"
"80'sTOYS132_2"
"80'sTOYS132_2_au.txt"
"80'sTOYS132_2_gaze.txt"
"80'sTOYS134_2"
"80'sTOYS134_2_au.txt"
"80'sTOY

In [108]:
# ==========================================
# FINAL TRAIN VIDEO ID CHECK
# ==========================================

import os
import re

# Read the actual Train_feat folder again
TRAIN_FEATURE_PATH = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Train_feat"
)

all_train_files = os.listdir(TRAIN_FEATURE_PATH)

print("Total files:", len(all_train_files))


# ------------------------------------------
# Extract VIDEO IDs from feature filenames
# ------------------------------------------

train_video_ids = set()

for filename in all_train_files:

    if not filename.endswith(".txt"):
        continue

    # Remove feature type + .txt
    # Example:
    # 80'sTOYS105_2.params.txt -> 80'sTOYS105_2
    # 80'sTOYS105_2.au.txt     -> 80'sTOYS105_2
    # 80'sTOYS105_2.gaze.txt   -> 80'sTOYS105_2

    video_id = re.sub(
        r"\.(params|au|gaze)\.txt$",
        "",
        filename
    )

    # Remove accidental surrounding quotes
    video_id = video_id.strip("'\"").strip()

    # Remove duplicate-copy marker if present
    video_id = re.sub(r"\s+\(\d+\)$", "", video_id)

    train_video_ids.add(video_id)


# ------------------------------------------
# Compare with TRAIN names
# ------------------------------------------

train_names_set = set(
    x.strip("'\"").strip()
    for x in clean_train_names
)

common_ids = train_names_set & train_video_ids
missing_ids = train_names_set - train_video_ids
extra_ids = train_video_ids - train_names_set


print("\n========== FINAL CHECK ==========")

print("Train names:", len(train_names_set))
print("Unique feature VIDEO IDs:", len(train_video_ids))
print("Common IDs:", len(common_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))


print("\nFirst 10 feature VIDEO IDs:")
for x in sorted(train_video_ids)[:10]:
    print(repr(x))


print("\nMissing IDs:")
for x in sorted(missing_ids):
    print(repr(x))


print("\nExtra IDs:")
for x in sorted(extra_ids):
    print(repr(x))

Total files: 1297

========== FINAL CHECK ==========
Train names: 432
Unique feature VIDEO IDs: 1297
Common IDs: 420
Missing IDs: 12
Extra IDs: 877

First 10 feature VIDEO IDs:
"80'sTOYS105_2"
"80'sTOYS105_2_au.txt"
"80'sTOYS105_2_gaze.txt"
"80'sTOYS118_2"
"80'sTOYS118_2_au.txt"
"80'sTOYS118_2_gaze.txt"
"80'sTOYS127_2"
"80'sTOYS127_2_au.txt"
"80'sTOYS127_2_gaze.txt"
"80'sTOYS128_2"

Missing IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

Extra IDs:
"80'sTOYS105_2"
"80'sTOYS105_2_au.txt"
"80'sTOYS105_2_gaze.txt"
"80'sTOYS118_2"
"80'sTOYS118_2_au.txt"
"80'sTOYS118_2_gaze.txt"
"80'sTOYS127_2"
"80'sTOYS127_2_au.txt"
"80'sTOYS127_2_gaze.txt"
"80'sTOYS128_2"
"80'sTOYS128_2_au.txt"
"80'sTOYS128_2_gaze.txt"
"80'sTOYS129_2"
"80'sTOYS129_2_au.txt"
"80'sTOYS129_2_gaze.txt"
"80'sTOYS132_2"
"80'sTOYS132_2_au.txt"
"80'sTOYS132_2_gaze.txt"
"80'sTOYS134_2"
"80'sTOYS134_2_

In [109]:
# ==========================================
# FIX TRAIN FEATURE VIDEO IDs - FINAL
# ==========================================

import os

TRAIN_FEATURE_PATH = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Train_feat"
)

all_train_files = os.listdir(TRAIN_FEATURE_PATH)

# Only the 3 actual feature file types
feature_suffixes = [
    ".params.txt",
    ".au.txt",
    ".gaze.txt"
]

train_video_ids = set()

for filename in all_train_files:

    # Check which feature type this file belongs to
    for suffix in feature_suffixes:

        if filename.endswith(suffix):

            # Remove EXACT suffix
            video_id = filename[:-len(suffix)]

            # Remove surrounding quotes
            video_id = video_id.strip("'\"").strip()

            # Remove duplicate copy marker if present
            if video_id.endswith(" (1)"):
                video_id = video_id[:-4]

            train_video_ids.add(video_id)

            break


# ==========================================
# COMPARE WITH TRAIN NAMES
# ==========================================

train_names_set = set(
    x.strip("'\"").strip()
    for x in clean_train_names
)

common_ids = train_names_set & train_video_ids
missing_ids = train_names_set - train_video_ids
extra_ids = train_video_ids - train_names_set


print("========== FINAL TRAIN FEATURE CHECK ==========")

print("Total feature files:", len(all_train_files))
print("Train names:", len(train_names_set))
print("Unique feature VIDEO IDs:", len(train_video_ids))
print("Common IDs:", len(common_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))


print("\nFirst 10 VIDEO IDs:")
for x in sorted(train_video_ids)[:10]:
    print(repr(x))


print("\nMissing IDs:")
for x in sorted(missing_ids):
    print(repr(x))


print("\nExtra IDs:")
for x in sorted(extra_ids):
    print(repr(x))

========== FINAL TRAIN FEATURE CHECK ==========
Total feature files: 1297
Train names: 432
Unique feature VIDEO IDs: 432
Common IDs: 420
Missing IDs: 12
Extra IDs: 12

First 10 VIDEO IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"

Missing IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

Extra IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"
"80'sTOYS36_2"
"80'sTOYS96_2"


In [110]:
# ==========================================
# FIX QUOTATION MARK MISMATCH
# ==========================================

def normalize_video_id(x):
    x = str(x).strip()

    # Remove surrounding quotation marks repeatedly
    while len(x) >= 2 and x[0] in ["'", '"'] and x[-1] in ["'", '"']:
        x = x[1:-1].strip()

    return x


# Normalize TRAIN names
normalized_train_names = [
    normalize_video_id(x)
    for x in clean_train_names
]

# Normalize FEATURE IDs
normalized_feature_ids = [
    normalize_video_id(x)
    for x in train_video_ids
]

train_names_set = set(normalized_train_names)
feature_ids_set = set(normalized_feature_ids)

common_ids = train_names_set & feature_ids_set
missing_ids = train_names_set - feature_ids_set
extra_ids = feature_ids_set - train_names_set


print("========== NORMALIZED TRAIN CHECK ==========")

print("Train names:", len(train_names_set))
print("Feature video IDs:", len(feature_ids_set))
print("Common IDs:", len(common_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))


print("\nMissing IDs:")
for x in sorted(missing_ids):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(extra_ids):
    print(repr(x))

========== NORMALIZED TRAIN CHECK ==========
Train names: 432
Feature video IDs: 432
Common IDs: 420
Missing IDs: 12
Extra IDs: 12

Missing IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

Extra IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"
"80'sTOYS36_2"
"80'sTOYS96_2"


In [111]:
# ==========================================
# INSPECT THE EXACT CHARACTER DIFFERENCE
# ==========================================

test_id = "80'sTOYS105_2"

print("TRAIN NAME RAW:")
for x in clean_train_names:
    if "TOYS105" in x:
        print(repr(x))
        print("Characters:", [ord(c) for c in x])
        print("Length:", len(x))
        break


print("\nFEATURE ID RAW:")
for x in train_video_ids:
    if "TOYS105" in x:
        print(repr(x))
        print("Characters:", [ord(c) for c in x])
        print("Length:", len(x))
        break

TRAIN NAME RAW:
'80sTOYS105_2'
Characters: [56, 48, 115, 84, 79, 89, 83, 49, 48, 53, 95, 50]
Length: 12

FEATURE ID RAW:
"80'sTOYS105_2"
Characters: [56, 48, 39, 115, 84, 79, 89, 83, 49, 48, 53, 95, 50]
Length: 13


In [112]:
# ==========================================
# FINAL EMOREACT TRAIN ID NORMALIZATION
# ==========================================

import os

def normalize_emoreact_id(x):
    x = str(x).strip()

    # Remove quotation marks only from the OUTSIDE
    x = x.strip("'\"")

    return x


# ------------------------------------------
# 1. Normalize TRAIN names
# ------------------------------------------

final_train_ids = set(
    normalize_emoreact_id(x)
    for x in clean_train_names
)


# ------------------------------------------
# 2. Build FEATURE ID mapping
#    normalized ID -> actual filename prefix
# ------------------------------------------

feature_id_map = {}

for filename in all_train_files:

    if filename.endswith(".params.txt"):
        actual_id = filename[:-len(".params.txt")]

    elif filename.endswith(".au.txt"):
        actual_id = filename[:-len(".au.txt")]

    elif filename.endswith(".gaze.txt"):
        actual_id = filename[:-len(".gaze.txt")]

    else:
        continue

    normalized_id = normalize_emoreact_id(actual_id)

    feature_id_map[normalized_id] = actual_id


final_feature_ids = set(feature_id_map.keys())


# ------------------------------------------
# 3. Compare
# ------------------------------------------

common_ids = final_train_ids & final_feature_ids
missing_ids = final_train_ids - final_feature_ids
extra_ids = final_feature_ids - final_train_ids


print("========== FINAL ID MATCH ==========")

print("Train IDs:", len(final_train_ids))
print("Feature IDs:", len(final_feature_ids))
print("Common IDs:", len(common_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))


print("\nFirst 10 TRAIN IDs:")
for x in sorted(final_train_ids)[:10]:
    print(repr(x))


print("\nFirst 10 FEATURE IDs:")
for x in sorted(final_feature_ids)[:10]:
    print(repr(x))


print("\nMissing IDs:")
for x in sorted(missing_ids):
    print(repr(x))


print("\nExtra IDs:")
for x in sorted(extra_ids):
    print(repr(x))

========== FINAL ID MATCH ==========
Train IDs: 432
Feature IDs: 432
Common IDs: 420
Missing IDs: 12
Extra IDs: 12

First 10 TRAIN IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'

First 10 FEATURE IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"

Missing IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

Extra IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"
"80'sTOYS36_2"
"80'sTOYS96_2"


In [113]:
# ============================================================
# FINAL FIX - NORMALIZE EMOREACT VIDEO IDS
# ============================================================

import re
import unicodedata

def normalize_video_id(x):
    """
    Convert Train/Validation names and feature filenames
    into the same clean video ID.
    """

    x = str(x)

    # Unicode normalize
    x = unicodedata.normalize("NFKC", x)

    # Remove whitespace
    x = x.strip()

    # Remove surrounding quotes repeatedly
    x = x.strip("\"'`“”‘’ ")

    # Remove common filename extensions
    for ext in [
        ".mp4",
        ".txt",
        ".params",
        ".params.txt",
        ".au",
        ".gaze"
    ]:
        if x.lower().endswith(ext):
            x = x[:-len(ext)]

    # Remove feature suffixes if present
    for suffix in [
        "_params",
        "_au",
        "_gaze"
    ]:
        if x.lower().endswith(suffix):
            x = x[:-len(suffix)]

    # Remove any remaining surrounding quotes/spaces
    x = x.strip("\"'`“”‘’ ")

    return x


# ------------------------------------------------------------
# NORMALIZE TRAIN NAMES
# ------------------------------------------------------------

clean_train_names = [
    normalize_video_id(x)
    for x in train_names_raw
]


# ------------------------------------------------------------
# NORMALIZE TRAIN FEATURE FILES
# ONLY USE .params.txt FILES AS VIDEO IDs
# ------------------------------------------------------------

train_params_files = [
    f for f in os.listdir(TRAIN_FEAT)
    if f.lower().endswith(".params.txt")
]

clean_train_feature_ids = [
    normalize_video_id(f)
    for f in train_params_files
]


# ------------------------------------------------------------
# SET COMPARISON
# ------------------------------------------------------------

train_name_set = set(clean_train_names)
train_feature_set = set(clean_train_feature_ids)

common_ids = train_name_set & train_feature_set
missing_ids = train_name_set - train_feature_set
extra_ids = train_feature_set - train_name_set


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("FINAL NORMALIZED TRAIN FEATURE MATCH")
print("=" * 60)

print("Train names:", len(clean_train_names))
print("Unique train names:", len(train_name_set))

print("Train .params feature files:", len(train_params_files))
print("Unique feature IDs:", len(train_feature_set))

print("Common IDs:", len(common_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))


print("\nMissing IDs:")
for x in sorted(missing_ids):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(extra_ids):
    print(repr(x))

FINAL NORMALIZED TRAIN FEATURE MATCH
Train names: 432
Unique train names: 432
Train .params feature files: 432
Unique feature IDs: 432
Common IDs: 420
Missing IDs: 12
Extra IDs: 12

Missing IDs:
"80''sTOYS105_2"
"80''sTOYS118_2"
"80''sTOYS127_2"
"80''sTOYS128_2"
"80''sTOYS129_2"
"80''sTOYS132_2"
"80''sTOYS134_2"
"80''sTOYS24_2"
"80''sTOYS2_2"
"80''sTOYS30_2"
"80''sTOYS36_2"
"80''sTOYS96_2"

Extra IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"
"80'sTOYS36_2"
"80'sTOYS96_2"


In [114]:
# ============================================================
# FIX EMOREACT IDS - REMOVE ALL QUOTE CHARACTERS
# ============================================================

def clean_emoreact_id(x):
    x = str(x)

    # Unicode normalization
    x = unicodedata.normalize("NFKC", x)

    # Remove ALL quote characters
    x = x.replace("'", "")
    x = x.replace('"', "")
    x = x.replace("`", "")
    x = x.replace("“", "")
    x = x.replace("”", "")
    x = x.replace("‘", "")
    x = x.replace("’", "")

    # Remove spaces
    x = x.strip()

    # Remove file extensions
    for ext in [".params.txt", ".mp4", ".params", ".au", ".gaze", ".txt"]:
        if x.lower().endswith(ext):
            x = x[:-len(ext)]

    # Remove feature suffixes
    for suffix in ["_params", "_au", "_gaze"]:
        if x.lower().endswith(suffix):
            x = x[:-len(suffix)]

    return x.strip()


# Clean TRAIN names
clean_train_names_final = [
    clean_emoreact_id(x)
    for x in train_names_raw
]


# Get .params files
train_params_files = [
    f for f in os.listdir(TRAIN_FEAT)
    if f.lower().endswith(".params.txt")
]


# Clean feature IDs
clean_train_feature_ids_final = [
    clean_emoreact_id(f)
    for f in train_params_files
]


# Compare
train_set_final = set(clean_train_names_final)
feature_set_final = set(clean_train_feature_ids_final)

common_final = train_set_final & feature_set_final
missing_final = train_set_final - feature_set_final
extra_final = feature_set_final - train_set_final


print("=" * 60)
print("FINAL TRAIN ID MATCH AFTER QUOTE FIX")
print("=" * 60)

print("Train names:", len(clean_train_names_final))
print("Unique train IDs:", len(train_set_final))

print("Feature files:", len(train_params_files))
print("Unique feature IDs:", len(feature_set_final))

print("Common IDs:", len(common_final))
print("Missing IDs:", len(missing_final))
print("Extra IDs:", len(extra_final))


print("\nMissing IDs:")
for x in sorted(missing_final):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(extra_final):
    print(repr(x))

FINAL TRAIN ID MATCH AFTER QUOTE FIX
Train names: 432
Unique train IDs: 432
Feature files: 432
Unique feature IDs: 432
Common IDs: 432
Missing IDs: 0
Extra IDs: 0

Missing IDs:

Extra IDs:


In [115]:
# ============================================================
# BUILD TRAIN VISUAL FEATURE DATASET
# ============================================================

train_feature_rows = []

for video_id in clean_train_names_final:

    params_path = os.path.join(
        TRAIN_FEAT,
        video_id + ".params.txt"
    )

    au_path = os.path.join(
        TRAIN_FEAT,
        video_id + "_au.txt"
    )

    gaze_path = os.path.join(
        TRAIN_FEAT,
        video_id + "_gaze.txt"
    )

    try:
        # Read feature files
        params = pd.read_csv(params_path)
        au = pd.read_csv(au_path)
        gaze = pd.read_csv(gaze_path)

        features = {
            "video_id": video_id
        }

        # ----------------------------------------------------
        # PARAMS
        # ----------------------------------------------------

        params_numeric = params.select_dtypes(
            include=np.number
        ).drop(
            columns=["frame", "timestamp"],
            errors="ignore"
        )

        for col in params_numeric.columns:

            values = pd.to_numeric(
                params_numeric[col],
                errors="coerce"
            )

            values = values.replace(-100, np.nan)

            features[f"{col}_mean"] = values.mean()
            features[f"{col}_std"] = values.std()

        # ----------------------------------------------------
        # AU
        # ----------------------------------------------------

        au_numeric = au.select_dtypes(
            include=np.number
        ).drop(
            columns=["frame", "timestamp"],
            errors="ignore"
        )

        for col in au_numeric.columns:

            values = pd.to_numeric(
                au_numeric[col],
                errors="coerce"
            )

            values = values.replace(-100, np.nan)

            features[f"{col}_mean"] = values.mean()
            features[f"{col}_std"] = values.std()

        # ----------------------------------------------------
        # GAZE
        # ----------------------------------------------------

        gaze_numeric = gaze.select_dtypes(
            include=np.number
        ).drop(
            columns=["frame", "timestamp"],
            errors="ignore"
        )

        for col in gaze_numeric.columns:

            values = pd.to_numeric(
                gaze_numeric[col],
                errors="coerce"
            )

            values = values.replace(-100, np.nan)

            features[f"{col}_mean"] = values.mean()
            features[f"{col}_std"] = values.std()

        train_feature_rows.append(features)

    except Exception as e:
        print(f"Error processing {video_id}: {e}")


# Convert to DataFrame
train_visual_features = pd.DataFrame(train_feature_rows)

print("=" * 60)
print("TRAIN VISUAL FEATURE DATASET")
print("=" * 60)

print("Shape:", train_visual_features.shape)
print("Expected rows:", len(clean_train_names_final))
print("Actual rows:", len(train_visual_features))

print("\nFirst 5 rows:")
display(train_visual_features.head())

Error processing 80sTOYS105_2: [Errno 2] No such file or directory: '/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80sTOYS105_2.params.txt'
Error processing 80sTOYS118_2: [Errno 2] No such file or directory: '/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80sTOYS118_2.params.txt'
Error processing 80sTOYS127_2: [Errno 2] No such file or directory: '/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80sTOYS127_2.params.txt'
Error processing 80sTOYS128_2: [Errno 2] No such file or directory: '/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80sTOYS128_2.params.txt'
Error processing 80sTOYS129_2: [Errno 2] No such file or directory: '/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80sTOYS129_2.params.txt'
Error processing 80sTOYS132_2: [Err

,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,ANNOYING_ORANGE105_2,1.835167,1.077577,0.969798,0.004329,1.0,0.0,1.273815,0.027521,0.269296,...,0.542937,0.054635,-0.552933,0.046337,0.498727,0.058552,0.467983,0.046922,-0.724690,0.039234
1,ANNOYING_ORANGE119_2,1.647479,0.969214,0.941390,0.039892,1.0,0.0,1.245697,0.029645,0.066726,...,0.324343,0.034190,-0.903425,0.019423,0.055097,0.041188,0.324100,0.034777,-0.942812,0.012721
2,ANNOYING_ORANGE11_2,3.524354,2.052838,0.981597,0.015345,1.0,0.0,1.623457,0.032629,-0.029637,...,0.255915,0.064985,-0.941921,0.021662,-0.442279,0.072558,0.177775,0.091807,-0.870970,0.024252
3,ANNOYING_ORANGE123_2,1.584917,0.933093,0.974706,0.004804,1.0,0.0,1.792782,0.038580,0.253426,...,0.376226,0.094453,-0.890915,0.028635,0.008951,0.055033,0.365244,0.135555,-0.918104,0.050078
4,ANNOYING_ORANGE126_2,2.523354,1.474907,0.976877,0.006221,1.0,0.0,1.196154,0.018492,-0.039224,...,0.230680,0.046296,-0.970785,0.010848,-0.071258,0.029203,0.213486,0.039115,-0.973094,0.008390


In [116]:
# ============================================================
# CHECK THE 12 MISSING TRAIN FILES
# ============================================================

missing_train_ids = sorted(missing_final)

print("Missing IDs:", len(missing_train_ids))
print()

for video_id in missing_train_ids:
    print("=" * 60)
    print("VIDEO ID:", repr(video_id))

    # Show any files containing this ID
    matches = [
        f for f in os.listdir(TRAIN_FEAT)
        if video_id.lower() in f.lower()
    ]

    print("Matching files:")
    if matches:
        for f in matches:
            print("   ", repr(f))
    else:
        print("    NO FILE FOUND")

Missing IDs: 0



In [117]:
# ============================================================
# DIRECTLY CHECK ALL 432 TRAIN IDs AGAINST .params.txt FILES
# ============================================================

train_ids = set(clean_train_names_final)

actual_params_ids = set(
    normalize_video_id(f)
    for f in os.listdir(TRAIN_FEAT)
    if f.lower().endswith(".params.txt")
)

missing_now = sorted(train_ids - actual_params_ids)
common_now = train_ids & actual_params_ids
extra_now = sorted(actual_params_ids - train_ids)

print("=" * 60)
print("DIRECT TRAIN FEATURE FILE CHECK")
print("=" * 60)

print("Train IDs:", len(train_ids))
print("Actual .params IDs:", len(actual_params_ids))
print("Common:", len(common_now))
print("Missing:", len(missing_now))
print("Extra:", len(extra_now))

print("\nMISSING IDs:")
for x in missing_now:
    print(repr(x))

print("\nEXTRA IDs:")
for x in extra_now:
    print(repr(x))

DIRECT TRAIN FEATURE FILE CHECK
Train IDs: 432
Actual .params IDs: 432
Common: 420
Missing: 12
Extra: 12

MISSING IDs:
'80sTOYS105_2'
'80sTOYS118_2'
'80sTOYS127_2'
'80sTOYS128_2'
'80sTOYS129_2'
'80sTOYS132_2'
'80sTOYS134_2'
'80sTOYS24_2'
'80sTOYS2_2'
'80sTOYS30_2'
'80sTOYS36_2'
'80sTOYS96_2'

EXTRA IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"
"80'sTOYS36_2"
"80'sTOYS96_2"


In [119]:
# ============================================================
# FINAL ROBUST EMOREACT ID NORMALIZATION
# ====================================
========================

import re

def canonical_emoreact_id(x):
    x = str(x).strip()

    # Remove filename extensions first
    extensions = [
        ".params.txt",
        ".params",
        ".au.txt",
        ".au",
        ".gaze.txt",
        ".gaze",
        ".txt",
        ".mp4"
    ]

    changed = True
    while changed:
        changed = False

        for ext in extensions:
            if x.lower().endswith(ext.lower()):
                x = x[:-len(ext)].strip()
                changed = True

    # Remove ONLY outer quote characters
    # (keep the apostrophe inside 80'sTOYS)
    x = x.strip().strip("\"'`“”‘’").strip()

    return x


# ------------------------------------------------------------
# 1. CLEAN TRAIN NAME IDS
# ------------------------------------------------------------

train_ids_final = set(
    canonical_emoreact_id(x)
    for x in train_names_raw
)


# ------------------------------------------------------------
# 2. GET ALL .params FILE IDS
# ------------------------------------------------------------

train_params_files_final = [
    f for f in os.listdir(TRAIN_FEAT)
    if f.lower().endswith(".params.txt")
]

feature_ids_final = set(
    canonical_emoreact_id(f)
    for f in train_params_files_final
)


# ------------------------------------------------------------
# 3. COMPARE
# ------------------------------------------------------------

common_ids_final = train_ids_final & feature_ids_final
missing_ids_final = train_ids_final - feature_ids_final
extra_ids_final = feature_ids_final - train_ids_final


print("=" * 65)
print("FINAL EMOREACT TRAIN ID MATCH")
print("=" * 65)

print("Train IDs:", len(train_ids_final))
print("Feature IDs:", len(feature_ids_final))
print("Common IDs:", len(common_ids_final))
print("Missing IDs:", len(missing_ids_final))
print("Extra IDs:", len(extra_ids_final))


# ------------------------------------------------------------
# 4. SHOW FIRST FEW CLEAN IDS
# ------------------------------------------------------------

print("\nFirst 10 TRAIN IDs:")
for x in sorted(train_ids_final)[:10]:
    print(repr(x))

print("\nFirst 10 FEATURE IDs:")
for x in sorted(feature_ids_final)[:10]:
    print(repr(x))


# ------------------------------------------------------------
# 5. SHOW PROBLEMS ONLY
# ------------------------------------------------------------

print("\nMissing IDs:")
for x in sorted(missing_ids_final):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(extra_ids_final):
    print(repr(x))

FINAL EMOREACT TRAIN ID MATCH
Train IDs: 432
Feature IDs: 432
Common IDs: 0
Missing IDs: 432
Extra IDs: 432

First 10 TRAIN IDs:
"80''sTOYS105_2.mp4"
"80''sTOYS118_2.mp4"
"80''sTOYS127_2.mp4"
"80''sTOYS128_2.mp4"
"80''sTOYS129_2.mp4"
"80''sTOYS132_2.mp4"
"80''sTOYS134_2.mp4"
"80''sTOYS24_2.mp4"
"80''sTOYS2_2.mp4"
"80''sTOYS30_2.mp4"

First 10 FEATURE IDs:
"80'sTOYS105_2"
"80'sTOYS118_2"
"80'sTOYS127_2"
"80'sTOYS128_2"
"80'sTOYS129_2"
"80'sTOYS132_2"
"80'sTOYS134_2"
"80'sTOYS24_2"
"80'sTOYS2_2"
"80'sTOYS30_2"

Missing IDs:
"80''sTOYS105_2.mp4"
"80''sTOYS118_2.mp4"
"80''sTOYS127_2.mp4"
"80''sTOYS128_2.mp4"
"80''sTOYS129_2.mp4"
"80''sTOYS132_2.mp4"
"80''sTOYS134_2.mp4"
"80''sTOYS24_2.mp4"
"80''sTOYS2_2.mp4"
"80''sTOYS30_2.mp4"
"80''sTOYS36_2.mp4"
"80''sTOYS96_2.mp4"
'ANNOYING_ORANGE105_2.mp4'
'ANNOYING_ORANGE119_2.mp4'
'ANNOYING_ORANGE11_2.mp4'
'ANNOYING_ORANGE123_2.mp4'
'ANNOYING_ORANGE126_2.mp4'
'ANNOYING_ORANGE155_2.mp4'
'ANNOYING_ORANGE157_2.mp4'
'ANNOYING_ORANGE159_2.mp4'
'ANNOYING_O

In [120]:
# ============================================================
# DEFINITIVE EMOREACT ID NORMALIZATION
# Remove apostrophe differences
# ============================================================

def final_emoreact_id(x):
    x = str(x).strip()

    # Remove file extensions
    for ext in [
        ".params.txt",
        ".params",
        ".au.txt",
        ".au",
        ".gaze.txt",
        ".gaze",
        ".txt",
        ".mp4"
    ]:
        if x.lower().endswith(ext):
            x = x[:-len(ext)]

    # IMPORTANT:
    # Remove ALL apostrophe / quote characters
    # because Train uses 80sTOYS...
    # while feature files use 80'sTOYS...
    for q in ["'", '"', "`", "“", "”", "‘", "’"]:
        x = x.replace(q, "")

    # Remove spaces
    x = x.replace(" ", "")

    return x


# ------------------------------------------------------------
# TRAIN IDS
# ------------------------------------------------------------

final_train_ids = set(
    final_emoreact_id(x)
    for x in train_names_raw
)


# ------------------------------------------------------------
# FEATURE IDS
# ------------------------------------------------------------

params_files = [
    f for f in os.listdir(TRAIN_FEAT)
    if f.lower().endswith(".params.txt")
]

final_feature_ids = set(
    final_emoreact_id(f)
    for f in params_files
)


# ------------------------------------------------------------
# MATCH
# ------------------------------------------------------------

final_common = final_train_ids & final_feature_ids
final_missing = final_train_ids - final_feature_ids
final_extra = final_feature_ids - final_train_ids


print("=" * 65)
print("DEFINITIVE EMOREACT TRAIN ID MATCH")
print("=" * 65)

print("Train IDs:", len(final_train_ids))
print("Feature IDs:", len(final_feature_ids))
print("Common IDs:", len(final_common))
print("Missing IDs:", len(final_missing))
print("Extra IDs:", len(final_extra))


print("\nMissing IDs:")
for x in sorted(final_missing):
    print(repr(x))

print("\nExtra IDs:")
for x in sorted(final_extra):
    print(repr(x))

DEFINITIVE EMOREACT TRAIN ID MATCH
Train IDs: 432
Feature IDs: 432
Common IDs: 0
Missing IDs: 432
Extra IDs: 432

Missing IDs:
'80sTOYS105_2.mp4'
'80sTOYS118_2.mp4'
'80sTOYS127_2.mp4'
'80sTOYS128_2.mp4'
'80sTOYS129_2.mp4'
'80sTOYS132_2.mp4'
'80sTOYS134_2.mp4'
'80sTOYS24_2.mp4'
'80sTOYS2_2.mp4'
'80sTOYS30_2.mp4'
'80sTOYS36_2.mp4'
'80sTOYS96_2.mp4'
'ANNOYING_ORANGE105_2.mp4'
'ANNOYING_ORANGE119_2.mp4'
'ANNOYING_ORANGE11_2.mp4'
'ANNOYING_ORANGE123_2.mp4'
'ANNOYING_ORANGE126_2.mp4'
'ANNOYING_ORANGE155_2.mp4'
'ANNOYING_ORANGE157_2.mp4'
'ANNOYING_ORANGE159_2.mp4'
'ANNOYING_ORANGE162_2.mp4'
'ANNOYING_ORANGE166_2.mp4'
'ANNOYING_ORANGE169_2.mp4'
'ANNOYING_ORANGE171_2.mp4'
'ANNOYING_ORANGE172_2.mp4'
'ANNOYING_ORANGE174_2.mp4'
'ANNOYING_ORANGE175_2.mp4'
'ANNOYING_ORANGE30_2.mp4'
'ANNOYING_ORANGE39_2.mp4'
'ANNOYING_ORANGE72_2.mp4'
'ANNOYING_ORANGE73_2.mp4'
'ANNOYING_ORANGE93_2.mp4'
'APPLEWATCH16_2.mp4'
'APPLEWATCH18_2.mp4'
'APPLEWATCH19_2.mp4'
'APPLEWATCH2_2.mp4'
'APPLEWATCH31_2.mp4'
'APPLEWATCH4_

In [121]:
# ============================================================
# FINAL CHECK + BUILD CLEAN TRAIN VISUAL DATASET
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Get the 432 official training video IDs
# ------------------------------------------------------------

train_ids = list(clean_train_names)

# Remove accidental duplicates
train_ids = list(dict.fromkeys(train_ids))

print("Official TRAIN IDs:", len(train_ids))


# ------------------------------------------------------------
# 2. Get IDs that actually have feature files
# ------------------------------------------------------------

available_feature_ids = set()

for f in os.listdir(TRAIN_FEAT):

    if f.endswith(".params.txt"):
        video_id = f[:-len(".params.txt")]
        available_feature_ids.add(video_id)

print("Available PARAMS feature IDs:", len(available_feature_ids))


# ------------------------------------------------------------
# 3. Find exact missing IDs
# ------------------------------------------------------------

missing_ids = sorted(set(train_ids) - available_feature_ids)
matched_ids = sorted(set(train_ids) & available_feature_ids)

print("\n==============================")
print("FINAL TRAIN FEATURE CHECK")
print("==============================")

print("Train IDs:", len(train_ids))
print("Matched IDs:", len(matched_ids))
print("Missing IDs:", len(missing_ids))

print("\nMissing IDs:")
for x in missing_ids:
    print(repr(x))


# ------------------------------------------------------------
# 4. Build dataset ONLY from available matched IDs
# ------------------------------------------------------------

rows = []
errors = []

for video_id in matched_ids:

    try:
        row = aggregate_visual_features_v2(video_id)
        rows.append(row)

    except Exception as e:
        errors.append((video_id, str(e)))


# ------------------------------------------------------------
# 5. Convert to DataFrame
# ------------------------------------------------------------

train_visual_features = pd.DataFrame(rows)

print("\n==============================")
print("TRAIN VISUAL FEATURE DATASET")
print("==============================")

print("Shape:", train_visual_features.shape)
print("Expected available videos:", len(matched_ids))
print("Successful rows:", len(train_visual_features))
print("Errors:", len(errors))

if errors:
    print("\nFirst errors:")
    for item in errors[:10]:
        print(item)


# ------------------------------------------------------------
# 6. Final sanity check
# ------------------------------------------------------------

if len(train_visual_features) > 0:

    print("\nFirst 5 rows:")
    display(train_visual_features.head())

    print("\nMissing values:")
    print(train_visual_features.isna().sum().sum())

    print("\nDuplicate video IDs:")
    print(
        train_visual_features["video_id"].duplicated().sum()
        if "video_id" in train_visual_features.columns
        else "video_id column not found"
    )

Official TRAIN IDs: 432
Available PARAMS feature IDs: 432

FINAL TRAIN FEATURE CHECK
Train IDs: 432
Matched IDs: 420
Missing IDs: 12

Missing IDs:
"80''sTOYS105_2"
"80''sTOYS118_2"
"80''sTOYS127_2"
"80''sTOYS128_2"
"80''sTOYS129_2"
"80''sTOYS132_2"
"80''sTOYS134_2"
"80''sTOYS24_2"
"80''sTOYS2_2"
"80''sTOYS30_2"
"80''sTOYS36_2"
"80''sTOYS96_2"

TRAIN VISUAL FEATURE DATASET
Shape: (0, 0)
Expected available videos: 420
Successful rows: 0
Errors: 420

First errors:
('ANNOYING_ORANGE105_2', "aggregate_visual_features_v2() missing 1 required positional argument: 'feature_dir'")
('ANNOYING_ORANGE119_2', "aggregate_visual_features_v2() missing 1 required positional argument: 'feature_dir'")
('ANNOYING_ORANGE11_2', "aggregate_visual_features_v2() missing 1 required positional argument: 'feature_dir'")
('ANNOYING_ORANGE123_2', "aggregate_visual_features_v2() missing 1 required positional argument: 'feature_dir'")
('ANNOYING_ORANGE126_2', "aggregate_visual_features_v2() missing 1 required positio

In [122]:
# ============================================
# FIX: CALL aggregate_visual_features_v2 CORRECTLY
# ============================================

train_results = []

for video_id in clean_train_names:

    try:
        features = aggregate_visual_features_v2(
            video_id,
            TRAIN_FEAT
        )

        train_results.append(features)

    except Exception as e:
        print(f"Error processing {video_id}: {e}")

# Create DataFrame
train_visual_features = pd.DataFrame(train_results)

print("\n============================================")
print("TRAIN VISUAL FEATURE DATASET")
print("============================================")

print("Shape:", train_visual_features.shape)
print("Expected rows:", len(clean_train_names))
print("Actual rows:", len(train_visual_features))

display(train_visual_features.head())

Error processing 80''sTOYS105_2: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS105_2.params.txt
Error processing 80''sTOYS118_2: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS118_2.params.txt
Error processing 80''sTOYS127_2: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS127_2.params.txt
Error processing 80''sTOYS128_2: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS128_2.params.txt
Error processing 80''sTOYS129_2: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS129_2.params.txt
Error processing 80''sTOYS132_2: Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_

,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,ANNOYING_ORANGE105_2,1.835167,1.077577,0.969798,0.004329,1.0,0.0,1.273815,0.027521,0.269296,...,0.542937,0.054635,-0.552933,0.046337,0.498727,0.058552,0.467983,0.046922,-0.724690,0.039234
1,ANNOYING_ORANGE119_2,1.647479,0.969214,0.941390,0.039892,1.0,0.0,1.245697,0.029645,0.066726,...,0.324343,0.034190,-0.903425,0.019423,0.055097,0.041188,0.324100,0.034777,-0.942812,0.012721
2,ANNOYING_ORANGE11_2,3.524354,2.052838,0.981597,0.015345,1.0,0.0,1.623457,0.032629,-0.029637,...,0.255915,0.064985,-0.941921,0.021662,-0.442279,0.072558,0.177775,0.091807,-0.870970,0.024252
3,ANNOYING_ORANGE123_2,1.584917,0.933093,0.974706,0.004804,1.0,0.0,1.792782,0.038580,0.253426,...,0.376226,0.094453,-0.890915,0.028635,0.008951,0.055033,0.365244,0.135555,-0.918104,0.050078
4,ANNOYING_ORANGE126_2,2.523354,1.474907,0.976877,0.006221,1.0,0.0,1.196154,0.018492,-0.039224,...,0.230680,0.046296,-0.970785,0.010848,-0.071258,0.029203,0.213486,0.039115,-0.973094,0.008390


In [123]:
# Check the ACTUAL filenames for one missing video

target = "80'sTOYS105_2"

print("Files containing TOYS105:")
for f in os.listdir(TRAIN_FEAT):
    if "TOYS105" in f:
        print(repr(f))

Files containing TOYS105:
"80'sTOYS105_2.params.txt"
"80'sTOYS105_2_gaze.txt"
"80'sTOYS105_2_au.txt"


In [124]:
# ============================================
# FINAL FIX: NORMALIZE VIDEO IDs BEFORE READING
# ============================================

def aggregate_visual_features_v3(video_id, feature_dir):

    # Clean the ID
    video_id = str(video_id).strip()

    # Remove accidental surrounding quotes
    video_id = video_id.strip('"').strip("'")

    # Build exact file paths
    params_path = os.path.join(
        feature_dir,
        video_id + ".params.txt"
    )

    au_path = os.path.join(
        feature_dir,
        video_id + "_au.txt"
    )

    gaze_path = os.path.join(
        feature_dir,
        video_id + "_gaze.txt"
    )

    # Check files
    if not os.path.exists(params_path):
        raise FileNotFoundError(f"Missing params file: {params_path}")

    if not os.path.exists(au_path):
        raise FileNotFoundError(f"Missing AU file: {au_path}")

    if not os.path.exists(gaze_path):
        raise FileNotFoundError(f"Missing gaze file: {gaze_path}")

    # Read files
    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # ============================================
    # PARAMS
    # ============================================

    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:

        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # ============================================
    # AU
    # ============================================

    au_numeric = au.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in au_numeric.columns:

        values = pd.to_numeric(
            au_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # ============================================
    # GAZE
    # ============================================

    gaze_numeric = gaze.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in gaze_numeric.columns:

        values = pd.to_numeric(
            gaze_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    return features


print("aggregate_visual_features_v3 created successfully!")

aggregate_visual_features_v3 created successfully!


In [125]:
# ============================================
# EXTRACT TRAIN FEATURES USING V3
# ============================================

train_results = []
train_errors = []

for i, video_id in enumerate(clean_train_names):

    try:
        features = aggregate_visual_features_v3(
            video_id,
            TRAIN_FEAT
        )

        train_results.append(features)

    except Exception as e:
        train_errors.append((video_id, str(e)))

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(clean_train_names)}")


# Create DataFrame
train_visual_features = pd.DataFrame(train_results)

print("\n============================================")
print("TRAIN VISUAL FEATURE DATASET")
print("============================================")

print("Expected rows :", len(clean_train_names))
print("Successful rows:", len(train_visual_features))
print("Errors         :", len(train_errors))
print("Shape          :", train_visual_features.shape)

if train_errors:
    print("\nFIRST ERRORS:")
    for item in train_errors[:15]:
        print(item)

print("\nFirst 5 rows:")
display(train_visual_features.head())

Processed 50/432
Processed 100/432
Processed 150/432
Processed 200/432
Processed 250/432
Processed 300/432
Processed 350/432
Processed 400/432

TRAIN VISUAL FEATURE DATASET
Expected rows : 432
Successful rows: 420
Errors         : 12
Shape          : (420, 151)

FIRST ERRORS:
("80''sTOYS105_2", "Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS105_2.params.txt")
("80''sTOYS118_2", "Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS118_2.params.txt")
("80''sTOYS127_2", "Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS127_2.params.txt")
("80''sTOYS128_2", "Missing params file: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Train_feat/80''sTOYS128_2.params.txt")
("80''sTOYS129_2", "Missing params file: /content/d

,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,ANNOYING_ORANGE105_2,1.835167,1.077577,0.969798,0.004329,1.0,0.0,1.273815,0.027521,0.269296,...,0.542937,0.054635,-0.552933,0.046337,0.498727,0.058552,0.467983,0.046922,-0.724690,0.039234
1,ANNOYING_ORANGE119_2,1.647479,0.969214,0.941390,0.039892,1.0,0.0,1.245697,0.029645,0.066726,...,0.324343,0.034190,-0.903425,0.019423,0.055097,0.041188,0.324100,0.034777,-0.942812,0.012721
2,ANNOYING_ORANGE11_2,3.524354,2.052838,0.981597,0.015345,1.0,0.0,1.623457,0.032629,-0.029637,...,0.255915,0.064985,-0.941921,0.021662,-0.442279,0.072558,0.177775,0.091807,-0.870970,0.024252
3,ANNOYING_ORANGE123_2,1.584917,0.933093,0.974706,0.004804,1.0,0.0,1.792782,0.038580,0.253426,...,0.376226,0.094453,-0.890915,0.028635,0.008951,0.055033,0.365244,0.135555,-0.918104,0.050078
4,ANNOYING_ORANGE126_2,2.523354,1.474907,0.976877,0.006221,1.0,0.0,1.196154,0.018492,-0.039224,...,0.230680,0.046296,-0.970785,0.010848,-0.071258,0.029203,0.213486,0.039115,-0.973094,0.008390


In [126]:
# ============================================================
# ROBUST FEATURE EXTRACTION - USE ACTUAL FILENAMES
# ============================================================

def normalize_id(x):
    """
    Normalize IDs only for matching.
    Removes quotes, spaces and other non-alphanumeric characters.
    """
    return ''.join(
        c.lower()
        for c in str(x)
        if c.isalnum()
    )


# Get ACTUAL filenames from the folder
actual_files = os.listdir(TRAIN_FEAT)

# Create lookup using normalized filename stems
file_lookup = {}

for f in actual_files:
    if f.endswith(".params.txt"):
        stem = f[:-len(".params.txt")]
        file_lookup[normalize_id(stem)] = f


print("Actual params files found:", len(file_lookup))


def aggregate_visual_features_v4(video_id, feature_dir):

    video_id = str(video_id).strip()

    # Normalize only for matching
    key = normalize_id(video_id)

    if key not in file_lookup:
        raise FileNotFoundError(
            f"No matching params file found for: {video_id}"
        )

    # Get the REAL filename from the folder
    params_file = file_lookup[key]

    # Corresponding AU and GAZE filenames
    params_path = os.path.join(
        feature_dir,
        params_file
    )

    base_name = params_file[:-len(".params.txt")]

    au_path = os.path.join(
        feature_dir,
        base_name + "_au.txt"
    )

    gaze_path = os.path.join(
        feature_dir,
        base_name + "_gaze.txt"
    )

    # Safety checks
    if not os.path.exists(au_path):
        raise FileNotFoundError(
            f"Missing AU file: {au_path}"
        )

    if not os.path.exists(gaze_path):
        raise FileNotFoundError(
            f"Missing gaze file: {gaze_path}"
        )

    # Read files
    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # ========================================================
    # PARAMS
    # ========================================================

    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:

        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # ========================================================
    # AU
    # ========================================================

    au_numeric = au.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in au_numeric.columns:

        values = pd.to_numeric(
            au_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # ========================================================
    # GAZE
    # ========================================================

    gaze_numeric = gaze.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in gaze_numeric.columns:

        values = pd.to_numeric(
            gaze_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    return features


print("aggregate_visual_features_v4 created successfully!")


Actual params files found: 432
aggregate_visual_features_v4 created successfully!


In [127]:
# ============================================================
# EXTRACT ALL 432 TRAIN VIDEOS
# ============================================================

train_results = []
train_errors = []

for i, video_id in enumerate(clean_train_names):

    try:
        features = aggregate_visual_features_v4(
            video_id,
            TRAIN_FEAT
        )

        train_results.append(features)

    except Exception as e:
        train_errors.append((video_id, str(e)))

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(clean_train_names)}")


train_visual_features = pd.DataFrame(train_results)

print("\n============================================")
print("FINAL TRAIN VISUAL FEATURE DATASET")
print("============================================")

print("Expected rows :", len(clean_train_names))
print("Successful rows:", len(train_visual_features))
print("Errors         :", len(train_errors))
print("Shape          :", train_visual_features.shape)

if train_errors:
    print("\nERRORS:")
    for item in train_errors:
        print(item)

print("\nFirst 5 rows:")
display(train_visual_features.head())

Processed 50/432
Processed 100/432
Processed 150/432
Processed 200/432
Processed 250/432
Processed 300/432
Processed 350/432
Processed 400/432

FINAL TRAIN VISUAL FEATURE DATASET
Expected rows : 432
Successful rows: 432
Errors         : 0
Shape          : (432, 151)

First 5 rows:


,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,80''sTOYS105_2,2.147979,1.258182,0.970218,0.003600,1.0,0.0,1.460534,0.024545,0.190502,...,0.398612,0.082259,-0.895826,0.034312,-0.108495,0.033026,0.299370,0.083425,-0.943173,0.032689
1,80''sTOYS118_2,4.984146,2.895652,0.954697,0.022784,1.0,0.0,1.249333,0.054770,0.092727,...,0.321813,0.092623,-0.897008,0.092924,-0.056610,0.235858,0.297562,0.080987,-0.918309,0.055178
2,80''sTOYS127_2,1.897729,1.113699,0.981012,0.008426,1.0,0.0,1.620695,0.022446,0.105488,...,0.269444,0.080947,-0.929847,0.028388,-0.363157,0.094172,0.421468,0.088916,-0.818253,0.066337
3,80''sTOYS128_2,2.314813,1.354504,0.965176,0.013088,1.0,0.0,1.436963,0.020159,0.073711,...,0.253662,0.090014,-0.887938,0.089816,0.056984,0.226363,0.237928,0.104632,-0.936400,0.040982
4,80''sTOYS129_2,1.835167,1.077577,0.984900,0.009282,1.0,0.0,1.266610,0.025625,0.145561,...,0.224451,0.039168,-0.937590,0.052074,-0.179699,0.245991,0.225923,0.046226,-0.924283,0.020695


In [128]:
# ============================================================
# SAVE FINAL TRAIN VISUAL FEATURES
# ============================================================

TRAIN_VISUAL_CSV = os.path.join(
    EMOREACT_PATH,
    "train_visual_features.csv"
)

train_visual_features.to_csv(
    TRAIN_VISUAL_CSV,
    index=False
)

print("Saved successfully!")
print("Path:", TRAIN_VISUAL_CSV)
print("Shape:", train_visual_features.shape)
print("Columns:", len(train_visual_features.columns))

Saved successfully!
Path: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/train_visual_features.csv
Shape: (432, 151)
Columns: 151


In [129]:
# ============================================================
# EMOREACT - VALIDATION VISUAL FEATURE EXTRACTION
# ============================================================

import os
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------
# 1. Validation feature folder
# ------------------------------------------------------------

VAL_FEAT = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Val_feat"
)

print("Validation feature folder:")
print(VAL_FEAT)
print("Folder exists:", os.path.exists(VAL_FEAT))


# ------------------------------------------------------------
# 2. Check validation names
# ------------------------------------------------------------

print("\nValidation videos:", len(clean_val_names))
print("First 10 validation IDs:")

for x in clean_val_names[:10]:
    print(repr(x))


# ------------------------------------------------------------
# 3. Check actual .params files
# ------------------------------------------------------------

params_files = [
    f for f in os.listdir(VAL_FEAT)
    if f.lower().endswith("_params.txt")
    or f.lower().endswith(".params.txt")
]

print("\nActual validation params files:", len(params_files))


# ------------------------------------------------------------
# 4. Extract validation visual features
# ------------------------------------------------------------

val_results = []
val_errors = []

total = len(clean_val_names)

for i, video_id in enumerate(clean_val_names, start=1):

    try:
        result = aggregate_visual_features_v4(
            video_id,
            VAL_FEAT
        )

        val_results.append(result)

    except Exception as e:
        val_errors.append(
            (video_id, str(e))
        )

    if i % 50 == 0 or i == total:
        print(f"Processed {i}/{total}")


# ------------------------------------------------------------
# 5. Create validation dataframe
# ------------------------------------------------------------

val_visual_features = pd.DataFrame(val_results)

print("\n" + "=" * 55)
print("FINAL VALIDATION VISUAL FEATURE DATASET")
print("=" * 55)

print("Expected rows :", len(clean_val_names))
print("Successful rows:", len(val_visual_features))
print("Errors         :", len(val_errors))
print("Shape          :", val_visual_features.shape)


# ------------------------------------------------------------
# 6. Show errors if any
# ------------------------------------------------------------

if len(val_errors) > 0:

    print("\nFIRST ERRORS:")

    for err in val_errors[:20]:
        print(err)

else:
    print("\nNo errors! All validation files processed successfully.")


# ------------------------------------------------------------
# 7. Preview
# ------------------------------------------------------------

print("\nFirst 5 rows:")

display(val_visual_features.head())


# ------------------------------------------------------------
# 8. Save validation features
# ------------------------------------------------------------

VAL_OUTPUT = os.path.join(
    EMOREACT_PATH,
    "val_visual_features.csv"
)

val_visual_features.to_csv(
    VAL_OUTPUT,
    index=False
)

print("\nSaved successfully!")
print("Path:", VAL_OUTPUT)
print("Shape:", val_visual_features.shape)
print("Columns:", len(val_visual_features.columns))

Validation feature folder:
/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Val_feat
Folder exists: True

Validation videos: 303
First 10 validation IDs:
'80sTOYS115_2'
'80sTOYS119_2'
'80sTOYS120_2'
'80sTOYS126_2'
'80sTOYS133_2'
'80sTOYS16_2'
'80sTOYS27_2'
'80sTOYS39_2'
'80sTOYS40_2'
'ANNOYING_ORANGE137_2'

Actual validation params files: 303
Processed 50/303
Processed 100/303
Processed 150/303
Processed 200/303
Processed 250/303
Processed 300/303
Processed 303/303

FINAL VALIDATION VISUAL FEATURE DATASET
Expected rows : 303
Successful rows: 0
Errors         : 303
Shape          : (0, 0)

FIRST ERRORS:
('80sTOYS115_2', 'No matching params file found for: 80sTOYS115_2')
('80sTOYS119_2', 'No matching params file found for: 80sTOYS119_2')
('80sTOYS120_2', 'No matching params file found for: 80sTOYS120_2')
('80sTOYS126_2', 'No matching params file found for: 80sTOYS126_2')
('80sTOYS133_2', 'No matching params file found for: 80sTOYS133_2')
('80sTOYS16

""



Saved successfully!
Path: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/val_visual_features.csv
Shape: (0, 0)
Columns: 0


In [130]:
# ============================================================
# FIX VALIDATION FILE MATCHING
# ============================================================

import os
import re

def normalize_emoreact_id(x):
    """
    Normalize EmoReact video IDs so that
    apostrophes/quotes do not break filename matching.
    """
    x = str(x).strip()

    # Remove surrounding quotes if present
    x = x.strip('"').strip("'")

    # Normalize whitespace
    x = re.sub(r"\s+", "", x)

    return x


def find_validation_feature_file(video_id, suffix):
    """
    Find the actual validation feature file by comparing
    normalized IDs instead of assuming an exact filename.
    """

    target = normalize_emoreact_id(video_id)

    files = os.listdir(VAL_FEAT)

    # Possible file extensions
    candidates = []

    for f in files:

        fl = f.lower()

        # We only need the requested feature type
        if not fl.endswith(suffix.lower()):
            continue

        # Remove extension
        base = f

        if base.lower().endswith(suffix.lower()):
            base = base[:-len(suffix)]

        base = normalize_emoreact_id(base)

        if base == target:
            candidates.append(f)

    if len(candidates) == 0:
        return None

    return os.path.join(VAL_FEAT, candidates[0])


# ------------------------------------------------------------
# TEST ONE VALIDATION VIDEO
# ------------------------------------------------------------

test_id = clean_val_names[0]

print("Test video ID:")
print(repr(test_id))

print("\nSearching files...")

params_file = find_validation_feature_file(
    test_id,
    ".params.txt"
)

au_file = find_validation_feature_file(
    test_id,
    "_au.txt"
)

gaze_file = find_validation_feature_file(
    test_id,
    "_gaze.txt"
)

print("\nPARAMS:")
print(params_file)

print("\nAU:")
print(au_file)

print("\nGAZE:")
print(gaze_file)

Test video ID:
'80sTOYS115_2'

Searching files...

PARAMS:
None

AU:
None

GAZE:
None


In [131]:
# ============================================================
# CHECK ACTUAL VALIDATION FILENAMES
# ============================================================

print("VAL_FEAT:")
print(VAL_FEAT)

print("\nFiles containing TOYS115:")
toys_files = []

for f in os.listdir(VAL_FEAT):
    if "TOYS115" in f.upper():
        toys_files.append(f)

for f in sorted(toys_files):
    print(repr(f))

print("\nTotal matching files:", len(toys_files))

VAL_FEAT:
/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Val_feat

Files containing TOYS115:
"80'sTOYS115_2.params.txt"
"80'sTOYS115_2_au.txt"
"80'sTOYS115_2_gaze.txt"

Total matching files: 3


In [132]:
# ============================================================
# EMOREACT VALIDATION - FINAL ROBUST EXTRACTION
# ============================================================

import os
import pandas as pd
import numpy as np
from IPython.display import display


def normalize_id(x):
    """
    Normalize video ID only for matching by removing any internal
    and external apostrophes, quotes, and whitespaces.
    """
    x = str(x).strip()
    for q in ["'", '"', "`", "“", "”", "‘", "’"]:
        x = x.replace(q, "")
    return x.replace(" ", "")


def get_actual_file(video_id, file_type):
    """
    Find the real file from Val_feat directory.
    """

    target = normalize_id(video_id)

    for f in os.listdir(VAL_FEAT):

        if file_type == "params":
            if not f.endswith(".params.txt"):
                continue

        elif file_type == "au":
            if not f.endswith("_au.txt"):
                continue

        elif file_type == "gaze":
            if not f.endswith("_gaze.txt"):
                continue

        # Remove suffix to obtain ID
        if file_type == "params":
            file_id = f[:-len(".params.txt")]

        elif file_type == "au":
            file_id = f[:-len("_au.txt")]

        elif file_type == "gaze":
            file_id = f[:-len("_gaze.txt")]

        file_id = normalize_id(file_id)

        if file_id == target:
            return os.path.join(VAL_FEAT, f)

    return None


def aggregate_validation_features(video_id):

    params_path = get_actual_file(video_id, "params")
    au_path = get_actual_file(video_id, "au")
    gaze_path = get_actual_file(video_id, "gaze")

    if params_path is None:
        raise FileNotFoundError(
            f"PARAMS file not found for {video_id}"
        )

    if au_path is None:
        raise FileNotFoundError(
            f"AU file not found for {video_id}"
        )

    if gaze_path is None:
        raise FileNotFoundError(
            f"GAZE file not found for {video_id}"
        )

    # --------------------------------------------------------
    # Read files
    # --------------------------------------------------------

    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # --------------------------------------------------------
    # PARAMS
    # --------------------------------------------------------

    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:

        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # --------------------------------------------------------
    # AU
    # --------------------------------------------------------

    au_numeric = au.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in au_numeric.columns:

        values = pd.to_numeric(
            au_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # --------------------------------------------------------
    # GAZE
    # --------------------------------------------------------

    gaze_numeric = gaze.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in gaze_numeric.columns:

        values = pd.to_numeric(
            gaze_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    return features


# ============================================================
# TEST FIRST
# ============================================================

test_id = clean_val_names[0]

print("Testing:", repr(test_id))

test_result = aggregate_validation_features(
    test_id
)

print("Test successful!")
print("Number of features:", len(test_result))

print("\nFirst 10 features:")

for k, v in list(test_result.items())[:10]:
    print(k, "=", v)


# ============================================================
# PROCESS ALL 303 VALIDATION VIDEOS
# ============================================================

val_results = []
val_errors = []

total = len(clean_val_names)

for i, video_id in enumerate(clean_val_names, start=1):

    try:

        result = aggregate_validation_features(
            video_id
        )

        val_results.append(result)

    except Exception as e:

        val_errors.append(
            (video_id, str(e))
        )

    if i % 50 == 0 or i == total:
        print(f"Processed {i}/{total}")


# ============================================================
# FINAL DATASET
# ============================================================

val_visual_features = pd.DataFrame(
    val_results
)

print("\n" + "=" * 60)
print("FINAL VALIDATION VISUAL FEATURE DATASET")
print("=" * 60)

print("Expected rows :", len(clean_val_names))
print("Successful rows:", len(val_visual_features))
print("Errors         :", len(val_errors))
print("Shape          :", val_visual_features.shape)


# ============================================================
# ERRORS
# ============================================================

if val_errors:

    print("\nFIRST ERRORS:")

    for error in val_errors[:20]:
        print(error)

else:

    print("\nALL 303 VALIDATION VIDEOS PROCESSED SUCCESSFULLY!")


# ============================================================
# PREVIEW
# ============================================================

display(
    val_visual_features.head()
)


# ============================================================
# SAVE
# ============================================================

VAL_OUTPUT = os.path.join(
    EMOREACT_PATH,
    "val_visual_features.csv"
)

val_visual_features.to_csv(
    VAL_OUTPUT,
    index=False
)

print("\nSaved successfully!")
print("Path:", VAL_OUTPUT)
print("Shape:", val_visual_features.shape)
print("Columns:", len(val_visual_features.columns))

Testing: '80sTOYS115_2'
Test successful!
Number of features: 151

First 10 features:
video_id = 80sTOYS115_2
 timestamp_mean = 3.2949583647798746
 timestamp_std = 1.920395793920963
 confidence_mean = 0.9810200691823899
 confidence_std = 0.004669830772581659
 success_mean = 1.0
 success_std = 0.0
 scale_mean = 1.3354757861635218
 scale_std = 0.02633261760351066
 rx_mean = 0.04492768457861635
Processed 50/303
Processed 100/303
Processed 150/303
Processed 200/303
Processed 250/303
Processed 300/303
Processed 303/303

FINAL VALIDATION VISUAL FEATURE DATASET
Expected rows : 303
Successful rows: 303
Errors         : 0
Shape          : (303, 151)

ALL 303 VALIDATION VIDEOS PROCESSED SUCCESSFULLY!


,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,80sTOYS115_2,3.294958,1.920396,0.981020,0.004670,1.0,0.0,1.335476,0.026333,0.044928,...,0.234017,0.097895,-0.859555,0.083028,0.259845,0.154880,0.259541,0.085078,-0.911344,0.059698
1,80sTOYS119_2,2.690188,1.571229,0.985106,0.003668,1.0,0.0,1.300640,0.015539,0.020175,...,0.225071,0.075401,-0.790817,0.087345,0.397534,0.116721,0.263768,0.044335,-0.867741,0.063094
2,80sTOYS120_2,2.481646,1.450827,0.931419,0.040345,1.0,0.0,1.504722,0.047561,-0.347563,...,-0.159514,0.296050,-0.925442,0.058506,-0.124381,0.226856,-0.177033,0.349942,-0.881115,0.066234
3,80sTOYS126_2,6.464792,3.750505,0.976011,0.011059,1.0,0.0,1.379093,0.038091,-0.043010,...,0.156790,0.099751,-0.970930,0.021199,-0.331723,0.075192,0.110506,0.097566,-0.928322,0.029082
4,80sTOYS133_2,5.359520,3.112374,0.987025,0.002156,1.0,0.0,1.400949,0.004880,0.061888,...,0.262421,0.049904,-0.949806,0.023385,-0.043458,0.098686,0.308553,0.034215,-0.944386,0.013218



Saved successfully!
Path: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/val_visual_features.csv
Shape: (303, 151)
Columns: 151


In [133]:
# ============================================================
# EMOREACT TEST - FINAL ROBUST EXTRACTION
# ============================================================

import os
import pandas as pd
import numpy as np
from IPython.display import display


# ------------------------------------------------------------
# TEST FEATURE FOLDER & NAMES
# ------------------------------------------------------------

TEST_FEAT = os.path.join(
    EMOREACT_PATH,
    "Visual_features",
    "Test_feat"
)

TEST_NAMES_FILE = os.path.join(EMOREACT_PATH, "Test_names.txt")

print("Test feature folder:")
print(TEST_FEAT)
print("Folder exists:", os.path.exists(TEST_FEAT))

# Read and clean test names
with open(TEST_NAMES_FILE, "r", encoding="utf-8") as f:
    test_names_raw = [line.strip() for line in f if line.strip()]

def clean_video_id(x):
    x = str(x).strip()
    x = x.replace('"', '')
    x = x.replace("'", '')
    if x.lower().endswith(".mp4"):
        x = x[:-4]
    return x.strip()

clean_test_names = [
    clean_video_id(x)
    for x in test_names_raw
]

print("\nTest videos loaded:", len(clean_test_names))
print("First 10 TEST IDs:")
for x in clean_test_names[:10]:
    print(repr(x))


# ------------------------------------------------------------
# FIND ACTUAL FILE
# ------------------------------------------------------------

def get_actual_test_file(video_id, file_type):

    target = normalize_id(video_id)

    for f in os.listdir(TEST_FEAT):

        if file_type == "params":
            if not f.endswith(".params.txt"):
                continue
            file_id = f[:-len(".params.txt")]

        elif file_type == "au":
            if not f.endswith("_au.txt"):
                continue
            file_id = f[:-len("_au.txt")]

        elif file_type == "gaze":
            if not f.endswith("_gaze.txt"):
                continue
            file_id = f[:-len("_gaze.txt")]

        else:
            continue

        file_id = normalize_id(file_id)

        if file_id == target:
            return os.path.join(TEST_FEAT, f)

    return None


# ------------------------------------------------------------
# AGGREGATE TEST FEATURES
# ------------------------------------------------------------

def aggregate_test_features(video_id):

    params_path = get_actual_test_file(
        video_id,
        "params"
    )

    au_path = get_actual_test_file(
        video_id,
        "au"
    )

    gaze_path = get_actual_test_file(
        video_id,
        "gaze"
    )

    if params_path is None:
        raise FileNotFoundError(
            f"PARAMS file not found for {video_id}"
        )

    if au_path is None:
        raise FileNotFoundError(
            f"AU file not found for {video_id}"
        )

    if gaze_path is None:
        raise FileNotFoundError(
            f"GAZE file not found for {video_id}"
        )

    # --------------------------------------------------------
    # READ FILES
    # --------------------------------------------------------

    params = pd.read_csv(params_path)
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)

    features = {
        "video_id": video_id
    }

    # --------------------------------------------------------
    # PARAMS
    # --------------------------------------------------------

    params_numeric = params.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in params_numeric.columns:

        values = pd.to_numeric(
            params_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # --------------------------------------------------------
    # AU
    # --------------------------------------------------------

    au_numeric = au.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in au_numeric.columns:

        values = pd.to_numeric(
            au_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    # --------------------------------------------------------
    # GAZE
    # --------------------------------------------------------

    gaze_numeric = gaze.select_dtypes(
        include=np.number
    ).drop(
        columns=["frame", "timestamp"],
        errors="ignore"
    )

    for col in gaze_numeric.columns:

        values = pd.to_numeric(
            gaze_numeric[col],
            errors="coerce"
        )

        values = values.replace(-100, np.nan)

        features[f"{col}_mean"] = values.mean()
        features[f"{col}_std"] = values.std()

    return features


# ============================================================
# TEST ONE VIDEO FIRST
# ============================================================

test_id = clean_test_names[0]

print("\nTesting:", repr(test_id))

test_result = aggregate_test_features(
    test_id
)

print("Test successful!")
print("Number of features:", len(test_result))

print("\nFirst 10 features:")

for k, v in list(test_result.items())[:10]:
    print(k, "=", v)


# ============================================================
# PROCESS ALL TEST VIDEOS
# ============================================================

test_results = []
test_errors = []

total = len(clean_test_names)

for i, video_id in enumerate(
    clean_test_names,
    start=1
):

    try:

        result = aggregate_test_features(
            video_id
        )

        test_results.append(result)

    except Exception as e:

        test_errors.append(
            (video_id, str(e))
        )

    if i % 50 == 0 or i == total:
        print(f"Processed {i}/{total}")


# ============================================================
# FINAL TEST DATASET
# ============================================================

test_visual_features = pd.DataFrame(
    test_results
)

print("\n" + "=" * 60)
print("FINAL TEST VISUAL FEATURE DATASET")
print("=" * 60)

print("Expected rows :", len(clean_test_names))
print("Successful rows:", len(test_visual_features))
print("Errors         :", len(test_errors))
print("Shape          :", test_visual_features.shape)


# ============================================================
# SHOW ERRORS
# ============================================================

if test_errors:

    print("\nFIRST ERRORS:")

    for error in test_errors[:20]:
        print(error)

else:

    print(
        "\nALL TEST VIDEOS PROCESSED SUCCESSFULLY!"
    )


# ============================================================
# PREVIEW
# ============================================================

display(
    test_visual_features.head()
)


# ============================================================
# SAVE
# ============================================================

TEST_OUTPUT = os.path.join(
    EMOREACT_PATH,
    "test_visual_features.csv"
)

test_visual_features.to_csv(
    TEST_OUTPUT,
    index=False
)

print("\nSaved successfully!")
print("Path:", TEST_OUTPUT)
print("Shape:", test_visual_features.shape)
print("Columns:", len(test_visual_features.columns))

Test feature folder:
/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Visual_features/Test_feat
Folder exists: True

Test videos loaded: 367
First 10 TEST IDs:
'80sTOYS104_2'
'80sTOYS123_2'
'80sTOYS130_2'
'80sTOYS139_2'
'80sTOYS28_2'
'80sTOYS32_2'
'80sTOYS49_2'
'80sTOYS4_2'
'80sTOYS9_2'
'ANNOYING_ORANGE120_2'

Testing: '80sTOYS104_2'
Test successful!
Number of features: 151

First 10 features:
video_id = 80sTOYS104_2
 timestamp_mean = 5.547208202247191
 timestamp_std = 3.2207366461357547
 confidence_mean = 0.9448869775280898
 confidence_std = 0.063339965557332
 success_mean = 0.9812734082397003
 success_std = 0.13581232961953854
 scale_mean = 1.4210774157303372
 scale_std = 0.025506582882936557
 rx_mean = 0.05651624204119852
Processed 50/367
Processed 100/367
Processed 150/367
Processed 200/367
Processed 250/367
Processed 300/367
Processed 350/367
Processed 367/367

FINAL TEST VISUAL FEATURE DATASET
Expected rows : 367
Successful rows: 367
Errors         : 0
Shap

,video_id,timestamp_mean,timestamp_std,confidence_mean,confidence_std,success_mean,success_std,scale_mean,scale_std,rx_mean,...,y_h0_mean,y_h0_std,z_h0_mean,z_h0_std,x_h1_mean,x_h1_std,y_h1_mean,y_h1_std,z_h1_mean,z_h1_std
0,80sTOYS104_2,5.547208,3.220737,0.944887,0.063340,0.981273,0.135812,1.421077,0.025507,0.056516,...,0.233791,0.149155,-0.846956,0.190479,0.066712,0.294247,0.303309,0.120551,-0.873613,0.144808
1,80sTOYS123_2,2.314813,1.354504,0.972616,0.015179,1.000000,0.000000,1.355707,0.030685,0.072290,...,0.219195,0.113718,-0.680809,0.165434,0.440054,0.244197,0.243662,0.092201,-0.817225,0.108188
2,80sTOYS130_2,7.132125,4.135790,0.939950,0.051082,1.000000,0.000000,1.420397,0.028075,0.049090,...,0.243955,0.131964,-0.906633,0.122969,-0.126487,0.284380,0.344045,0.100300,-0.877882,0.065595
3,80sTOYS139_2,1.647479,0.969214,0.970425,0.018654,1.000000,0.000000,1.394960,0.025577,0.112284,...,0.386652,0.041529,-0.867356,0.039797,0.024000,0.177545,0.403692,0.051463,-0.895479,0.029000
4,80sTOYS28_2,1.772604,1.041456,0.980791,0.009078,1.000000,0.000000,1.377158,0.038195,0.068681,...,0.220504,0.211112,-0.856568,0.100995,0.104649,0.205957,0.245305,0.202916,-0.918205,0.056479



Saved successfully!
Path: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/test_visual_features.csv
Shape: (367, 151)
Columns: 151


In [134]:
# ============================================================
# FIX TEST IDs - REMOVE .mp4 EXTENSION
# ============================================================

clean_test_names = [
    normalize_id(x).replace(".mp4", "")
    for x in test_names_raw
    if str(x).strip()
]

# Remove duplicates
clean_test_names = list(
    dict.fromkeys(clean_test_names)
)

print("Test videos:", len(clean_test_names))

print("\nFirst 10 CLEAN TEST IDs:")

for x in clean_test_names[:10]:
    print(repr(x))

Test videos: 367

First 10 CLEAN TEST IDs:
'80sTOYS104_2'
'80sTOYS123_2'
'80sTOYS130_2'
'80sTOYS139_2'
'80sTOYS28_2'
'80sTOYS32_2'
'80sTOYS49_2'
'80sTOYS4_2'
'80sTOYS9_2'
'ANNOYING_ORANGE120_2'


In [135]:
# ============================================================
# PROCESS ALL 367 TEST VIDEOS
# ============================================================

test_results = []
test_errors = []

total = len(clean_test_names)

for i, video_id in enumerate(clean_test_names, start=1):

    try:
        result = aggregate_test_visual_features(video_id)
        test_results.append(result)

    except Exception as e:
        test_errors.append((video_id, str(e)))

    if i % 50 == 0 or i == total:
        print(f"Processed {i}/{total}")


# ============================================================
# FINAL TEST DATASET
# ============================================================

test_visual_features = pd.DataFrame(test_results)

print("\n" + "=" * 60)
print("FINAL TEST VISUAL FEATURE DATASET")
print("=" * 60)

print("Expected rows :", len(clean_test_names))
print("Successful rows:", len(test_visual_features))
print("Errors         :", len(test_errors))
print("Shape          :", test_visual_features.shape)


# ============================================================
# ERRORS
# ============================================================

if test_errors:

    print("\nFIRST ERRORS:")

    for error in test_errors[:20]:
        print(error)

else:

    print("\nALL 367 TEST VIDEOS PROCESSED SUCCESSFULLY!")


# ============================================================
# PREVIEW
# ============================================================

display(test_visual_features.head())


# ============================================================
# SAVE
# ============================================================

TEST_OUTPUT = os.path.join(
    EMOREACT_PATH,
    "test_visual_features.csv"
)

test_visual_features.to_csv(
    TEST_OUTPUT,
    index=False
)

print("\nSaved successfully!")
print("Path:", TEST_OUTPUT)
print("Shape:", test_visual_features.shape)
print("Columns:", len(test_visual_features.columns))

Processed 50/367
Processed 100/367
Processed 150/367
Processed 200/367
Processed 250/367
Processed 300/367
Processed 350/367
Processed 367/367

FINAL TEST VISUAL FEATURE DATASET
Expected rows : 367
Successful rows: 0
Errors         : 367
Shape          : (0, 0)

FIRST ERRORS:
('80sTOYS104_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS123_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS130_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS139_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS28_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS32_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS49_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS4_2', "name 'aggregate_test_visual_features' is not defined")
('80sTOYS9_2', "name 'aggregate_test_visual_features' is not defined")
('ANNOYING_ORANGE120_2', "name 'aggregate_test_visual_features' is not de

""



Saved successfully!
Path: /content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/test_visual_features.csv
Shape: (0, 0)
Columns: 0


In [136]:
# ============================================================
# EMOREACT LABEL FILE CHECK
# ============================================================

import os
import pandas as pd

LABEL_DIR = os.path.join(
    EMOREACT_PATH,
    "Labels"
)

print("Label folder:")
print(LABEL_DIR)
print("Exists:", os.path.exists(LABEL_DIR))

print("\nFiles in Labels folder:")
for f in os.listdir(LABEL_DIR):
    print(f)

Label folder:
/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/Labels
Exists: True

Files in Labels folder:
test_labels.text
val_labels.text
train_labels.text


In [137]:
# ============================================================
# EMOREACT LABEL STRUCTURE CHECK
# ============================================================

import os

label_files = [
    "train_labels.text",
    "val_labels.text",
    "test_labels.text"
]

for filename in label_files:

    path = os.path.join(LABEL_DIR, filename)

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print("Number of lines:", len(lines))

    print("\nFirst 10 lines:")
    for i, line in enumerate(lines[:10]):
        print(i, repr(line.strip()))

    print("\nFirst line values:")
    first_values = lines[0].strip().split()

    print(first_values)
    print("Number of values:", len(first_values))


train_labels.text
Number of lines: 432

First 10 lines:
0 '0,0,0,1,0,0,0,0,3.3333'
1 '0,0,1,1,0,0,0,0,6.3333'
2 '0,0,0,0,0,0,0,0,2.6667'
3 '0,0,0,0,0,0,0,0,2.3333'
4 '0,0,0,1,0,0,0,0,4.6667'
5 '0,0,0,0,0,0,0,0,2.6667'
6 '0,0,1,1,0,0,0,0,6'
7 '0,0,1,1,0,0,0,0,5'
8 '1,1,1,0,1,0,0,0,5'
9 '1,1,0,0,0,0,0,0,2.6667'

First line values:
['0,0,0,1,0,0,0,0,3.3333']
Number of values: 1

val_labels.text
Number of lines: 303

First 10 lines:
0 '0,0,0,0,1,0,0,0,3.6667'
1 '0,0,0,0,0,0,0,0,4'
2 '0,0,1,1,0,0,0,0,6.6667'
3 '0,0,0,1,0,0,0,0,4.3333'
4 '0,0,0,0,0,0,0,0,4'
5 '0,0,1,1,1,0,0,0,6.6667'
6 '0,0,1,1,0,0,0,0,6.3333'
7 '0,0,1,1,0,0,0,0,5.6667'
8 '0,0,0,1,1,0,0,0,5.6667'
9 '0,1,1,1,0,0,0,0,6'

First line values:
['0,0,0,0,1,0,0,0,3.6667']
Number of values: 1

test_labels.text
Number of lines: 367

First 10 lines:
0 '0,0,0,0,0,0,0,0,4.3333'
1 '0,0,1,1,0,0,0,0,5.6667'
2 '0,0,1,1,0,0,0,0,4.6667'
3 '0,1,0,1,0,0,0,0,3.3333'
4 '1,0,1,1,0,0,0,0,6'
5 '0,0,1,1,0,0,0,0,6'
6 '0,1,1,0,1,0,1,0,3.3333'
7 '1,0,1,

In [138]:
# ============================================================
# EMOREACT LABEL PARSING - CORRECT COMMA FORMAT
# ============================================================

for filename in label_files:

    path = os.path.join(LABEL_DIR, filename)

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    parsed = []

    for line in lines:
        values = line.split(",")
        parsed.append(values)

    print("Number of rows:", len(parsed))
    print("Number of columns:", len(parsed[0]))

    print("\nFirst 10 parsed rows:")

    for row in parsed[:10]:
        print(row)

    print("\nFirst row:")
    print(parsed[0])


train_labels.text
Number of rows: 432
Number of columns: 9

First 10 parsed rows:
['0', '0', '0', '1', '0', '0', '0', '0', '3.3333']
['0', '0', '1', '1', '0', '0', '0', '0', '6.3333']
['0', '0', '0', '0', '0', '0', '0', '0', '2.6667']
['0', '0', '0', '0', '0', '0', '0', '0', '2.3333']
['0', '0', '0', '1', '0', '0', '0', '0', '4.6667']
['0', '0', '0', '0', '0', '0', '0', '0', '2.6667']
['0', '0', '1', '1', '0', '0', '0', '0', '6']
['0', '0', '1', '1', '0', '0', '0', '0', '5']
['1', '1', '1', '0', '1', '0', '0', '0', '5']
['1', '1', '0', '0', '0', '0', '0', '0', '2.6667']

First row:
['0', '0', '0', '1', '0', '0', '0', '0', '3.3333']

val_labels.text
Number of rows: 303
Number of columns: 9

First 10 parsed rows:
['0', '0', '0', '0', '1', '0', '0', '0', '3.6667']
['0', '0', '0', '0', '0', '0', '0', '0', '4']
['0', '0', '1', '1', '0', '0', '0', '0', '6.6667']
['0', '0', '0', '1', '0', '0', '0', '0', '4.3333']
['0', '0', '0', '0', '0', '0', '0', '0', '4']
['0', '0', '1', '1', '1', '0', '0

In [139]:
# ============================================================
# EMOREACT README - LABEL DEFINITION CHECK
# ============================================================

README_PATH = os.path.join(
    EMOREACT_PATH,
    "ReadMe.txt"
)

print("ReadMe path:")
print(README_PATH)

print("\nExists:", os.path.exists(README_PATH))

if os.path.exists(README_PATH):

    with open(README_PATH, "r", encoding="utf-8", errors="ignore") as f:
        readme_text = f.read()

    print("\n" + "=" * 80)
    print("EMOREACT README")
    print("=" * 80)

    print(readme_text[:15000])

else:
    print("ReadMe.txt NOT FOUND")

ReadMe path:
/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0/ReadMe.txt

Exists: True

EMOREACT README
This folder contains the followings:
Data:
This folder contains all videos from EmoReact. We have divided videos into train, validation and test folders. This
division is done in a person independent manner( people from training data do not appear in validation or test data.)

Labels:
This folder contains labels for the following affects ( First column --> curiosity , second column --> Uncertainty,...)
1- Curiosity
2- Uncertainty
3- Excitement
4- Happiness
5- Surprise
6- Disgust
7- Fear
8- Frustration
9- Valence

Visual features:
We realese the visual features that we have extracted using OpenFace. If you decide to use these features, please consider
citing :OpenFace: an open source facial behavior analysis toolkit Tadas Baltruaitis, Peter Robinson, and Louis-Philippe Morency, in IEEE Winter Conference on Applications of Computer Vision, 2016"




In [140]:
# ============================================================
# EMOREACT LABEL DATAFRAMES
# ============================================================

LABEL_COLUMNS = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]


def load_emoreact_labels(filename, video_names):

    path = os.path.join(LABEL_DIR, filename)

    # Read comma-separated labels
    df = pd.read_csv(
        path,
        header=None,
        names=LABEL_COLUMNS
    )

    # Add video IDs according to official split order
    df.insert(
        0,
        "video_id",
        video_names
    )

    return df


# ------------------------------------------------------------
# LOAD ALL LABELS
# ------------------------------------------------------------

train_labels = load_emoreact_labels(
    "train_labels.text",
    clean_train_names
)

val_labels = load_emoreact_labels(
    "val_labels.text",
    clean_val_names
)

test_labels = load_emoreact_labels(
    "test_labels.text",
    clean_test_names
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print("TRAIN LABELS")
print("Shape:", train_labels.shape)
display(train_labels.head())

print("\nVALIDATION LABELS")
print("Shape:", val_labels.shape)
display(val_labels.head())

print("\nTEST LABELS")
print("Shape:", test_labels.shape)
display(test_labels.head())

TRAIN LABELS
Shape: (432, 10)


,video_id,curiosity,uncertainty,excitement,happiness,surprise,disgust,fear,frustration,valence
0,80''sTOYS105_2,0,0,0,1,0,0,0,0,3.3333
1,80''sTOYS118_2,0,0,1,1,0,0,0,0,6.3333
2,80''sTOYS127_2,0,0,0,0,0,0,0,0,2.6667
3,80''sTOYS128_2,0,0,0,0,0,0,0,0,2.3333
4,80''sTOYS129_2,0,0,0,1,0,0,0,0,4.6667



VALIDATION LABELS
Shape: (303, 10)


,video_id,curiosity,uncertainty,excitement,happiness,surprise,disgust,fear,frustration,valence
0,80sTOYS115_2,0,0,0,0,1,0,0,0,3.6667
1,80sTOYS119_2,0,0,0,0,0,0,0,0,4.0000
2,80sTOYS120_2,0,0,1,1,0,0,0,0,6.6667
3,80sTOYS126_2,0,0,0,1,0,0,0,0,4.3333
4,80sTOYS133_2,0,0,0,0,0,0,0,0,4.0000



TEST LABELS
Shape: (367, 10)


,video_id,curiosity,uncertainty,excitement,happiness,surprise,disgust,fear,frustration,valence
0,80sTOYS104_2,0,0,0,0,0,0,0,0,4.3333
1,80sTOYS123_2,0,0,1,1,0,0,0,0,5.6667
2,80sTOYS130_2,0,0,1,1,0,0,0,0,4.6667
3,80sTOYS139_2,0,1,0,1,0,0,0,0,3.3333
4,80sTOYS28_2,1,0,1,1,0,0,0,0,6.0000


In [152]:
import os
import pandas as pd

# Re-load or assign the successfully processed test visual features
# In cell kdbi8GBkWWJD, the variable was created as test_visual_features
if 'test_visual_features' in globals() and not test_visual_features.empty:
    # Save the valid features to overwrite the empty CSV for future use
    TEST_OUTPUT = os.path.join(EMOREACT_PATH, 'test_visual_features.csv')
    test_visual_features.to_csv(TEST_OUTPUT, index=False)
else:
    # Fallback to load from CSV if it was already fixed
    test_visual_features = pd.read_csv(os.path.join(EMOREACT_PATH, 'test_visual_features.csv'))

# Make copies
train_final = train_visual_features.copy()
val_final = val_visual_features.copy()
test_final = test_visual_features.copy()

# Normalize video IDs for merging
for df in [train_final, val_final, test_final]:
    df["video_id"] = df["video_id"].astype(str).str.strip().str.replace('"', '').str.replace("'", "")

for df in [train_labels, val_labels, test_labels]:
    df["video_id"] = df["video_id"].astype(str).str.strip().str.replace('"', '').str.replace("'", "")

# Merge
train_final = train_final.merge(
    train_labels,
    on="video_id",
    how="inner",
    validate="one_to_one"
)

val_final = val_final.merge(
    val_labels,
    on="video_id",
    how="inner",
    validate="one_to_one"
)

test_final = test_final.merge(
    test_labels,
    on="video_id",
    how="inner",
    validate="one_to_one"
)

print("=" * 70)
print("FINAL EMOREACT DATASETS")
print("=" * 70)

print("\nTRAIN")
print("Shape:", train_final.shape)
print("Missing labels:", train_final[LABEL_COLUMNS].isna().sum().sum())

print("\nVALIDATION")
print("Shape:", val_final.shape)
print("Missing labels:", val_final[LABEL_COLUMNS].isna().sum().sum())

print("\nTEST")
print("Shape:", test_final.shape)
print("Missing labels:", test_final[LABEL_COLUMNS].isna().sum().sum())

EmptyDataError: No columns to parse from file

In [143]:
# ============================================================
# EMOREACT LABEL DISTRIBUTION CHECK
# ============================================================

label_cols = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration"
]

print("=" * 60)
print("EMOREACT LABEL DISTRIBUTION")
print("=" * 60)

for label in label_cols:
    print(f"\n{label.upper()}")

    print("TRAIN:")
    print(train_final[label].value_counts().sort_index())

    print("VALIDATION:")
    print(val_final[label].value_counts().sort_index())

    print("TEST:")
    print(test_final[label].value_counts().sort_index())

EMOREACT LABEL DISTRIBUTION

CURIOSITY
TRAIN:
curiosity
0    284
1    148
Name: count, dtype: int64
VALIDATION:
curiosity
0    197
1    106
Name: count, dtype: int64
TEST:


KeyError: 'curiosity'

In [144]:
# ============================================================
# EMOREACT - FRUSTRATION AUXILIARY MODEL DATA PREPARATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 60)
print("PREPARING EMOREACT FRUSTRATION DATA")
print("=" * 60)

# ------------------------------------------------------------
# 1. Label columns from EmoReact
# ------------------------------------------------------------

emoreact_label_cols = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]

# ------------------------------------------------------------
# 2. Feature columns
# ------------------------------------------------------------
# IMPORTANT:
# Do NOT use any EmoReact labels as input features.
# Especially do NOT use valence because it is an annotation.

feature_cols = [
    col for col in train_final.columns
    if col not in ["video_id"] + emoreact_label_cols
]

print("Number of input features:", len(feature_cols))

# ------------------------------------------------------------
# 3. Create X and y
# ------------------------------------------------------------

X_train = train_final[feature_cols].copy()
X_val   = val_final[feature_cols].copy()
X_test  = test_final[feature_cols].copy()

y_train = train_final["frustration"].astype(int)
y_val   = val_final["frustration"].astype(int)
y_test  = test_final["frustration"].astype(int)

# ------------------------------------------------------------
# 4. Convert everything to numeric
# ------------------------------------------------------------

X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_val   = X_val.apply(pd.to_numeric, errors="coerce")
X_test  = X_test.apply(pd.to_numeric, errors="coerce")

# ------------------------------------------------------------
# 5. Check missing values
# ------------------------------------------------------------

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

# ------------------------------------------------------------
# 6. Check infinite values
# ------------------------------------------------------------

print("\nInfinite values:")
print("Train:", np.isinf(X_train.to_numpy()).sum())
print("Validation:", np.isinf(X_val.to_numpy()).sum())
print("Test:", np.isinf(X_test.to_numpy()).sum())

# ------------------------------------------------------------
# 7. Replace invalid values
# ------------------------------------------------------------

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Use TRAIN medians only
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_val   = X_val.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

# ------------------------------------------------------------
# 8. Final shapes
# ------------------------------------------------------------

print("\nFinal shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

# ------------------------------------------------------------
# 9. Final class distribution
# ------------------------------------------------------------

print("\nFrustration class distribution:")

print("\nTRAIN:")
print(y_train.value_counts().sort_index())

print("\nVALIDATION:")
print(y_val.value_counts().sort_index())

print("\nTEST:")
print(y_test.value_counts().sort_index())

print("\n" + "=" * 60)
print("FRUSTRATION DATA PREPARATION COMPLETE")
print("=" * 60)

PREPARING EMOREACT FRUSTRATION DATA
Number of input features: 150


KeyError: "None of [Index([' timestamp_mean', ' timestamp_std', ' confidence_mean',\n       ' confidence_std', ' success_mean', ' success_std', ' scale_mean',\n       ' scale_std', ' rx_mean', ' rx_std',\n       ...\n       ' y_h0_mean', ' y_h0_std', ' z_h0_mean', ' z_h0_std', ' x_h1_mean',\n       ' x_h1_std', ' y_h1_mean', ' y_h1_std', ' z_h1_mean', ' z_h1_std'],\n      dtype='object', length=150)] are in the [columns]"

In [145]:
# ============================================================
# EMOREACT FRUSTRATION - MODEL TRAINING
# Logistic Regression + Random Forest + XGBoost
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("TRAINING EMOREACT FRUSTRATION MODELS")
print("=" * 60)


# ============================================================
# 1. LOGISTIC REGRESSION
# ============================================================

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)


# ============================================================
# 2. RANDOM FOREST
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)


# ============================================================
# 3. XGBOOST
# ============================================================

try:

    from xgboost import XGBClassifier

    # Calculate scale_pos_weight for class imbalance
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()

    scale_pos_weight = negative_count / positive_count

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )

    xgb_model.fit(X_train, y_train)

    xgb_pred = xgb_model.predict(X_test)

    xgb_available = True

except Exception as e:

    print("\nXGBoost could not be trained.")
    print("Reason:", e)

    xgb_available = False


# ============================================================
# 4. EVALUATION FUNCTION
# ============================================================

def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Not Frustrated",
                "Frustrated"
            ],
            zero_division=0
        )
    )

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }


# ============================================================
# 5. EVALUATE
# ============================================================

results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        lr_pred
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

if xgb_available:

    results.append(
        evaluate_model(
            "XGBoost",
            y_test,
            xgb_pred
        )
    )


# ============================================================
# 6. COMPARISON TABLE
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

display(
    results_df.sort_values(
        "F1",
        ascending=False
    )
)

TRAINING EMOREACT FRUSTRATION MODELS


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# ============================================================
# EMOREACT FRUSTRATION - VALIDATION THRESHOLD TUNING
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 60)
print("VALIDATION-BASED THRESHOLD TUNING")
print("=" * 60)


# ------------------------------------------------------------
# 1. Get probability of Frustrated class
# ------------------------------------------------------------

lr_val_prob = lr_model.predict_proba(X_val)[:, 1]

print("Validation probabilities obtained.")
print("Number of validation samples:", len(lr_val_prob))


# ------------------------------------------------------------
# 2. Try different thresholds
# ------------------------------------------------------------

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    val_pred = (
        lr_val_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val,
        val_pred
    )

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


# ------------------------------------------------------------
# 3. Create comparison table
# ------------------------------------------------------------

threshold_df = pd.DataFrame(
    threshold_results
)

print("\nValidation threshold results:")

display(
    threshold_df.sort_values(
        "F1",
        ascending=False
    )
)


# ------------------------------------------------------------
# 4. Select best threshold using validation F1
# ------------------------------------------------------------

best_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_threshold = best_row["Threshold"]

print("\n" + "=" * 60)
print("BEST VALIDATION THRESHOLD")
print("=" * 60)

print("Threshold :", best_threshold)
print("Accuracy  :", round(best_row["Accuracy"], 4))
print("Precision :", round(best_row["Precision"], 4))
print("Recall    :", round(best_row["Recall"], 4))
print("F1        :", round(best_row["F1"], 4))


# ------------------------------------------------------------
# 5. Validation confusion matrix
# ------------------------------------------------------------

best_val_pred = (
    lr_val_prob >= best_threshold
).astype(int)

print("\nValidation Confusion Matrix:")
print(
    confusion_matrix(
        y_val,
        best_val_pred
    )
)

In [ ]:
# ============================================================
# CLASS-IMBALANCE AWARE MODEL TRAINING
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("CLASS-IMBALANCE AWARE MODEL TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# 1. Logistic Regression
# ------------------------------------------------------------

lr_balanced = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

lr_balanced.fit(X_train, y_train)

lr_val_prob = lr_balanced.predict_proba(X_val)[:, 1]

print("\nLOGISTIC REGRESSION - BALANCED")
print("Validation probability range:",
      lr_val_prob.min(),
      "to",
      lr_val_prob.max())


# ------------------------------------------------------------
# 2. Random Forest
# ------------------------------------------------------------

rf_balanced = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_balanced.fit(X_train, y_train)

rf_val_prob = rf_balanced.predict_proba(X_val)[:, 1]

print("\nRANDOM FOREST - BALANCED")
print("Validation probability range:",
      rf_val_prob.min(),
      "to",
      rf_val_prob.max())


# ------------------------------------------------------------
# 3. XGBoost
# ------------------------------------------------------------

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("\nXGBOOST scale_pos_weight:",
      scale_pos_weight)

xgb_balanced = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb_balanced.fit(X_train, y_train)

xgb_val_prob = xgb_balanced.predict_proba(X_val)[:, 1]

print("\nXGBOOST - BALANCED")
print("Validation probability range:",
      xgb_val_prob.min(),
      "to",
      xgb_val_prob.max())


print("\n" + "=" * 60)
print("BALANCED MODELS TRAINED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# BALANCED MODEL - VALIDATION THRESHOLD TUNING
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np


def tune_threshold(y_true, probabilities, model_name):

    results = []

    thresholds = np.arange(0.10, 0.91, 0.05)

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        results.append({
            "Model": model_name,
            "Threshold": round(threshold, 2),
            "Accuracy": accuracy_score(
                y_true, predictions
            ),
            "Precision": precision_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_true,
                predictions,
                zero_division=0
            )
        })

    return pd.DataFrame(results)


# ------------------------------------------------------------
# Tune all 3 balanced models
# ------------------------------------------------------------

lr_tuning = tune_threshold(
    y_val,
    lr_val_prob,
    "Balanced Logistic Regression"
)

rf_tuning = tune_threshold(
    y_val,
    rf_val_prob,
    "Balanced Random Forest"
)

xgb_tuning = tune_threshold(
    y_val,
    xgb_val_prob,
    "Balanced XGBoost"
)


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

threshold_results = pd.concat(
    [
        lr_tuning,
        rf_tuning,
        xgb_tuning
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Sort by F1
# ------------------------------------------------------------

threshold_results = threshold_results.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)


print("=" * 70)
print("BALANCED MODEL - VALIDATION THRESHOLD RESULTS")
print("=" * 70)

display(threshold_results)


# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

best_result = threshold_results.iloc[0]

print("\n" + "=" * 70)
print("BEST VALIDATION RESULT")
print("=" * 70)

print("Model     :", best_result["Model"])
print("Threshold :", best_result["Threshold"])
print("Accuracy  :", round(best_result["Accuracy"], 4))
print("Precision :", round(best_result["Precision"], 4))
print("Recall    :", round(best_result["Recall"], 4))
print("F1        :", round(best_result["F1"], 4))


# ------------------------------------------------------------
# Confusion matrix for best result
# ------------------------------------------------------------

best_model_name = best_result["Model"]
best_threshold = best_result["Threshold"]

if best_model_name == "Balanced Logistic Regression":
    best_prob = lr_val_prob

elif best_model_name == "Balanced Random Forest":
    best_prob = rf_val_prob

else:
    best_prob = xgb_val_prob


best_val_pred = (
    best_prob >= best_threshold
).astype(int)

print("\nValidation Confusion Matrix:")
print(
    confusion_matrix(
        y_val,
        best_val_pred
    )
)

In [ ]:
# ============================================================
# FINAL LOGISTIC REGRESSION - SCALED + BALANCED
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

print("=" * 60)
print("FINAL BALANCED LOGISTIC REGRESSION")
print("=" * 60)

final_lr = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            max_iter=5000,
            random_state=42
        )
    )
])

final_lr.fit(
    X_train,
    y_train
)

# Validation probabilities
final_lr_val_prob = final_lr.predict_proba(
    X_val
)[:, 1]

print("Training completed successfully!")

print(
    "Validation probability range:",
    final_lr_val_prob.min(),
    "to",
    final_lr_val_prob.max()
)

In [ ]:
# ============================================================
# FINAL LOGISTIC REGRESSION - VALIDATION THRESHOLD TUNING
# ============================================================

threshold_results_final = []

for threshold in np.arange(0.10, 0.91, 0.05):

    val_pred = (
        final_lr_val_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val,
        val_pred
    )

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    threshold_results_final.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


final_threshold_df = pd.DataFrame(
    threshold_results_final
)

final_threshold_df = final_threshold_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)


print("=" * 70)
print("FINAL LOGISTIC REGRESSION - VALIDATION THRESHOLD RESULTS")
print("=" * 70)

display(final_threshold_df)


# Best threshold
best_final = final_threshold_df.iloc[0]

print("\n" + "=" * 70)
print("BEST FINAL VALIDATION THRESHOLD")
print("=" * 70)

print("Threshold :", best_final["Threshold"])
print("Accuracy  :", round(best_final["Accuracy"], 4))
print("Precision :", round(best_final["Precision"], 4))
print("Recall    :", round(best_final["Recall"], 4))
print("F1        :", round(best_final["F1"], 4))


# Confusion matrix
best_threshold_final = best_final["Threshold"]

final_val_pred = (
    final_lr_val_prob >= best_threshold_final
).astype(int)

print("\nValidation Confusion Matrix:")
print(
    confusion_matrix(
        y_val,
        final_val_pred
    )
)

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print("=" * 70)
print("FINAL EMOREACT FRUSTRATION MODEL - TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Predict test probabilities
# ------------------------------------------------------------

final_lr_test_prob = final_lr.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# 2. Apply LOCKED validation threshold
# ------------------------------------------------------------

FINAL_THRESHOLD = 0.80

final_test_pred = (
    final_lr_test_prob >= FINAL_THRESHOLD
).astype(int)


# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    final_test_pred
)

test_precision = precision_score(
    y_test,
    final_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    final_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    final_test_pred,
    zero_division=0
)


# ------------------------------------------------------------
# 4. Results
# ------------------------------------------------------------

print("\nFINAL TEST RESULTS")
print("-" * 40)

print("Threshold :", FINAL_THRESHOLD)
print("Accuracy  :", round(test_accuracy, 4))
print("Precision :", round(test_precision, 4))
print("Recall    :", round(test_recall, 4))
print("F1 Score  :", round(test_f1, 4))


# ------------------------------------------------------------
# 5. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    final_test_pred
)

print("\nConfusion Matrix:")
print(cm)


# ------------------------------------------------------------
# 6. Classification Report
# ------------------------------------------------------------

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        final_test_pred,
        target_names=[
            "Not Frustrated",
            "Frustrated"
        ],
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("=" * 70)
print("FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
import pandas as pd
import os

BASE = "/content/drive/MyDrive/Behavior Detection/ETDD70"

print("ETDD70 folder:")
print(os.listdir(BASE))

In [ ]:
features_path = os.path.join(BASE, "ETDD70_features.csv")

etdd = pd.read_csv(features_path)

print("Shape:", etdd.shape)
print("\nColumns:")
print(etdd.columns.tolist())

print("\nFirst 5 rows:")
display(etdd.head())

In [ ]:
print("=== Dyslexia Label Distribution ===")
print(etdd["label"].value_counts())

print("\n=== Task Distribution ===")
print(etdd["task"].value_counts())

print("\n=== Unique Participants ===")
print("Number of subjects:", etdd["subject_id"].nunique())

print("\n=== Sessions per Participant ===")
print(etdd.groupby("subject_id").size().value_counts())

In [ ]:
# Eye-tracking feature summary

eye_features = [
    "n_fix_trial",
    "sum_fix_dur_trial",
    "dwell_time_trial",
    "mean_fix_dur_trial",
    "n_sacc_trial",
    "sum_sacc_dur_trial",
    "mean_sacc_dur_trial",
    "mean_sacc_ampl_trial",
    "ratio_progress_regress_trial",
    "n_between_line_regress_trial",
    "n_within_line_regress_trial",
    "n_regress_trial",
    "n_progress_trial",
    "n_transit_trial"
]

display(etdd[eye_features].describe().T)

In [ ]:
# Compare eye-tracking features between dyslexic and non-dyslexic groups

comparison = etdd.groupby("label")[eye_features].mean().T

display(comparison)

In [ ]:
from scipy.stats import mannwhitneyu

# Average the 3 reading tasks for each participant
subject_data = (
    etdd.groupby(["subject_id", "label"])[eye_features]
    .mean()
    .reset_index()
)

results = []

for feature in eye_features:
    dys = subject_data.loc[
        subject_data["label"] == "dyslexic", feature
    ]

    non_dys = subject_data.loc[
        subject_data["label"] == "non-dyslexic", feature
    ]

    stat, p_value = mannwhitneyu(
        dys,
        non_dys,
        alternative="two-sided"
    )

    results.append({
        "feature": feature,
        "dyslexic_mean": dys.mean(),
        "non_dyslexic_mean": non_dys.mean(),
        "p_value": p_value
    })

stats_results = pd.DataFrame(results)

display(
    stats_results.sort_values("p_value")
)

In [ ]:
from statsmodels.stats.multitest import multipletests

# Apply Benjamini-Hochberg FDR correction
stats_results["p_adjusted"] = multipletests(
    stats_results["p_value"],
    method="fdr_bh"
)[1]

stats_results["significant_fdr"] = (
    stats_results["p_adjusted"] < 0.05
)

display(
    stats_results.sort_values("p_adjusted")
)

In [ ]:
from scipy.stats import mannwhitneyu

effect_results = []

for feature in eye_features:
    dys = subject_data.loc[
        subject_data["label"] == "dyslexic", feature
    ].values

    non_dys = subject_data.loc[
        subject_data["label"] == "non-dyslexic", feature
    ].values

    U, p = mannwhitneyu(
        dys,
        non_dys,
        alternative="two-sided"
    )

    n1 = len(dys)
    n2 = len(non_dys)

    # Rank-biserial correlation
    r_rb = (2 * U) / (n1 * n2) - 1

    effect_results.append({
        "feature": feature,
        "rank_biserial_effect": abs(r_rb)
    })

effect_results = pd.DataFrame(effect_results)

effect_results = effect_results.sort_values(
    "rank_biserial_effect",
    ascending=False
)

display(effect_results)

In [ ]:
# Final ETDD70 features supported by the statistical analysis

selected_etdd_features = [
    "n_fix_trial",
    "sum_fix_dur_trial",
    "dwell_time_trial",
    "mean_fix_dur_trial",
    "n_sacc_trial",
    "sum_sacc_dur_trial",
    "mean_sacc_dur_trial",
    "mean_sacc_ampl_trial",
    "ratio_progress_regress_trial",
    "n_within_line_regress_trial",
    "n_regress_trial",
    "n_progress_trial",
    "n_transit_trial"
]

etdd_selected = etdd[
    ["subject_id", "task", "label"] + selected_etdd_features
].copy()

print("Shape:", etdd_selected.shape)
print("\nSelected features:")
print(selected_etdd_features)

display(etdd_selected.head())

In [ ]:
selected_path = os.path.join(
    BASE,
    "ETDD70_behavior_features.csv"
)

etdd_selected.to_csv(
    selected_path,
    index=False
)

print("Saved successfully:")
print(selected_path)

In [ ]:
import os

BASE_BEHAVIOR = "/content/drive/MyDrive/Behavior Detection"

print(os.listdir(BASE_BEHAVIOR))

In [ ]:
EMO_BASE = "/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0"

print("EmoReact folder:")
print(os.listdir(EMO_BASE))

In [ ]:
EMO_BASE = "/content/drive/MyDrive/Behavior Detection/EmoReact_V_1.0/EmoReact_V_1.0"

print("EmoReact dataset:")
print(os.listdir(EMO_BASE))

In [ ]:
LABELS_DIR = os.path.join(EMO_BASE, "Labels")

print("Labels folder:")
print(os.listdir(LABELS_DIR))

In [ ]:
# Inspect the actual EmoReact label files

for split in ["train", "val", "test"]:
    path = os.path.join(LABELS_DIR, f"{split}_labels.text")

    print(f"\n===== {split.upper()} LABELS =====")

    with open(path, "r") as f:
        lines = f.readlines()

    print("Number of rows:", len(lines))
    print("First 3 rows:")

    for line in lines[:3]:
        print(line.strip())

In [ ]:
# Exact EmoReact label order from ReadMe.txt

label_columns = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]

emoreact_labels = {}

for split in ["train", "val", "test"]:
    path = os.path.join(
        LABELS_DIR,
        f"{split}_labels.text"
    )

    df = pd.read_csv(
        path,
        header=None,
        names=label_columns
    )

    emoreact_labels[split] = df

    print(f"\n{split.upper()}")
    print("Shape:", df.shape)
    display(df.head())

In [ ]:
for split, df in emoreact_labels.items():

    print(f"\n===== {split.upper()} =====")

    print("Frustration distribution:")
    print(df["frustration"].value_counts().sort_index())

    print("\nFrustration summary:")
    print(df["frustration"].describe())

In [ ]:
# Verify video-name mapping files

name_files = {
    "train": "Train_names.txt",
    "val": "Val_names.txt",
    "test": "Test_names.txt"
}

for split, filename in name_files.items():

    path = os.path.join(EMO_BASE, filename)

    with open(path, "r") as f:
        names = [line.strip() for line in f if line.strip()]

    print(f"\n===== {split.upper()} =====")
    print("Number of names:", len(names))
    print("First 5 names:")
    print(names[:5])

    print("Label rows:", len(emoreact_labels[split]))

In [ ]:
# Inspect EmoReact visual feature files

visual_files = {
    "train": "train_visual_features.csv",
    "val": "val_visual_features.csv",
    "test": "test_visual_features.csv"
}

emoreact_visual = {}

for split, filename in visual_files.items():

    path = os.path.join(EMO_BASE, filename)

    df = pd.read_csv(path)

    emoreact_visual[split] = df

    print(f"\n===== {split.upper()} VISUAL FEATURES =====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist()[:20])
    display(df.head())

In [ ]:
emoreact_evidence = {}

for split in ["train", "val", "test"]:

    # Get labels
    labels_df = emoreact_labels[split].copy()

    # Get video names in the same order as labels/features
    names_path = os.path.join(
        EMO_BASE,
        name_files[split]
    )

    with open(names_path, "r") as f:
        video_names = [line.strip() for line in f if line.strip()]

    # Add video_id from mapping file
    labels_df["video_id"] = [
        os.path.splitext(name)[0]
        for name in video_names
    ]

    # Get visual features
    visual_df = emoreact_visual[split].copy()

    # Normalize video_id just in case
    visual_df["video_id"] = (
        visual_df["video_id"]
        .astype(str)
        .str.strip()
    )

    # Merge
    merged = labels_df.merge(
        visual_df,
        on="video_id",
        how="inner"
    )

    emoreact_evidence[split] = merged

    print(f"\n===== {split.upper()} =====")
    print("Labels:", len(labels_df))
    print("Visual features:", len(visual_df))
    print("Merged:", len(merged))
    print("Missing mappings:", len(labels_df) - len(merged))

In [ ]:
# Inspect exact video ID formatting

print("NAME FILE:")
print(repr(video_names[0]))

print("\nVISUAL FEATURE ID:")
print(repr(emoreact_visual["train"]["video_id"].iloc[0]))

print("\nFirst 5 name IDs:")
for x in video_names[:5]:
    print(repr(os.path.splitext(x)[0]))

print("\nFirst 5 visual IDs:")
for x in emoreact_visual["train"]["video_id"].head():
    print(repr(x))

In [ ]:
def normalize_video_id(x):
    x = str(x).strip()

    # Remove surrounding quotation marks
    x = x.strip('"').strip("'")

    # Remove .mp4 extension
    if x.lower().endswith(".mp4"):
        x = x[:-4]

    return x.strip()


# Rebuild EmoReact evidence tables
emoreact_evidence = {}

for split in ["train", "val", "test"]:

    # Labels
    labels_df = emoreact_labels[split].copy()

    # Names
    names_path = os.path.join(
        EMO_BASE,
        name_files[split]
    )

    with open(names_path, "r") as f:
        video_names = [line.strip() for line in f if line.strip()]

    labels_df["video_id"] = [
        normalize_video_id(name)
        for name in video_names
    ]

    # Visual features
    visual_df = emoreact_visual[split].copy()

    visual_df["video_id"] = (
        visual_df["video_id"]
        .apply(normalize_video_id)
    )

    # Merge
    merged = labels_df.merge(
        visual_df,
        on="video_id",
        how="inner"
    )

    emoreact_evidence[split] = merged

    print(f"\n===== {split.upper()} =====")
    print("Labels:", len(labels_df))
    print("Visual features:", len(visual_df))
    print("Merged:", len(merged))
    print("Missing mappings:", len(labels_df) - len(merged))

In [ ]:
# Find exactly which video IDs failed to match

for split in ["val", "test"]:

    labels_df = emoreact_labels[split].copy()

    names_path = os.path.join(
        EMO_BASE,
        name_files[split]
    )

    with open(names_path, "r") as f:
        video_names = [line.strip() for line in f if line.strip()]

    labels_df["video_id"] = [
        normalize_video_id(name)
        for name in video_names
    ]

    visual_ids = set(
        emoreact_visual[split]["video_id"]
        .apply(normalize_video_id)
    )

    missing_ids = [
        x for x in labels_df["video_id"]
        if x not in visual_ids
    ]

    print(f"\n===== {split.upper()} MISSING =====")
    print("Count:", len(missing_ids))

    for x in missing_ids:
        print(repr(x))

In [ ]:
VISUAL_DIR = os.path.join(EMO_BASE, "Visual_features")

print("Visual_features folder:")
print(os.listdir(VISUAL_DIR)[:30])

print("\nTotal items:", len(os.listdir(VISUAL_DIR)))

In [ ]:
# Check whether the 18 missing videos exist in the original feature folders

missing_check = {
    "val": [
        "80'sTOYS115_2",
        "80'sTOYS119_2",
        "80'sTOYS120_2",
        "80'sTOYS126_2",
        "80'sTOYS133_2",
        "80'sTOYS16_2",
        "80'sTOYS27_2",
        "80'sTOYS39_2",
        "80'sTOYS40_2"
    ],
    "test": [
        "80'sTOYS104_2",
        "80'sTOYS123_2",
        "80'sTOYS130_2",
        "80'sTOYS139_2",
        "80'sTOYS28_2",
        "80'sTOYS32_2",
        "80'sTOYS49_2",
        "80'sTOYS4_2",
        "80'sTOYS9_2"
    ]
}

for split, ids in missing_check.items():

    folder_name = {
        "val": "Val_feat",
        "test": "Test_feat"
    }[split]

    folder = os.path.join(VISUAL_DIR, folder_name)

    files = os.listdir(folder)

    print(f"\n===== {split.upper()} =====")

    for video_id in ids:
        matches = [
            f for f in files
            if video_id.lower() in f.lower()
        ]

        print(video_id, "->", matches[:5])

In [ ]:
# Inspect one existing feature file and one missing feature file

val_folder = os.path.join(VISUAL_DIR, "Val_feat")

print("=== MISSING VIDEO: 80'sTOYS115_2_au.txt ===")

au_path = os.path.join(
    val_folder,
    "80'sTOYS115_2_au.txt"
)

with open(au_path, "r", errors="ignore") as f:
    for i, line in enumerate(f):
        print(repr(line.strip()))
        if i >= 4:
            break


print("\n=== MISSING VIDEO: 80'sTOYS115_2_gaze.txt ===")

gaze_path = os.path.join(
    val_folder,
    "80'sTOYS115_2_gaze.txt"
)

with open(gaze_path, "r", errors="ignore") as f:
    for i, line in enumerate(f):
        print(repr(line.strip()))
        if i >= 4:
            break

In [ ]:
# Inspect the complete 151-column structure
train_visual = emoreact_visual["train"]

print("Number of columns:", len(train_visual.columns))

for i, col in enumerate(train_visual.columns):
    print(i, ":", col)

In [ ]:
# Inspect an existing video's raw feature files
# We will use the first video that already exists in the CSV.

known_id = train_visual["video_id"].iloc[0]

print("Known video:", known_id)

train_folder = os.path.join(VISUAL_DIR, "Train_feat")

files = [
    f"{known_id}_au.txt",
    f"{known_id}_gaze.txt",
    f"{known_id}.params.txt"
]

for filename in files:
    path = os.path.join(train_folder, filename)

    print("\n==============================")
    print(filename)
    print("==============================")

    if os.path.exists(path):
        with open(path, "r", errors="ignore") as f:
            for i, line in enumerate(f):
                print(line.strip())
                if i >= 2:
                    break
    else:
        print("NOT FOUND")

In [ ]:
# Check the exact filenames inside Train_feat

train_folder = os.path.join(VISUAL_DIR, "Train_feat")

files = os.listdir(train_folder)

print("Number of files:", len(files))

print("\nFirst 20 files:")
for f in files[:20]:
    print(repr(f))

print("\nFiles containing known video ID:")
for f in files:
    if "80" in f and "TOYS105" in f:
        print(repr(f))

In [ ]:
# Read one existing video's raw files and compare with its CSV row

known_id = train_visual["video_id"].iloc[0]
print("Known video ID:", repr(known_id))

train_folder = os.path.join(VISUAL_DIR, "Train_feat")

# Actual filenames contain surrounding quotes
au_file = os.path.join(train_folder, f'"{known_id}_au.txt"')
gaze_file = os.path.join(train_folder, f'"{known_id}_gaze.txt"')
params_file = os.path.join(train_folder, f'"{known_id}.params.txt"')

print("\nFiles:")
print(au_file, os.path.exists(au_file))
print(gaze_file, os.path.exists(gaze_file))
print(params_file, os.path.exists(params_file))

In [ ]:
# Fix the video ID formatting first

known_id = train_visual["video_id"].iloc[0]

known_id = str(known_id).strip().strip('"').strip("'")

print("Clean video ID:", repr(known_id))

train_folder = os.path.join(VISUAL_DIR, "Train_feat")

au_file = os.path.join(
    train_folder,
    f'"{known_id}_au.txt"'
)

gaze_file = os.path.join(
    train_folder,
    f'"{known_id}_gaze.txt"'
)

params_file = os.path.join(
    train_folder,
    f'"{known_id}.params.txt"'
)

print("\nFiles:")
print("AU:", os.path.exists(au_file))
print("Gaze:", os.path.exists(gaze_file))
print("Params:", os.path.exists(params_file))

print("\nPaths:")
print(au_file)
print(gaze_file)
print(params_file)

In [ ]:
# Robustly find the actual files using normalized filenames

def normalize_id(x):
    x = str(x).strip()

    # remove quotation marks from both ends
    x = x.strip('"').strip("'").strip()

    return x


known_id = normalize_id(train_visual["video_id"].iloc[0])

print("Clean ID:")
print(repr(known_id))

train_folder = os.path.join(VISUAL_DIR, "Train_feat")

all_files = os.listdir(train_folder)

# Find files whose normalized name starts with the video ID
matches = []

for filename in all_files:

    normalized_filename = filename.strip('"').strip("'").strip()

    if normalized_filename.startswith(known_id + "_") or \
       normalized_filename.startswith(known_id + "."):

        matches.append(filename)

print("\nMatching files:")
for f in matches:
    print(repr(f))

print("\nNumber of matching files:", len(matches))

In [ ]:
import re

def normalize_emoreact_name(x):
    x = str(x).strip()

    # Remove all quote characters
    x = re.sub(r'["\']', '', x)

    # Remove .mp4 if present
    x = re.sub(r'\.mp4$', '', x, flags=re.IGNORECASE)

    # Remove spaces
    x = x.replace(" ", "")

    return x.lower()


known_id = normalize_emoreact_name(
    train_visual["video_id"].iloc[0]
)

print("Normalized ID:")
print(repr(known_id))


train_folder = os.path.join(VISUAL_DIR, "Train_feat")

matches = []

for filename in os.listdir(train_folder):

    normalized_filename = normalize_emoreact_name(filename)

    if normalized_filename.startswith(known_id):
        matches.append(filename)


print("\nMatching files:")
for f in matches:
    print(repr(f))

print("\nNumber of matching files:", len(matches))

In [ ]:
# Inspect the header of the three raw feature files
# for one video whose CSV row already exists.

video_id = normalize_emoreact_name(
    train_visual["video_id"].iloc[0]
)

train_folder = os.path.join(VISUAL_DIR, "Train_feat")

for suffix in ["_au.txt", "_gaze.txt", ".params.txt"]:

    matches = []

    for filename in os.listdir(train_folder):
        if normalize_emoreact_name(filename).startswith(
            video_id + normalize_emoreact_name(suffix)
        ):
            matches.append(filename)

    print("\n==============================")
    print(suffix)
    print("==============================")

    for filename in matches[:1]:

        path = os.path.join(train_folder, filename)

        with open(path, "r", errors="ignore") as f:
            header = f.readline().strip()

        print(header)

In [ ]:
import pandas as pd
import numpy as np
import os
import re

def clean_name(x):
    x = str(x).strip()
    x = re.sub(r'["\']', '', x)
    x = x.replace(".mp4", "")
    x = x.replace(" ", "")
    return x.lower()


def find_file(folder, video_id, suffix):

    target = clean_name(video_id) + suffix.lower()

    for filename in os.listdir(folder):

        cleaned = clean_name(filename)

        if cleaned == target:
            return os.path.join(folder, filename)

    return None


# Use an existing TRAIN video
original_id = train_visual["video_id"].iloc[0]

video_id = clean_name(original_id)

print("Testing video:", video_id)


train_folder = os.path.join(
    VISUAL_DIR,
    "Train_feat"
)


# Find raw files
au_path = find_file(
    train_folder,
    video_id,
    "_au.txt"
)

gaze_path = find_file(
    train_folder,
    video_id,
    "_gaze.txt"
)

params_path = find_file(
    train_folder,
    video_id,
    ".params.txt"
)


print("\nFiles found:")
print("AU:", au_path is not None)
print("Gaze:", gaze_path is not None)
print("Params:", params_path is not None)


# Read raw files
au = pd.read_csv(au_path)
gaze = pd.read_csv(gaze_path)
params = pd.read_csv(params_path)


print("\nRaw shapes:")
print("AU:", au.shape)
print("Gaze:", gaze.shape)
print("Params:", params.shape)

In [ ]:
# Reconstruct mean/std features from the raw files

def aggregate_features(df):
    # Remove metadata columns
    feature_cols = [
        c for c in df.columns
        if c not in ["frame", "timestamp", "confidence", "success"]
    ]

    result = {}

    for col in feature_cols:
        result[f"{col}_mean"] = df[col].mean()
        result[f"{col}_std"] = df[col].std()

    return result


au_features = aggregate_features(au)
gaze_features = aggregate_features(gaze)
params_features = aggregate_features(params)

reconstructed = {}

reconstructed.update(au_features)
reconstructed.update(gaze_features)
reconstructed.update(params_features)

reconstructed_df = pd.DataFrame([reconstructed])

print("Reconstructed feature count:", reconstructed_df.shape[1])

print("\nFirst 10 columns:")
print(reconstructed_df.columns[:10].tolist())

print("\nLast 10 columns:")
print(reconstructed_df.columns[-10:].tolist())

In [ ]:
# Compare reconstructed features with the original CSV row

original_row = train_visual[
    train_visual["video_id"].apply(clean_name) == video_id
].iloc[0]

# Make both series use the same column order
original_features = original_row.drop(labels=["video_id"]).astype(float)

reconstructed_features = reconstructed_df.iloc[0].astype(float)

comparison = pd.DataFrame({
    "original": original_features.values,
    "reconstructed": reconstructed_features[
        original_features.index
    ].values
})

comparison["absolute_difference"] = (
    comparison["original"] -
    comparison["reconstructed"]
).abs()

print("Number of features compared:", len(comparison))

print(
    "Maximum absolute difference:",
    comparison["absolute_difference"].max()
)

print(
    "Mean absolute difference:",
    comparison["absolute_difference"].mean()
)

print("\nFeatures with largest differences:")
display(
    comparison
    .sort_values("absolute_difference", ascending=False)
    .head(10)
)

In [ ]:
df[col].std()

In [ ]:
# Check whether the original CSV used population std (ddof=0)

def aggregate_features_ddof0(df):
    feature_cols = [
        c for c in df.columns
        if c not in ["frame", "timestamp", "confidence", "success"]
    ]

    result = {}

    for col in feature_cols:
        result[f"{col}_mean"] = df[col].mean()
        result[f"{col}_std"] = df[col].std(ddof=0)

    return result


reconstructed_ddof0 = {}

# Include metadata columns too, because CSV has their mean/std
for df in [au, gaze, params]:
    feature_cols = [
        c for c in df.columns
    ]

    for col in feature_cols:
        if col not in ["frame"]:
            reconstructed_ddof0[f"{col}_mean"] = df[col].mean()
            reconstructed_ddof0[f"{col}_std"] = df[col].std(ddof=0)


# Keep only the 150 columns that exist in the original CSV
reconstructed_ddof0 = pd.Series(reconstructed_ddof0)

common_cols = [
    c for c in original_features.index
    if c in reconstructed_ddof0.index
]

comparison_ddof0 = pd.DataFrame({
    "original": original_features[common_cols].values,
    "reconstructed_ddof0": reconstructed_ddof0[common_cols].values
})

comparison_ddof0["absolute_difference"] = (
    comparison_ddof0["original"] -
    comparison_ddof0["reconstructed_ddof0"]
).abs()

print("Features compared:", len(comparison_ddof0))
print(
    "Maximum absolute difference:",
    comparison_ddof0["absolute_difference"].max()
)
print(
    "Mean absolute difference:",
    comparison_ddof0["absolute_difference"].mean()
)

display(
    comparison_ddof0
    .sort_values("absolute_difference", ascending=False)
    .head(10)
)

In [ ]:
# Check the original vs reconstructed values with high precision

comparison_check = comparison.sort_values(
    "absolute_difference",
    ascending=False
).head(10).copy()

pd.set_option("display.precision", 12)

display(comparison_check)

In [ ]:
# ============================================================
# RECONSTRUCT MISSING EMOREACT VISUAL FEATURES
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Helper function
# ------------------------------------------------------------

def aggregate_emoreact_file(path):
    """
    Read one EmoReact OpenFace file and calculate
    mean + sample standard deviation (ddof=1)
    for every numerical feature.
    """

    df = pd.read_csv(path)

    result = {}

    for col in df.columns:

        # Keep numerical OpenFace columns
        if col in ["frame"]:
            continue

        # Convert safely to numeric
        values = pd.to_numeric(df[col], errors="coerce")

        result[f"{col}_mean"] = values.mean()
        result[f"{col}_std"] = values.std(ddof=1)

    return result


# ------------------------------------------------------------
# 2. Reconstruct one video
# ------------------------------------------------------------

def reconstruct_video(video_id, feature_folder):

    video_id = clean_name(video_id)

    au_path = find_file(
        feature_folder,
        video_id,
        "_au.txt"
    )

    gaze_path = find_file(
        feature_folder,
        video_id,
        "_gaze.txt"
    )

    params_path = find_file(
        feature_folder,
        video_id,
        ".params.txt"
    )

    # Make sure all three files exist
    if not all([au_path, gaze_path, params_path]):
        print("MISSING RAW FILE:", video_id)
        return None

    # Read raw files
    au = pd.read_csv(au_path)
    gaze = pd.read_csv(gaze_path)
    params = pd.read_csv(params_path)

    # Combine all features
    reconstructed = {}

    for df in [au, gaze, params]:

        for col in df.columns:

            if col == "frame":
                continue

            values = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            reconstructed[f"{col}_mean"] = values.mean()
            reconstructed[f"{col}_std"] = values.std(ddof=1)

    # Add video ID
    reconstructed["video_id"] = video_id

    return reconstructed


# ------------------------------------------------------------
# 3. Find missing videos
# ------------------------------------------------------------

def get_missing_video_ids(label_names, visual_df):

    existing_ids = set(
        visual_df["video_id"]
        .apply(clean_name)
    )

    missing = []

    for name in label_names:

        vid = clean_name(name)

        if vid not in existing_ids:
            missing.append(vid)

    return missing


# ------------------------------------------------------------
# 4. Reconstruct VALIDATION missing videos
# ------------------------------------------------------------

# Use clean_val_names and emoreact_visual["val"] instead of undefined variables
val_missing_ids = get_missing_video_ids(
    clean_val_names,
    emoreact_visual["val"]
)

print("Validation missing:", len(val_missing_ids))
print(val_missing_ids)


val_folder = os.path.join(
    VISUAL_DIR,
    "Val_feat"
)

val_reconstructed_rows = []

for video_id in val_missing_ids:

    row = reconstruct_video(
        video_id,
        val_folder
    )

    if row is not None:
        val_reconstructed_rows.append(row)


val_missing_df = pd.DataFrame(
    val_reconstructed_rows
)

print("\nValidation reconstructed:")
print(val_missing_df.shape)


# ------------------------------------------------------------
# 5. Reconstruct TEST missing videos
# ------------------------------------------------------------

# Use clean_test_names and emoreact_visual["test"] instead of undefined variables
test_missing_ids = get_missing_video_ids(
    clean_test_names,
    emoreact_visual["test"]
)

print("\nTest missing:", len(test_missing_ids))
print(test_missing_ids)


test_folder = os.path.join(
    VISUAL_DIR,
    "Test_feat"
)

test_reconstructed_rows = []

for video_id in test_missing_ids:

    row = reconstruct_video(
        video_id,
        test_folder
    )

    if row is not None:
        test_reconstructed_rows.append(row)


test_missing_df = pd.DataFrame(
    test_reconstructed_rows
)

print("\nTest reconstructed:")
print(test_missing_df.shape)


# ------------------------------------------------------------
# 6. Save reconstructed datasets separately
# ------------------------------------------------------------

val_output = os.path.join(
    EMO_BASE,
    "val_missing_reconstructed.csv"
)

test_output = os.path.join(
    EMO_BASE,
    "test_missing_reconstructed.csv"
)

val_missing_df.to_csv(
    val_output,
    index=False
)

test_missing_df.to_csv(
    test_output,
    index=False
)


print("\n================================")
print("RECONSTRUCTION COMPLETE")
print("================================")

print("Validation:", val_output)
print("Test:", test_output)

print(
    "\nTotal reconstructed:",
    len(val_missing_df) + len(test_missing_df)
)

In [ ]:
# Get the actual visual feature DataFrames from the dictionary

print("emoreact_visual type:")
print(type(emoreact_visual))

print("\nAvailable keys:")
print(emoreact_visual.keys())

# Correct assignment
val_visual = emoreact_visual["val"]
test_visual = emoreact_visual["test"]

print("\n===== VISUAL DATA =====")
print("Validation rows:", len(val_visual))
print("Test rows:", len(test_visual))

print("\nValidation columns:", val_visual.shape)
print("Test columns:", test_visual.shape)

print("\nFirst 5 validation IDs:")
print(val_visual["video_id"].head().tolist())

print("\nFirst 5 test IDs:")
print(test_visual["video_id"].head().tolist())

In [ ]:
# ============================================================
# FIND THE EXACT MISSING EMOREACT VIDEO IDs
# ============================================================

# Normalize both sides
label_val_ids = set(clean_val_names)
label_test_ids = set(clean_test_names)

visual_val_ids = set(
    val_visual["video_id"]
    .astype(str)
)

visual_test_ids = set(
    test_visual["video_id"]
    .astype(str)
)

# Find missing IDs
val_missing_ids = sorted(
    label_val_ids - visual_val_ids
)

test_missing_ids = sorted(
    label_test_ids - visual_test_ids
)

print("========== VALIDATION ==========")
print("Labels:", len(label_val_ids))
print("Visual:", len(visual_val_ids))
print("Missing:", len(val_missing_ids))

for x in val_missing_ids:
    print(x)


print("\n========== TEST ==========")
print("Labels:", len(label_test_ids))
print("Visual:", len(visual_test_ids))
print("Missing:", len(test_missing_ids))

for x in test_missing_ids:
    print(x)

In [ ]:
# ============================================================
# MERGE EMOREACT LABELS + VISUAL FEATURES
# ============================================================

# Make copies
train_emo = emoreact_labels["train"].copy()
val_emo   = emoreact_labels["val"].copy()
test_emo  = emoreact_labels["test"].copy()

train_vis = emoreact_visual["train"].copy()
val_vis   = emoreact_visual["val"].copy()
test_vis  = emoreact_visual["test"].copy()


# Normalize video IDs
for df in [train_emo, val_emo, test_emo,
           train_vis, val_vis, test_vis]:

    df["video_id"] = df["video_id"].astype(str).apply(clean_name)


# Merge
emoreact_train = train_vis.merge(
    train_emo,
    on="video_id",
    how="inner"
)

emoreact_val = val_vis.merge(
    val_emo,
    on="video_id",
    how="inner"
)

emoreact_test = test_vis.merge(
    test_emo,
    on="video_id",
    how="inner"
)


# Display results
print("===== EMOREACT MERGED DATA =====")

print("Train:", emoreact_train.shape)
print("Validation:", emoreact_val.shape)
print("Test:", emoreact_test.shape)

print("\nTotal rows:",
      len(emoreact_train) +
      len(emoreact_val) +
      len(emoreact_test))

print("\nTrain missing values:",
      emoreact_train.isna().sum().sum())

print("Validation missing values:",
      emoreact_val.isna().sum().sum())

print("Test missing values:",
      emoreact_test.isna().sum().sum())


print("\n===== TRAIN SAMPLE =====")
display(emoreact_train.head())

In [ ]:
# ============================================================
# STEP 1: ADD VIDEO IDs TO EMOREACT LABELS
# ============================================================

# Make copies
train_emo = emoreact_labels["train"].copy()
val_emo   = emoreact_labels["val"].copy()
test_emo  = emoreact_labels["test"].copy()


# Add the video names in the exact order supplied by EmoReact
train_emo.insert(
    0,
    "video_id",
    [clean_name(x) for x in clean_train_names]
)

val_emo.insert(
    0,
    "video_id",
    [clean_name(x) for x in clean_val_names]
)

test_emo.insert(
    0,
    "video_id",
    [clean_name(x) for x in clean_test_names]
)


print("===== LABEL DATA WITH VIDEO IDs =====")

print("Train:", train_emo.shape)
print("Validation:", val_emo.shape)
print("Test:", test_emo.shape)

print("\nTrain first 5:")
display(train_emo.head())

print("\nValidation first 5:")
display(val_emo.head())

print("\nTest first 5:")
display(test_emo.head())

In [ ]:
# ============================================================
# STEP 2: MERGE LABELS + VISUAL FEATURES
# ============================================================

# Make copies
train_vis = emoreact_visual["train"].copy()
val_vis   = emoreact_visual["val"].copy()
test_vis  = emoreact_visual["test"].copy()

# Normalize visual feature IDs
for df in [train_vis, val_vis, test_vis]:
    df["video_id"] = df["video_id"].astype(str).apply(clean_name)

# Merge labels with visual features
emoreact_train = train_vis.merge(
    train_emo,
    on="video_id",
    how="inner"
)

emoreact_val = val_vis.merge(
    val_emo,
    on="video_id",
    how="inner"
)

emoreact_test = test_vis.merge(
    test_emo,
    on="video_id",
    how="inner"
)

# Check
print("===== EMOREACT MERGED DATA =====")
print("Train:", emoreact_train.shape)
print("Validation:", emoreact_val.shape)
print("Test:", emoreact_test.shape)

print("\nTotal:",
      len(emoreact_train) +
      len(emoreact_val) +
      len(emoreact_test))

print("\nMissing mappings:")
print("Train:", len(train_emo) - len(emoreact_train))
print("Validation:", len(val_emo) - len(emoreact_val))
print("Test:", len(test_emo) - len(emoreact_test))

print("\n===== SAMPLE =====")
display(emoreact_train.head())

In [ ]:
# ============================================================
# STEP 3: EMOREACT LABEL DISTRIBUTION
# ============================================================

emo_labels = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]

print("===== EMOREACT LABEL DISTRIBUTION =====\n")

for split_name, df in [
    ("TRAIN", emoreact_train),
    ("VALIDATION", emoreact_val),
    ("TEST", emoreact_test)
]:

    print("=" * 50)
    print(split_name)
    print("=" * 50)

    for label in emo_labels[:-1]:
        print(f"\n{label}:")
        print(df[label].value_counts().sort_index())

    print("\nValence:")
    print(df["valence"].describe())

In [ ]:
# ============================================================
# STEP 4: EMOREACT LABEL CO-OCCURRENCE
# ============================================================

emotion_cols = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration"
]

# Combine all person-independent splits
emo_all = pd.concat(
    [
        emoreact_train,
        emoreact_val,
        emoreact_test
    ],
    ignore_index=True
)

print("Total samples:", len(emo_all))

# Number of active emotion labels per video
emo_all["num_active_emotions"] = emo_all[emotion_cols].sum(axis=1)

print("\n===== NUMBER OF ACTIVE EMOTIONS =====")
print(
    emo_all["num_active_emotions"]
    .value_counts()
    .sort_index()
)

print("\n===== FRUSTRATION CO-OCCURRENCE =====")

frustrated = emo_all[emo_all["frustration"] == 1]

print("Videos with frustration:", len(frustrated))

for col in emotion_cols:
    if col != "frustration":
        count = frustrated[col].sum()
        percentage = count / len(frustrated) * 100

        print(
            f"{col}: {count} "
            f"({percentage:.2f}%)"
        )

In [ ]:
# ============================================
# STEP 3A - Inspect EmoReact Evidence Features
# ============================================

print("TRAIN SHAPE:", emoreact_train.shape)
print("VALIDATION SHAPE:", emoreact_val.shape)
print("TEST SHAPE:", emoreact_test.shape)

print("\nTOTAL SAMPLES:",
      len(emoreact_train) + len(emoreact_val) + len(emoreact_test))

print("\nTRAIN COLUMNS:")
for i, col in enumerate(emoreact_train.columns):
    print(i, ":", col)

print("\nLABEL COLUMNS:")
label_columns = [
    "curiosity",
    "uncertainty",
    "excitement",
    "happiness",
    "surprise",
    "disgust",
    "fear",
    "frustration",
    "valence"
]

print([col for col in label_columns if col in emoreact_train.columns])

print("\nVISUAL FEATURE COUNT:")

visual_columns = [
    col for col in emoreact_train.columns
    if col not in ["video_id"] + label_columns
]

print("Visual features:", len(visual_columns))
print("First 20 visual features:")
print(visual_columns[:20])

In [ ]:
# ============================================================
# STEP 3B - EmoReact Frustration vs Visual Feature Evidence
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Use TRAIN split only
df = emoreact_train.copy()

# Original EmoReact frustration label
frustration_col = "frustration"

# Visual features only
visual_cols = [
    col for col in df.columns
    if col not in ["video_id"] + label_columns
]

results = []

# Compare frustration = 1 vs frustration = 0
group_0 = df[df[frustration_col] == 0]
group_1 = df[df[frustration_col] == 1]

for feature in visual_cols:

    x = group_0[feature].dropna()
    y = group_1[feature].dropna()

    if len(x) == 0 or len(y) == 0:
        continue

    # Mann-Whitney U test
    stat, p_value = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    # Rank-biserial effect size
    n1 = len(x)
    n2 = len(y)

    rank_biserial = (2 * stat) / (n1 * n2) - 1

    results.append({
        "feature": feature,
        "group0_median": x.median(),
        "group1_median": y.median(),
        "p_value": p_value,
        "rank_biserial_effect": rank_biserial
    })

frustration_results = pd.DataFrame(results)

# FDR correction
frustration_results["p_fdr"] = multipletests(
    frustration_results["p_value"],
    method="fdr_bh"
)[1]

# Sort by strongest statistical evidence
frustration_results = frustration_results.sort_values(
    "p_fdr"
).reset_index(drop=True)

print("==============================================")
print("EMOREACT FRUSTRATION vs VISUAL FEATURES")
print("==============================================")

print("\nFrustration = 0:", len(group_0))
print("Frustration = 1:", len(group_1))

print("\nSignificant features after FDR (p < 0.05):")

significant = frustration_results[
    frustration_results["p_fdr"] < 0.05
]

print("Count:", len(significant))

print("\nTop 20 features:")
display(
    significant.head(20)
)

In [ ]:
# ============================================================
# STEP 3C - Select Candidate EmoReact Frustration Features
# ============================================================

candidate_features = (
    frustration_results[
        frustration_results["p_fdr"] < 0.05
    ]
    .copy()
)

# Add absolute effect size
candidate_features["abs_effect"] = (
    candidate_features["rank_biserial_effect"].abs()
)

# Sort by effect size
candidate_features = candidate_features.sort_values(
    "abs_effect",
    ascending=False
).reset_index(drop=True)

print("==============================================")
print("CANDIDATE FRUSTRATION-RELATED VISUAL FEATURES")
print("==============================================")

print("\nTotal significant features:", len(candidate_features))

display(
    candidate_features[
        [
            "feature",
            "group0_median",
            "group1_median",
            "p_fdr",
            "rank_biserial_effect",
            "abs_effect"
        ]
    ]
)

In [ ]:
# ============================================================
# STEP 3D - ETDD70 Reading / Eye-Movement Evidence
# ============================================================

import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

df = etdd_behavior.copy()

# Eye-movement features
etdd_features = [
    "n_fix_trial",
    "sum_fix_dur_trial",
    "dwell_time_trial",
    "mean_fix_dur_trial",
    "n_sacc_trial",
    "sum_sacc_dur_trial",
    "mean_sacc_dur_trial",
    "mean_sacc_ampl_trial",
    "ratio_progress_regress_trial",
    "n_between_line_regress_trial",
    "n_within_line_regress_trial",
    "n_regress_trial",
    "n_progress_trial",
    "n_transit_trial"
]

# Check groups
print("Dyslexia status counts:")
print(df["dyslexia_status"].value_counts())

results = []

group_0 = df[df["dyslexia_status"] == "non-dyslexic"]
group_1 = df[df["dyslexia_status"] == "dyslexic"]

for feature in etdd_features:

    x = group_0[feature].dropna()
    y = group_1[feature].dropna()

    stat, p_value = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    n1 = len(x)
    n2 = len(y)

    rank_biserial = (2 * stat) / (n1 * n2) - 1

    results.append({
        "feature": feature,
        "non_dyslexic_median": x.median(),
        "dyslexic_median": y.median(),
        "p_value": p_value,
        "rank_biserial_effect": rank_biserial
    })

etdd_results = pd.DataFrame(results)

# FDR correction
etdd_results["p_fdr"] = multipletests(
    etdd_results["p_value"],
    method="fdr_bh"
)[1]

etdd_results["abs_effect"] = (
    etdd_results["rank_biserial_effect"].abs()
)

etdd_results = etdd_results.sort_values(
    "p_fdr"
).reset_index(drop=True)

print("\n==============================================")
print("ETDD70 READING / EYE-MOVEMENT EVIDENCE")
print("==============================================")

print("\nSignificant features after FDR:")
print(
    (etdd_results["p_fdr"] < 0.05).sum()
)

display(etdd_results)

In [ ]:
# ============================================================
# STEP 3D - Check Actual ETDD70 Column Names
# ============================================================

print("ETDD70 shape:", etdd_behavior.shape)

print("\nActual columns:")
for i, col in enumerate(etdd_behavior.columns):
    print(i, ":", col)

print("\nFirst 5 rows:")
display(etdd_behavior.head())

In [ ]:
# ============================================================
# STEP 3D - ETDD70 Reading / Eye-Movement Evidence
# Using ACTUAL columns in etdd_behavior
# ============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

df = etdd_behavior.copy()

# Actual ETDD70 features available in our processed dataset
etdd_features = [
    "sum_sacc_dur",
    "sum_fix_dur",
    "dwell_time",
    "mean_fix_dur",
    "n_sacc",
    "n_regress"
]

# Check groups
print("Dyslexia status counts:")
print(df["dyslexia_status"].value_counts())

# Separate groups
non_dyslexic = df[df["dyslexia_status"] == "non-dyslexic"]
dyslexic = df[df["dyslexia_status"] == "dyslexic"]

results = []

for feature in etdd_features:

    x = non_dyslexic[feature].dropna()
    y = dyslexic[feature].dropna()

    # Mann-Whitney U test
    statistic, p_value = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    # Rank-biserial effect size
    n1 = len(x)
    n2 = len(y)

    rank_biserial = (2 * statistic) / (n1 * n2) - 1

    results.append({
        "feature": feature,
        "non_dyslexic_median": x.median(),
        "dyslexic_median": y.median(),
        "p_value": p_value,
        "rank_biserial_effect": rank_biserial
    })

etdd_results = pd.DataFrame(results)

# FDR correction for multiple comparisons
etdd_results["p_fdr"] = multipletests(
    etdd_results["p_value"],
    method="fdr_bh"
)[1]

# Absolute effect size
etdd_results["abs_effect"] = (
    etdd_results["rank_biserial_effect"].abs()
)

# Sort by statistical significance
etdd_results = etdd_results.sort_values(
    "p_fdr"
).reset_index(drop=True)

print("\n==============================================")
print("ETDD70 READING / EYE-MOVEMENT EVIDENCE")
print("==============================================")

print("\nTotal sessions:", len(df))
print("Non-dyslexic:", len(non_dyslexic))
print("Dyslexic:", len(dyslexic))

print("\nSignificant features after FDR (p < 0.05):")

significant = etdd_results[
    etdd_results["p_fdr"] < 0.05
]

print("Count:", len(significant))

print("\nResults:")
display(etdd_results)

In [ ]:
# ============================================================
# STEP 3E - Check Redundancy Among ETDD70 Features
# ============================================================

etdd_selected_features = [
    "sum_sacc_dur",
    "sum_fix_dur",
    "dwell_time",
    "mean_fix_dur",
    "n_sacc",
    "n_regress"
]

# Spearman correlation is suitable for behavioural/reading measures
corr_matrix = etdd_behavior[
    etdd_selected_features
].corr(method="spearman")

print("==============================================")
print("ETDD70 FEATURE CORRELATION")
print("==============================================")

display(corr_matrix)

# Find highly correlated pairs
high_corr_pairs = []

for i in range(len(etdd_selected_features)):
    for j in range(i + 1, len(etdd_selected_features)):

        f1 = etdd_selected_features[i]
        f2 = etdd_selected_features[j]

        rho = corr_matrix.loc[f1, f2]

        if abs(rho) >= 0.80:
            high_corr_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "spearman_rho": rho
            })

high_corr_pairs = pd.DataFrame(high_corr_pairs)

print("\nHighly correlated pairs (|rho| >= 0.80):")

if len(high_corr_pairs) == 0:
    print("No highly correlated pairs found.")
else:
    display(high_corr_pairs)

In [ ]:
# ============================================================
# STEP 3F - Final ETDD70 Evidence Feature Set
# ============================================================

etdd_final_features = [
    "sum_sacc_dur",
    "sum_fix_dur",
    "mean_fix_dur",
    "n_sacc",
    "n_regress"
]

print("==============================================")
print("FINAL ETDD70 EVIDENCE FEATURES")
print("==============================================")

print("\nSelected features:")
for i, feature in enumerate(etdd_final_features, start=1):
    print(f"{i}. {feature}")

print("\nRemoved due to near-duplicate information:")
print("dwell_time")
print("Reason: Spearman correlation with sum_fix_dur = 0.9992")

# Create a clean evidence dataframe
etdd_evidence = etdd_behavior[
    ["session_id", "subject_id",
     "dyslexia_status"] + etdd_final_features
].copy()

print("\nEvidence dataset shape:", etdd_evidence.shape)

display(etdd_evidence.head())

In [ ]:
# ============================================================
# STEP 4A - Check Available Nena-Man / Behavioral Data Files
# ============================================================

import os

base_path = "/content/drive/MyDrive/Behavior Detection"

print("Behavior Detection folder contents:\n")

for item in os.listdir(base_path):
    print(item)

In [ ]:
# ============================================================
# STEP 4B - Search for Available CSV / JSON Data
# ============================================================

import os

base_path = "/content/drive/MyDrive"

data_files = []

for root, dirs, files in os.walk(base_path):

    # Skip unnecessary system folders
    dirs[:] = [
        d for d in dirs
        if d not in [".ipynb_checkpoints"]
    ]

    for file in files:
        if file.lower().endswith((".csv", ".json", ".xlsx", ".parquet")):
            full_path = os.path.join(root, file)
            data_files.append(full_path)

print("Total data files found:", len(data_files))

print("\nRelevant files:")
for path in data_files:
    if any(word in path.lower() for word in [
        "behavior",
        "nena",
        "interaction",
        "telemetry",
        "reading",
        "voice",
        "session",
        "dataset"
    ]):
        print(path)

In [ ]:
# ============================================================
# STEP 4C - FINAL BEHAVIORAL STATE OPERATIONAL DEFINITIONS
# ============================================================

behavioral_state_definitions = {

    "Engaged": {
        "primary_evidence": [
            "sustained task-directed attention",
            "regular task interaction",
            "low idle behaviour",
            "successful task participation"
        ],
        "supporting_features": [
            "attentionDuration",
            "responseTime",
            "idleTime",
            "taskCompletion"
        ]
    },

    "Distracted": {
        "primary_evidence": [
            "reduced task-directed attention",
            "frequent looking away",
            "off-task or idle behaviour",
            "reduced task participation"
        ],
        "supporting_features": [
            "lookingAwayFrequency",
            "attentionDuration",
            "idleTime",
            "skipFrequency",
            "responseTime"
        ]
    },

    "Frustrated": {
        "primary_evidence": [
            "frustration-related affective evidence",
            "repeated errors or retries",
            "hesitation and prolonged pauses",
            "difficulty-related interaction disruption"
        ],
        "supporting_features": [
            "frustration evidence",
            "hesitationCount",
            "pauseDuration",
            "retryCount",
            "errorCount",
            "responseTime"
        ]
    },

    "Overwhelmed": {
        "primary_evidence": [
            "task difficulty exceeding current coping capacity",
            "prolonged hesitation",
            "reduced participation",
            "withdrawal or resignation behaviour"
        ],
        "supporting_features": [
            "pauseDuration",
            "responseTime",
            "retryCount",
            "idleTime",
            "taskCompletion",
            "reading performance"
        ]
    },

    "Overloaded": {
        "primary_evidence": [
            "persistent accumulation of task demands",
            "multiple simultaneous difficulty signals",
            "difficulty maintaining attention and performance"
        ],
        "supporting_features": [
            "responseTime",
            "pauseDuration",
            "retryCount",
            "idleTime",
            "reading performance",
            "attentionDuration"
        ]
    }
}

print("FINAL BEHAVIORAL STATE DEFINITIONS")
print("=" * 60)

for state, definition in behavioral_state_definitions.items():
    print(f"\n{state}")
    print("-" * 30)

    print("Primary evidence:")
    for item in definition["primary_evidence"]:
        print(" •", item)

    print("Supporting features:")
    for item in definition["supporting_features"]:
        print(" •", item)

In [ ]:
# ============================================================
# STEP 4D - DATASET TO BEHAVIORAL STATE EVIDENCE MATRIX
# ============================================================

dataset_evidence = {

    "EmoReact": {
        "Engaged": "Possible supporting evidence from affective/visual behavioural features",
        "Distracted": "Insufficient direct evidence from the available labels",
        "Frustrated": "Direct original frustration annotation provides supporting affective evidence",
        "Overwhelmed": "Insufficient direct evidence",
        "Overloaded": "Insufficient direct evidence"
    },

    "ETDD70": {
        "Engaged": "Reading-related eye-movement evidence may support task engagement",
        "Distracted": "Insufficient direct evidence",
        "Frustrated": "Reading difficulty indicators may provide supporting evidence only",
        "Overwhelmed": "Insufficient direct evidence",
        "Overloaded": "Insufficient direct evidence"
    },

    "Nena-Man": {
        "Engaged": "Interaction, reading performance and attention signals can provide direct behavioural evidence",
        "Distracted": "Looking-away, idle and reduced interaction signals can provide behavioural evidence",
        "Frustrated": "Errors, retries, hesitation and pauses can provide behavioural evidence when combined with affective evidence",
        "Overwhelmed": "Multiple difficulty and withdrawal signals can provide supporting evidence",
        "Overloaded": "Persistent simultaneous task-load signals can provide supporting evidence"
    }
}


print("DATASET → BEHAVIORAL STATE EVIDENCE MATRIX")
print("=" * 75)

for dataset, states in dataset_evidence.items():

    print(f"\n{dataset}")
    print("-" * 75)

    for state, evidence in states.items():
        print(f"{state:15} : {evidence}")

In [ ]:
# ============================================================
# STEP 4E - LABEL HARMONIZATION RULES
# ============================================================

label_harmonization_rules = {

    "Engaged": {
        "allowed_sources": [
            "EmoReact: supporting visual/affective evidence",
            "ETDD70: supporting reading-related evidence",
            "Nena-Man: direct interaction and task-performance evidence"
        ],
        "rule": (
            "Assign only when sustained task-directed attention and "
            "active task participation are sufficiently supported."
        )
    },

    "Distracted": {
        "allowed_sources": [
            "Nena-Man: looking-away, idle and reduced interaction signals"
        ],
        "rule": (
            "Assign only when reduced task-directed attention or "
            "off-task behaviour is supported by sufficient evidence."
        )
    },

    "Frustrated": {
        "allowed_sources": [
            "EmoReact: original frustration annotation",
            "Nena-Man: errors, retries, hesitation and pauses",
            "ETDD70: supporting reading-difficulty evidence only"
        ],
        "rule": (
            "Assign only when frustration-related affective evidence "
            "is supported by relevant difficulty, error, hesitation "
            "or interaction-disruption evidence."
        )
    },

    "Overwhelmed": {
        "allowed_sources": [
            "Nena-Man: multiple difficulty and withdrawal signals"
        ],
        "rule": (
            "Assign only when task difficulty appears to exceed "
            "current coping capacity, supported by multiple indicators."
        )
    },

    "Overloaded": {
        "allowed_sources": [
            "Nena-Man: persistent and simultaneous task-load signals"
        ],
        "rule": (
            "Assign only when multiple persistent load indicators "
            "co-occur and interfere with maintaining performance or attention."
        )
    }
}


print("LABEL HARMONIZATION RULES")
print("=" * 75)

for state, information in label_harmonization_rules.items():

    print(f"\n{state}")
    print("-" * 75)

    print("Allowed evidence sources:")
    for source in information["allowed_sources"]:
        print(" •", source)

    print("Rule:")
    print(" ", information["rule"])

print("\nIMPORTANT:")
print("Original dataset labels must be preserved.")
print("Unsupported behavioural states must NOT be artificially assigned.")

In [ ]:
# ============================================================
# STEP 4F - INSPECT ORIGINAL EMOREACT LABELS
# ============================================================

print("EMOREACT ORIGINAL LABELS")
print("=" * 70)

for split_name, df in [
    ("Train", emoreact_train),
    ("Validation", emoreact_val),
    ("Test", emoreact_test)
]:

    print(f"\n{split_name} split")
    print("-" * 70)

    for label in [
        "curiosity",
        "uncertainty",
        "excitement",
        "happiness",
        "surprise",
        "disgust",
        "fear",
        "frustration"
    ]:

        if label in df.columns:
            counts = df[label].value_counts().sort_index().to_dict()
            print(f"{label:15} : {counts}")

print("\n")
print("IMPORTANT:")
print("These are ORIGINAL EmoReact annotations.")
print("They will NOT be automatically renamed into the five")
print("proposed behavioural states.")

In [ ]:
# ============================================================
# STEP 4G - CANDIDATE LABEL HARMONIZATION MATRIX
# ============================================================

candidate_mappings = {

    "EmoReact": {
        "curiosity": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Curiosity is not equivalent to any proposed behavioural state."
        },

        "uncertainty": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Uncertainty alone does not establish distraction, frustration, overwhelm or overload."
        },

        "excitement": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Excitement is not equivalent to engaged behaviour without additional task-context evidence."
        },

        "happiness": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Happiness alone does not establish task engagement."
        },

        "surprise": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Surprise does not directly represent a proposed behavioural state."
        },

        "disgust": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Disgust does not directly represent a proposed behavioural state."
        },

        "fear": {
            "proposed_state": None,
            "status": "NOT_MAPPED",
            "reason": "Fear is not equivalent to overwhelmed or overloaded without additional evidence."
        },

        "frustration": {
            "proposed_state": "Frustrated",
            "status": "CANDIDATE_MAPPING",
            "reason": (
                "Direct semantic relevance to frustration-related affective evidence. "
                "Requires additional behavioural/difficulty evidence and expert validation "
                "before being treated as a harmonized behavioural-state label."
            )
        }
    }
}


print("CANDIDATE LABEL HARMONIZATION MATRIX")
print("=" * 80)

for dataset, labels in candidate_mappings.items():

    print(f"\n{dataset}")
    print("-" * 80)

    for original_label, mapping in labels.items():

        print(f"\nOriginal label : {original_label}")
        print(f"Proposed state : {mapping['proposed_state']}")
        print(f"Status         : {mapping['status']}")
        print(f"Reason         : {mapping['reason']}")


print("\n" + "=" * 80)
print("IMPORTANT")
print("=" * 80)
print("Candidate mappings are NOT final ground-truth labels.")
print("Expert/domain validation is required before model training.")
print("Unsupported mappings remain unmapped.")

In [ ]:
# ============================================================
# STEP 4H - ETDD70 SUPPORTING EVIDENCE VALIDATION
# ============================================================

etdd70_evidence_mapping = {

    "sum_sacc_dur": {
        "role": "supporting_feature",
        "supports": [
            "reading difficulty",
            "reading behaviour"
        ],
        "behavioral_state_label": None
    },

    "sum_fix_dur": {
        "role": "supporting_feature",
        "supports": [
            "reading difficulty",
            "reading behaviour"
        ],
        "behavioral_state_label": None
    },

    "mean_fix_dur": {
        "role": "supporting_feature",
        "supports": [
            "reading difficulty",
            "reading behaviour"
        ],
        "behavioral_state_label": None
    },

    "n_sacc": {
        "role": "supporting_feature",
        "supports": [
            "reading behaviour",
            "visual reading processing"
        ],
        "behavioral_state_label": None
    },

    "n_regress": {
        "role": "supporting_feature",
        "supports": [
            "reading difficulty",
            "reading behaviour"
        ],
        "behavioral_state_label": None
    }
}


print("ETDD70 SUPPORTING EVIDENCE")
print("=" * 75)

for feature, info in etdd70_evidence_mapping.items():

    print(f"\nFeature: {feature}")
    print(f"Role: {info['role']}")

    print("Supports:")
    for item in info["supports"]:
        print(" •", item)

    print("Behavioural-state label:", info["behavioral_state_label"])


print("\n" + "=" * 75)
print("CONCLUSION")
print("=" * 75)

print("ETDD70 features are retained as supporting reading-related evidence.")
print("They are NOT converted directly into the five behavioural-state labels.")
print("Dyslexia status is NOT treated as a behavioural-state label.")

In [2]:
import pandas as pd
import os
import glob

# ETDD70 paths
ETDD_BASE = "/content/drive/MyDrive/Behavior Detection/ETDD70"
ETDD_DATA = os.path.join(ETDD_BASE, "data", "data")

print("ETDD70 folder exists:", os.path.exists(ETDD_BASE))
print("ETDD70 data exists:", os.path.exists(ETDD_DATA))

# Find metrics files
metrics_files = glob.glob(os.path.join(ETDD_DATA, "*_metrics.csv"))

print("Metrics files found:", len(metrics_files))

rows = []

for file in metrics_files:

    df = pd.read_csv(file)

    if df.empty:
        continue

    # One row per trial
    trial_df = df.drop_duplicates(subset=["trialid"]).iloc[0]

    filename = os.path.basename(file)
    name = filename.replace("_metrics.csv", "")

    parts = name.split("_")
    subject_id = int(parts[1])

    row = {
        "session_id": name,
        "subject_id": subject_id,

        "sum_sacc_dur": trial_df["sum_sacc_dur_trial"],
        "sum_fix_dur": trial_df["sum_fix_dur_trial"],
        "dwell_time": trial_df["dwell_time_trial"],
        "mean_fix_dur": trial_df["mean_fix_dur_trial"],
        "n_sacc": trial_df["n_sacc_trial"],
        "n_regress": trial_df["n_regress_trial"],

        # DO NOT assign behavioral state
        "behavior_state": None
    }

    rows.append(row)

etdd_behavior = pd.DataFrame(rows)

print("\nETDD70 FEATURE DATASET")
print("=====================")
print("Shape:", etdd_behavior.shape)
print("Subjects:", etdd_behavior["subject_id"].nunique())
print("Sessions:", len(etdd_behavior))

display(etdd_behavior.head())

ETDD70 folder exists: True
ETDD70 data exists: True
Metrics files found: 210

ETDD70 FEATURE DATASET
Shape: (210, 9)
Subjects: 70
Sessions: 210


,session_id,subject_id,sum_sacc_dur,sum_fix_dur,dwell_time,mean_fix_dur,n_sacc,n_regress,behavior_state
0,Subject_1235_T5_Pseudo_Text,1235,9798.889,118469.347,127308.354,493.622279,239,16,None
1,Subject_1090_T5_Pseudo_Text,1090,8547.732,142779.041,150346.851,582.771596,244,15,None
2,Subject_1040_T1_Syllables,1040,10379.919,67457.308,77077.278,355.038463,189,21,None
3,Subject_1019_T1_Syllables,1019,6415.297,54347.358,60130.722,343.970620,157,17,None
4,Subject_1417_T5_Pseudo_Text,1417,11850.152,134584.873,145383.124,511.729555,262,23,None


In [3]:
import os

path = "/content/drive/MyDrive/Behavior Detection/ETDD70"

print(os.listdir(path))

['dyslexia_class_label.csv', 'data.zip', 'data', 'ETDD70_features.csv', 'Behavior_Detection_ETDD70_Feature_Extraction.ipynb', 'ETDD70_behavior_features.csv']


In [5]:
import pandas as pd

file_path = "/content/drive/MyDrive/Behavior Detection/ETDD70/ETDD70_behavior_features.csv"

etdd70_final = pd.read_csv(file_path)

print("Shape:", etdd70_final.shape)
print("\nColumns:")
print(etdd70_final.columns.tolist())

print("\nBehavior State values:")
print(etdd70_final["behavior_state"].value_counts(dropna=False))

display(etdd70_final.head())

Shape: (210, 16)

Columns:
['subject_id', 'task', 'label', 'n_fix_trial', 'sum_fix_dur_trial', 'dwell_time_trial', 'mean_fix_dur_trial', 'n_sacc_trial', 'sum_sacc_dur_trial', 'mean_sacc_dur_trial', 'mean_sacc_ampl_trial', 'ratio_progress_regress_trial', 'n_within_line_regress_trial', 'n_regress_trial', 'n_progress_trial', 'n_transit_trial']

Behavior State values:


KeyError: 'behavior_state'

In [6]:
import pandas as pd

file_path = "/content/drive/MyDrive/Behavior Detection/ETDD70/ETDD70_behavior_features.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)

print("\nLabel values:")
print(df["label"].value_counts(dropna=False))

print("\nTask values:")
print(df["task"].value_counts(dropna=False))

print("\nFirst 5 rows:")
display(df.head())

Shape: (210, 16)

Label values:
label
non-dyslexic    105
dyslexic        105
Name: count, dtype: int64

Task values:
task
T5_Pseudo_Text        70
T1_Syllables          70
T4_Meaningful_Text    70
Name: count, dtype: int64

First 5 rows:


,subject_id,task,label,n_fix_trial,sum_fix_dur_trial,dwell_time_trial,mean_fix_dur_trial,n_sacc_trial,sum_sacc_dur_trial,mean_sacc_dur_trial,mean_sacc_ampl_trial,ratio_progress_regress_trial,n_within_line_regress_trial,n_regress_trial,n_progress_trial,n_transit_trial
0,1235,T5_Pseudo_Text,non-dyslexic,240,118469.347,127308.354,493.622279,239,9798.889,40.999536,57.502624,6.812500,16,16,109,125
1,1090,T5_Pseudo_Text,non-dyslexic,245,142779.041,150346.851,582.771596,244,8547.732,35.031689,59.283245,7.333333,15,15,110,125
2,1040,T1_Syllables,non-dyslexic,190,67457.308,77077.278,355.038463,189,10379.919,54.920206,137.143121,4.952381,21,21,104,125
3,1019,T1_Syllables,non-dyslexic,158,54347.358,60130.722,343.970620,157,6415.297,40.861764,137.947652,5.764706,17,17,98,115
4,1417,T5_Pseudo_Text,dyslexic,263,134584.873,145383.124,511.729555,262,11850.152,45.229588,62.180238,4.956522,23,23,114,137


In [7]:
# ============================================================
# ETDD70 FINAL BEHAVIOR-SUPPORTING EVIDENCE DATASET
# ============================================================

import pandas as pd

input_file = "/content/drive/MyDrive/Behavior Detection/ETDD70/ETDD70_behavior_features.csv"

df = pd.read_csv(input_file)

# Create a unique session ID from subject + task
df["session_id"] = (
    df["subject_id"].astype(str)
    + "_"
    + df["task"].astype(str)
)

# Select only evidence-supported eye-movement features
final_etdd70 = df[
    [
        "session_id",
        "subject_id",
        "task",
        "label",
        "sum_sacc_dur_trial",
        "sum_fix_dur_trial",
        "mean_fix_dur_trial",
        "n_sacc_trial",
        "n_regress_trial"
    ]
].copy()

# Rename label so it is clear that this is dyslexia status,
# NOT behavioral state.
final_etdd70 = final_etdd70.rename(
    columns={
        "label": "dyslexia_status"
    }
)

print("ETDD70 FINAL EVIDENCE DATASET")
print("=" * 60)

print("Shape:", final_etdd70.shape)
print("Subjects:", final_etdd70["subject_id"].nunique())
print("Sessions:", final_etdd70["session_id"].nunique())

print("\nDyslexia status:")
print(final_etdd70["dyslexia_status"].value_counts())

print("\nFinal columns:")
print(final_etdd70.columns.tolist())

display(final_etdd70.head())

# Save final evidence dataset
output_file = "/content/drive/MyDrive/Behavior Detection/ETDD70/ETDD70_behavior_evidence.csv"

final_etdd70.to_csv(
    output_file,
    index=False
)

print("\nSaved successfully to:")
print(output_file)

ETDD70 FINAL EVIDENCE DATASET
Shape: (210, 9)
Subjects: 70
Sessions: 210

Dyslexia status:
dyslexia_status
non-dyslexic    105
dyslexic        105
Name: count, dtype: int64

Final columns:
['session_id', 'subject_id', 'task', 'dyslexia_status', 'sum_sacc_dur_trial', 'sum_fix_dur_trial', 'mean_fix_dur_trial', 'n_sacc_trial', 'n_regress_trial']


,session_id,subject_id,task,dyslexia_status,sum_sacc_dur_trial,sum_fix_dur_trial,mean_fix_dur_trial,n_sacc_trial,n_regress_trial
0,1235_T5_Pseudo_Text,1235,T5_Pseudo_Text,non-dyslexic,9798.889,118469.347,493.622279,239,16
1,1090_T5_Pseudo_Text,1090,T5_Pseudo_Text,non-dyslexic,8547.732,142779.041,582.771596,244,15
2,1040_T1_Syllables,1040,T1_Syllables,non-dyslexic,10379.919,67457.308,355.038463,189,21
3,1019_T1_Syllables,1019,T1_Syllables,non-dyslexic,6415.297,54347.358,343.970620,157,17
4,1417_T5_Pseudo_Text,1417,T5_Pseudo_Text,dyslexic,11850.152,134584.873,511.729555,262,23



Saved successfully to:
/content/drive/MyDrive/Behavior Detection/ETDD70/ETDD70_behavior_evidence.csv


In [8]:
# STEP 5 - FINAL MULTIMODAL EVIDENCE FEATURE SET

final_features = {
    "ETDD70_reading_features": [
        "sum_sacc_dur_trial",
        "sum_fix_dur_trial",
        "mean_fix_dur_trial",
        "n_sacc_trial",
        "n_regress_trial"
    ],

    "PInSoRo_interaction_features": [
        "hand_movement_std"
    ],

    "EmoReact_affective_evidence": [
        "frustration"
    ],

    "NenaMan_behavioral_features": [
        "reading_accuracy",
        "reading_speed",
        "hesitation_count",
        "pause_duration",
        "fluency_score",
        "error_count",
        "repeated_errors",
        "response_latency",
        "retry_count",
        "skip_count",
        "idle_time",
        "task_completion_rate",
        "gaze_away_frequency",
        "attention_duration",
        "head_pose_variation",
        "face_visibility"
    ]
}

print("=" * 70)
print("FINAL MULTIMODAL EVIDENCE FEATURE SET")
print("=" * 70)

for source, features in final_features.items():
    print(f"\n{source}")
    print("-" * len(source))
    for f in features:
        print("✓", f)

print("\n" + "=" * 70)
print("IMPORTANT")
print("=" * 70)
print("These are evidence/features from different sources.")
print("They must NOT be treated as if they belong to the same child/session.")
print("Behavioral-state labels are NOT created yet.")

FINAL MULTIMODAL EVIDENCE FEATURE SET

ETDD70_reading_features
-----------------------
✓ sum_sacc_dur_trial
✓ sum_fix_dur_trial
✓ mean_fix_dur_trial
✓ n_sacc_trial
✓ n_regress_trial

PInSoRo_interaction_features
----------------------------
✓ hand_movement_std

EmoReact_affective_evidence
---------------------------
✓ frustration

NenaMan_behavioral_features
---------------------------
✓ reading_accuracy
✓ reading_speed
✓ hesitation_count
✓ pause_duration
✓ fluency_score
✓ error_count
✓ repeated_errors
✓ response_latency
✓ retry_count
✓ skip_count
✓ idle_time
✓ task_completion_rate
✓ gaze_away_frequency
✓ attention_duration
✓ head_pose_variation
✓ face_visibility

IMPORTANT
These are evidence/features from different sources.
They must NOT be treated as if they belong to the same child/session.
Behavioral-state labels are NOT created yet.


In [9]:
# STEP 6 - BEHAVIORAL STATE LABEL HARMONIZATION FRAMEWORK

label_harmonization = {
    "Engaged": {
        "required_evidence": [
            "sustained task-directed attention",
            "active participation",
            "successful task interaction"
        ],
        "supporting_sources": [
            "Nena-Man",
            "ETDD70"
        ],
        "direct_label_allowed": False,
        "rule": "Assign only when sustained task-directed attention and active participation are sufficiently supported."
    },

    "Distracted": {
        "required_evidence": [
            "reduced task-directed attention",
            "looking away / off-task behavior",
            "increased idle time or reduced participation"
        ],
        "supporting_sources": [
            "Nena-Man"
        ],
        "direct_label_allowed": False,
        "rule": "Assign only when reduced task attention or off-task behavior is sufficiently supported."
    },

    "Frustrated": {
        "required_evidence": [
            "frustration-related affective evidence",
            "repeated errors or retries",
            "hesitation or prolonged pauses",
            "difficulty-related interaction disruption"
        ],
        "supporting_sources": [
            "EmoReact",
            "PInSoRo",
            "Nena-Man",
            "ETDD70"
        ],
        "direct_label_allowed": False,
        "rule": "Frustration affective evidence should be supported by relevant difficulty, error, hesitation, or interaction-disruption evidence."
    },

    "Overwhelmed": {
        "required_evidence": [
            "multiple task-difficulty indicators",
            "prolonged hesitation",
            "reduced participation",
            "withdrawal / resignation"
        ],
        "supporting_sources": [
            "Nena-Man"
        ],
        "direct_label_allowed": False,
        "rule": "Assign only when task difficulty appears to exceed coping capacity and multiple indicators co-occur."
    },

    "Overloaded": {
        "required_evidence": [
            "persistent accumulation of task demands",
            "multiple simultaneous difficulty signals",
            "difficulty maintaining attention or performance"
        ],
        "supporting_sources": [
            "Nena-Man"
        ],
        "direct_label_allowed": False,
        "rule": "Assign only when multiple persistent load indicators co-occur and interfere with maintaining performance or attention."
    }
}

print("=" * 80)
print("BEHAVIORAL STATE LABEL HARMONIZATION FRAMEWORK")
print("=" * 80)

for state, info in label_harmonization.items():
    print(f"\n[{state}]")
    print("Required evidence:")
    for evidence in info["required_evidence"]:
        print("  -", evidence)

    print("Supporting sources:", ", ".join(info["supporting_sources"]))
    print("Direct label allowed:", info["direct_label_allowed"])
    print("Rule:", info["rule"])

print("\n" + "=" * 80)
print("VALIDATION RULE")
print("=" * 80)
print("Original dataset labels must be preserved.")
print("Unsupported behavioral states must NOT be artificially assigned.")
print("Expert/domain validation is required before final label assignment.")

BEHAVIORAL STATE LABEL HARMONIZATION FRAMEWORK

[Engaged]
Required evidence:
  - sustained task-directed attention
  - active participation
  - successful task interaction
Supporting sources: Nena-Man, ETDD70
Direct label allowed: False
Rule: Assign only when sustained task-directed attention and active participation are sufficiently supported.

[Distracted]
Required evidence:
  - reduced task-directed attention
  - looking away / off-task behavior
  - increased idle time or reduced participation
Supporting sources: Nena-Man
Direct label allowed: False
Rule: Assign only when reduced task attention or off-task behavior is sufficiently supported.

[Frustrated]
Required evidence:
  - frustration-related affective evidence
  - repeated errors or retries
  - hesitation or prolonged pauses
  - difficulty-related interaction disruption
Supporting sources: EmoReact, PInSoRo, Nena-Man, ETDD70
Direct label allowed: False
Rule: Frustration affective evidence should be supported by relevant diffic

In [10]:
# STEP 7 - SOURCE TO BEHAVIORAL STATE EVIDENCE MATRIX

evidence_matrix = {
    "EmoReact": {
        "Engaged": "Supporting evidence only",
        "Distracted": "Insufficient direct evidence",
        "Frustrated": "Original frustration label provides supporting affective evidence",
        "Overwhelmed": "Insufficient direct evidence",
        "Overloaded": "Insufficient direct evidence"
    },

    "ETDD70": {
        "Engaged": "Supporting reading/eye-movement evidence",
        "Distracted": "Insufficient direct evidence",
        "Frustrated": "Supporting reading-difficulty evidence only",
        "Overwhelmed": "Insufficient direct evidence",
        "Overloaded": "Insufficient direct evidence"
    },

    "PInSoRo": {
        "Engaged": "Supporting interaction/engagement evidence",
        "Distracted": "Supporting disengagement evidence only",
        "Frustrated": "Supporting frustration-related interaction evidence",
        "Overwhelmed": "Insufficient direct evidence",
        "Overloaded": "Insufficient direct evidence"
    },

    "Nena-Man": {
        "Engaged": "Direct behavioral/reading-session evidence",
        "Distracted": "Direct interaction/attention evidence",
        "Frustrated": "Errors, retries, hesitation and pauses + affective evidence",
        "Overwhelmed": "Multiple difficulty and withdrawal indicators",
        "Overloaded": "Persistent simultaneous load indicators"
    }
}

print("=" * 85)
print("SOURCE → BEHAVIORAL STATE EVIDENCE MATRIX")
print("=" * 85)

states = [
    "Engaged",
    "Distracted",
    "Frustrated",
    "Overwhelmed",
    "Overloaded"
]

for source, evidence in evidence_matrix.items():
    print(f"\n{source}")
    print("-" * 85)

    for state in states:
        print(f"{state:15} : {evidence[state]}")

print("\n" + "=" * 85)
print("INTERPRETATION")
print("=" * 85)
print("✓ Direct evidence = potentially usable after validation")
print("✓ Supporting evidence = cannot independently define the state")
print("✓ Insufficient evidence = state must NOT be assigned from that source")
print("✓ Original dataset labels must remain unchanged")

SOURCE → BEHAVIORAL STATE EVIDENCE MATRIX

EmoReact
-------------------------------------------------------------------------------------
Engaged         : Supporting evidence only
Distracted      : Insufficient direct evidence
Frustrated      : Original frustration label provides supporting affective evidence
Overwhelmed     : Insufficient direct evidence
Overloaded      : Insufficient direct evidence

ETDD70
-------------------------------------------------------------------------------------
Engaged         : Supporting reading/eye-movement evidence
Distracted      : Insufficient direct evidence
Frustrated      : Supporting reading-difficulty evidence only
Overwhelmed     : Insufficient direct evidence
Overloaded      : Insufficient direct evidence

PInSoRo
-------------------------------------------------------------------------------------
Engaged         : Supporting interaction/engagement evidence
Distracted      : Supporting disengagement evidence only
Frustrated      : Support

In [11]:
# ============================================================
# STEP 5 — FINAL MULTIMODAL FEATURE SCHEMA
# ============================================================

import pandas as pd

feature_schema = {

    "ETDD70_reading_features": [
        "sum_sacc_dur_trial",
        "sum_fix_dur_trial",
        "mean_fix_dur_trial",
        "n_sacc_trial",
        "n_regress_trial"
    ],

    "PInSoRo_interaction_features": [
        "hand_movement_std"
    ],

    "EmoReact_affective_features": [
        "frustration"
    ],

    "NenaMan_behavioral_features": [
        "reading_accuracy",
        "reading_speed",
        "hesitation_count",
        "pause_duration",
        "fluency_score",
        "error_count",
        "repeated_errors",
        "response_latency",
        "retry_count",
        "skip_count",
        "idle_time",
        "task_completion_rate",
        "gaze_away_frequency",
        "attention_duration",
        "head_pose_variation",
        "face_visibility"
    ]
}

# Display schema
rows = []

for source, features in feature_schema.items():
    for feature in features:
        rows.append({
            "source": source,
            "feature": feature
        })

final_schema_df = pd.DataFrame(rows)

print("=" * 70)
print("FINAL MULTIMODAL FEATURE SCHEMA")
print("=" * 70)

print("\nTotal features:", len(final_schema_df))

display(final_schema_df)

# Save
output_path = "/content/drive/MyDrive/Behavior Detection/Behavioral_State_Model/final_multimodal_feature_schema.csv"

final_schema_df.to_csv(output_path, index=False)

print("\nSaved successfully:")
print(output_path)

FINAL MULTIMODAL FEATURE SCHEMA

Total features: 23


,source,feature
0,ETDD70_reading_features,sum_sacc_dur_trial
1,ETDD70_reading_features,sum_fix_dur_trial
2,ETDD70_reading_features,mean_fix_dur_trial
3,ETDD70_reading_features,n_sacc_trial
4,ETDD70_reading_features,n_regress_trial
5,PInSoRo_interaction_features,hand_movement_std
6,EmoReact_affective_features,frustration
7,NenaMan_behavioral_features,reading_accuracy
8,NenaMan_behavioral_features,reading_speed
9,NenaMan_behavioral_features,hesitation_count



Saved successfully:
/content/drive/MyDrive/Behavior Detection/Behavioral_State_Model/final_multimodal_feature_schema.csv
